# ✅ איך להריץ אחרי Kernel Reset (בלי להיתקע)

המטרה כאן: להריץ **EXP1 → EXP2 → EXP3 (POS/Lemma)** בצורה יציבה על GPU 6GB, עם מינימום ריצות מיותרות.

## סדר הרצה מומלץ אחרי Reset
1) **תא Setup (הבא מיד אחרי ההערה הזו)** – קובע allocator, Offline, וערכי `MAX_LEN/BATCH`.
2) תריצי את התאים המקוריים עד שיש לך:
   - `train_all, val_all, test_all` (ה־splits)
   - `tokenizer`, `collate_multibranch`, `mean_pool`, `grl`
   - `build_style_matrix(df)` (Style features)
3) רוצי את הסקשן שנוסף למטה: **"🔬 EXP Pipeline (Auto-added)"** לפי הסדר:
   - EXP1: Downsample EN_1 + rebuild STYLE/CHAR + loaders
   - (אימון EXP1 – אופציונלי אם כבר עשית)
   - EXP2: thresholds שונים ל־EN/GR (VAL→TEST)
   - EXP3: POS/Lemma (עם cache) + loaders + מודל + אימון חסכוני (AMP+checkpointing) + EXP2 מחדש

## אם יש OOM
- ודאי ש־`MAX_LEN=192` ו־`BATCH=2`.
- השתמשי ב־Gradient Checkpointing (מופעל בסקשן EXP3).
- אם עדיין תפוס זיכרון אחרי ניקוי: Kernel Restart.

---

In [ ]:
# -- PATH CONFIG -- edit this block if the folder moves ----------------
from pathlib import Path

BASE_DIR       = Path("C:/Users/Asoulin_Sapir/Desktop/עבודה עדכנית/thesis_final_project")
DATA_DIR       = BASE_DIR.parent   # English Excel files live in parent folder
GREEK_DATA_DIR = Path("C:/Users/Asoulin_Sapir/Desktop/Thesis/Data")

HP_REWRITES_PATH = DATA_DIR / "hp_book1_rewrites_by_paragraph.xlsx"
TW_ORIGINAL_PATH = DATA_DIR / "time_wizard_series_sentences.xlsx"
AP1_PATH         = GREEK_DATA_DIR / "data-Ap-1verif.xlsx"
AP2_PATH         = GREEK_DATA_DIR / "data-Ap-2.xlsx"

OUTPUTS_DIR = BASE_DIR / "outputs"
EXPORTS_DIR = BASE_DIR / "exports"
RUNS_DIR    = BASE_DIR / "runs"
for d in [OUTPUTS_DIR, EXPORTS_DIR, RUNS_DIR]:
    d.mkdir(parents=True, exist_ok=True)
# -----------------------------------------------------------------------

In [1]:
import os, gc, torch

# Better allocator to reduce fragmentation (Windows + 6GB GPUs love this)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# If you already have the model cached locally, this prevents HF from trying the internet
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_HUB_OFFLINE"] = "1"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MAX_LEN = 192   # memory-friendly
BATCH = 2       # memory-friendly

print("✅ DEVICE:", DEVICE, "| MAX_LEN:", MAX_LEN, "| BATCH:", BATCH)

def cuda_mem(msg=""):
    if DEVICE != "cuda":
        return
    torch.cuda.empty_cache()
    print(msg, "allocated(MB)=", round(torch.cuda.memory_allocated()/1024**2, 1),
          "reserved(MB)=", round(torch.cuda.memory_reserved()/1024**2, 1))

cuda_mem("Before running:")

✅ DEVICE: cuda | MAX_LEN: 192 | BATCH: 2
Before running: allocated(MB)= 0.0 reserved(MB)= 0.0


# מחברת מסודרת כרונולוגית – ניסויי Rewrite/Original (EN + Ancient Greek)

**מטרה:** לאפשר הרצה מחדש מקצה לקצה בסדר רציף, עם מינימום כפילויות והפרדה ברורה בין ניסויים.

**הנחיות הרצה:**
1. הריצי תאים לפי הסדר.
2. אם שינית נתיבים – עדכני רק בסקשן `CONFIG / PATHS`.
3. תאי `SAVE/LOAD` הם אופציונליים (חוסכים זמן).

---


# זיהוי שכתוב/כתיבה – הכנת דאטה + אימון מודלים (EN + Ancient Greek)

המחברת הזו מרכזת את כל השלבים שבנינו עד כה:

## חלק א — הכנת דאטה בפורמט אחיד (Sentence-level)
מייצרת טבלאות בפורמט: `lang, text, y`  
- `y=0` = כתיבה מקורית  
- `y=1` = שכתוב

### מקורות הדאטה
**Ancient Greek**
- `data-Ap-1verif.xlsx` — תוויות בעמודה `Category after review` (1=כתיבה, 3=שכתוב)
- `data-Ap-2.xlsx` — תוויות בעמודה `Category` (1=כתיבה, 3=שכתוב)

**English**
- `hp_book1_rewrites_by_paragraph.xlsx` — שכתוב לפי פסקאות → מפוצל למשפטים
- `time_wizard_series_sentences.xlsx` — כתיבה מקורית כבר ברמת משפט

## חלק ב — איחוד ופיצול סטים
- בניית `df_en`, `df_gr`
- איחוד ל-`df_all`
- פיצול 80/10/10 עם שמירת התפלגות לפי `(lang, y)`

## חלק ג — מודלים (אם קיימים במחברת / להמשך עבודה)
- מודל משותף (למשל XLM-R) או
- שני מודלים נפרדים: Ancient-Greek-BERT ליוונית + RoBERTa/BERT לאנגלית

> הערה: בגרסאות חדשות של `transformers` צריך לייבא `AdamW` מ-`torch.optim` ולא מ-`transformers`.


## A) בניית הדאטה (EN + GR) והכנת splits
---


## 1) טעינת דאטה – English

In [2]:
import pandas as pd
import re

hp_par = pd.read_excel(HP_REWRITES_PATH)     # columns: Book, Paragraph, HP Source, Text
tw_sent = pd.read_excel(TW_ORIGINAL_PATH)   # sentence_text + meta

print("hp_par:", hp_par.shape, hp_par.columns.tolist())
print("tw_sent:", tw_sent.shape, tw_sent.columns.tolist())


hp_par: (2974, 4) ['Book', 'Paragraph', 'HP Source', 'Text']
tw_sent: (280, 7) ['book_name', 'chapter_number', 'chapter_title', 'paragraph_number', 'sentence_number', 'sentence_id', 'sentence_text']


## 2) פיצול פסקאות למשפטים (English sentence splitter)

In [3]:
def split_english_sentences(text: str):
    """
    Split English paragraph into sentences with a light heuristic.
    Returns a list of sentences (strings).
    """
    if not isinstance(text, str):
        return []
    t = text.strip()
    if not t:
        return []

    # Normalize whitespace
    t = re.sub(r"\s+", " ", t)

    # Protect common abbreviations: replace "." with "<DOT>"
    abbr = [
        "Mr", "Mrs", "Ms", "Dr", "Prof", "Sr", "Jr",
        "St", "vs", "etc", "e.g", "i.e",
        "No", "Vol", "Ch", "Fig"
    ]
    for a in abbr:
        t = re.sub(rf"\b{re.escape(a)}\.", f"{a}<DOT>", t)

    # Protect initials like "J. K. Rowling"
    t = re.sub(r"\b([A-Z])\.", r"\1<DOT>", t)

    # Split on sentence-ending punctuation followed by space + capital/quote/bracket
    parts = re.split(r'(?<=[.!?])\s+(?=["(\[]?[A-Z])', t)

    # Restore dots and clean
    sents = []
    for p in parts:
        p = p.replace("<DOT>", ".").strip()
        if p:
            sents.append(p)

    return sents


## 3) בניית סט שכתוב באנגלית (HP → sentences, y=1)

In [4]:
# hp_par has: Book, Paragraph, HP Source, Text
hp_tmp = hp_par.copy()
hp_tmp["Text"] = hp_tmp["Text"].fillna("").astype(str)

hp_tmp["sent_list"] = hp_tmp["Text"].map(split_english_sentences)
hp_sents = hp_tmp.explode("sent_list").rename(columns={"sent_list": "text"}).copy()

hp_sents["text"] = hp_sents["text"].fillna("").astype(str).str.strip()
hp_sents = hp_sents[hp_sents["text"].str.len() > 0].copy()

# Add labels
hp_sents["y"] = 1
hp_sents["lang"] = "en"

# Optional: sentence index inside paragraph
hp_sents["sent_idx_in_par"] = hp_sents.groupby(["Book", "Paragraph"]).cumcount() + 1

# Keep useful columns
df_en_rewrite = hp_sents[["lang", "text", "y", "Book", "Paragraph", "HP Source", "sent_idx_in_par"]].reset_index(drop=True)

print("Rewrite sentences:", len(df_en_rewrite))
df_en_rewrite.head(5)


Rewrite sentences: 7260


,lang,text,y,Book,Paragraph,HP Source,sent_idx_in_par
0,en,"Mr. and Mrs. Dursley of number four, Privet Dr...",1,1,0,book1_para0,1
1,en,If anything peculiar ever brushed near their d...,1,1,0,book1_para0,2
2,en,They lived convinced that the world spun sensi...,1,1,0,book1_para0,3
3,en,"Mr. Dursley managed a company named Grunnings,...",1,1,1,book1_para1,1
4,en,"Physically, he was broad and stout, his neck b...",1,1,1,book1_para1,2


## 4) בניית סט כתיבה מקורית באנגלית (Time Wizard, y=0)

In [5]:
# tw_sent column is sentence_text
tw_tmp = tw_sent.copy()
tw_tmp["sentence_text"] = tw_tmp["sentence_text"].fillna("").astype(str).str.strip()
tw_tmp = tw_tmp[tw_tmp["sentence_text"].str.len() > 0].copy()

df_en_orig = tw_tmp.rename(columns={"sentence_text": "text"}).copy()
df_en_orig["y"] = 0
df_en_orig["lang"] = "en"

# Keep useful meta columns too
keep_cols = ["lang", "text", "y", "book_name", "chapter_number", "paragraph_number", "sentence_number", "sentence_id"]
df_en_orig = df_en_orig[[c for c in keep_cols if c in df_en_orig.columns]].reset_index(drop=True)

print("Original sentences:", len(df_en_orig))
df_en_orig.head(5)


Original sentences: 280


,lang,text,y,book_name,chapter_number,paragraph_number,sentence_number,sentence_id
0,en,Ori Ben‑Amitai is a bored seventh‑grader stari...,0,Book 1: The Boy Who Stepped Out of Time,1,1,1,Book 1: The Boy Who Stepped Out of Time__c01__...
1,en,"In a whisper of frustration, he wishes time wo...",0,Book 1: The Boy Who Stepped Out of Time,1,1,2,Book 1: The Boy Who Stepped Out of Time__c01__...
2,en,A falling pencil freezes midair while the teac...,0,Book 1: The Boy Who Stepped Out of Time,1,1,3,Book 1: The Boy Who Stepped Out of Time__c01__...
3,en,"When time snaps back, only Ori knows something...",0,Book 1: The Boy Who Stepped Out of Time,1,1,4,Book 1: The Boy Who Stepped Out of Time__c01__...
4,en,"At home, Ori finds a yellowed envelope address...",0,Book 1: The Boy Who Stepped Out of Time,2,1,1,Book 1: The Boy Who Stepped Out of Time__c02__...


## 5) איחוד סטים באנגלית → df_en

In [6]:
df_en = pd.concat([df_en_orig, df_en_rewrite], ignore_index=True)

print("df_en total:", len(df_en))
print(df_en["y"].value_counts())
df_en.sample(5, random_state=42)


df_en total: 7540
y
1    7260
0     280
Name: count, dtype: int64


,lang,text,y,book_name,chapter_number,paragraph_number,sentence_number,sentence_id,Book,Paragraph,HP Source,sent_idx_in_par
3045,en,"She stepped forward, bushy hair trembling, clu...",1,NaN,NaN,NaN,NaN,NaN,1.0,1236.0,book1_para1236,2.0
5484,en,"An owl called out, sharp and sudden, nearly se...",1,NaN,NaN,NaN,NaN,NaN,1.0,2341.0,book1_para2341,1.0
1616,en,"For a moment longer, he refused to let morning...",1,NaN,NaN,NaN,NaN,NaN,1.0,551.0,book1_para551,4.0
6348,en,"The heat pressed down in waves, heavier inside...",1,NaN,NaN,NaN,NaN,NaN,1.0,2721.0,book1_para2721,1.0
6357,en,But Neville didn’t know: Harry’s dreams had gr...,1,NaN,NaN,NaN,NaN,NaN,1.0,2723.0,book1_para2723,3.0


## 6) טעינת דאטה – Ancient Greek + מיפוי תוויות → df_gr

In [7]:
import pandas as pd

ap1 = pd.read_excel(AP1_PATH)
ap2 = pd.read_excel(AP2_PATH)

def build_lang_df(df, text_col, label_col, lang_tag):
    out = df[[text_col, label_col]].copy()
    out = out.rename(columns={text_col: "text", label_col: "label_raw"})
    # [EXP-A v2] Reverted to labels 1 and 3 only (labels 2 and 4 ignored).
    # Original mapping: 1 -> y=0 (original), 3 -> y=1 (rewrite).
    out = out[out["label_raw"].isin([1, 3])].copy()  # [EXP-A v2] reverted from isin([1,2,3,4])
    out["y"] = out["label_raw"].map({1: 0, 3: 1}).astype(int)  # [EXP-A v2] reverted from {1:0,2:0,3:1,4:1}
    out["lang"] = lang_tag
    out["text"] = out["text"].fillna("").astype(str)
    out = out[out["text"].str.len() > 0].copy()
    return out[["lang", "text", "y"]]

# NOTE: assumes text column is named 'Text' in both Greek files
df_gr_1 = build_lang_df(ap1, text_col="Text", label_col="Category after review", lang_tag="gr")
df_gr_2 = build_lang_df(ap2, text_col="Text", label_col="Category",             lang_tag="gr")

df_gr = pd.concat([df_gr_1, df_gr_2], ignore_index=True)

print("Greek rows:", len(df_gr))
print(df_gr["y"].value_counts())
df_gr.head()


Greek rows: 795
y
0    654
1    141
Name: count, dtype: int64


,lang,text,y
0,gr,Ἱκανῶς μὲν ὑπολαμβάνω καὶ διὰ τῆς περὶ τὴν ἀρχ...,0
1,gr,ἐπεὶ δὲ συχνοὺς ὁρῶ ταῖς ὑπὸ δυσμενείας ὑπό τι...,0
2,gr,χρήσομαι δὲ τῶν μὲν ὑπ᾽ ἐμοῦ λεγομένων μάρτυσι...,0
3,gr,"πειράσομαι δὲ καὶ τὰς αἰτίας ἀποδοῦναι, δι᾽ ἃς...",0
4,gr,Πρῶτον οὖν ἐπέρχεταί μοι πάνυ θαυμάζειν τοὺς ο...,0


## 7) פיצול יוונית 80/10/10 (baseline)

In [8]:
from sklearn.model_selection import train_test_split

SEED = 42

train_gr, temp_gr = train_test_split(
    df_gr, test_size=0.2, random_state=SEED, stratify=df_gr["y"]
)
val_gr, test_gr = train_test_split(
    temp_gr, test_size=0.5, random_state=SEED, stratify=temp_gr["y"]
)

print("Greek split sizes:", len(train_gr), len(val_gr), len(test_gr))


Greek split sizes: 636 79 80


## 8) איחוד EN+GR → df_all

In [9]:
df_all = pd.concat([df_en, df_gr], ignore_index=True)

print("df_all total:", len(df_all))
print(df_all.groupby(["lang","y"]).size())
df_all.sample(10, random_state=42)


df_all total: 8335
lang  y
en    0     280
      1    7260
gr    0     654
      1     141
dtype: int64


,lang,text,y,book_name,chapter_number,paragraph_number,sentence_number,sentence_id,Book,Paragraph,HP Source,sent_idx_in_par
1768,en,"Along the corridor, a sign glimmered with ster...",1,NaN,NaN,NaN,NaN,NaN,1.0,641.0,book1_para641,1.0
5550,en,What are you doing in the library?”—did Harry’...,1,NaN,NaN,NaN,NaN,NaN,1.0,2371.0,book1_para2371,3.0
7978,gr,δῆλον ὅτι οὐ ῥᾴδιον οὐδὲ ἀμαχεὶ στρατοπέδῳ διε...,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
932,en,"Across the room, Harry held his silence, certa...",1,NaN,NaN,NaN,NaN,NaN,1.0,237.0,book1_para237,3.0
5357,en,Torture—that was the only word for it.,1,NaN,NaN,NaN,NaN,NaN,1.0,2286.0,book1_para2286,4.0
5273,en,"Somehow, he had made his way through the portr...",1,NaN,NaN,NaN,NaN,NaN,1.0,2245.0,book1_para2245,2.0
1263,en,The giant’s voice shivered the air. “Last time...,1,NaN,NaN,NaN,NaN,NaN,1.0,396.0,book1_para396,1.0
2317,en,"Between them, at the center, there was nothing...",1,NaN,NaN,NaN,NaN,NaN,1.0,879.0,book1_para879,3.0
2888,en,I will take them from here.”,1,NaN,NaN,NaN,NaN,NaN,1.0,1154.0,book1_para1154,2.0
7429,en,"Across the back wall behind the High Table, a ...",1,NaN,NaN,NaN,NaN,NaN,1.0,3228.0,book1_para3228,4.0


## 9) פיצול משותף 80/10/10 לפי strata (lang_y)

In [10]:
# Split 80/10/10 on the combined dataset while preserving distribution of (lang, y)
strata = df_all[["lang","y"]].astype(str).agg("_".join, axis=1)

train_all, temp_all = train_test_split(
    df_all, test_size=0.2, random_state=SEED, stratify=strata
)
strata_temp = temp_all[["lang","y"]].astype(str).agg("_".join, axis=1)

val_all, test_all = train_test_split(
    temp_all, test_size=0.5, random_state=SEED, stratify=strata_temp
)

print("Combined split sizes:", len(train_all), len(val_all), len(test_all))
print("Train distribution:\n", train_all.groupby(["lang","y"]).size())


Combined split sizes: 6668 833 834
Train distribution:
 lang  y
en    0     224
      1    5808
gr    0     523
      1     113
dtype: int64


## B) Checkpointing – שמירה/טעינה כדי לא להריץ הכל מחדש
---


## 13) שמירת תוצרים (Checkpoint) כדי לא להריץ שוב

In [11]:
from pathlib import Path
import json, platform, sys
import pandas as pd
import numpy as np

RUN_DIR = RUNS_DIR / pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")
RUN_DIR.mkdir(parents=True, exist_ok=True)

# ---- Save datasets ----
# Requires: df_en, df_gr, df_all, train_all, val_all, test_all already exist
(df_en).to_parquet(RUN_DIR / "df_en.parquet", index=False)
(df_gr).to_parquet(RUN_DIR / "df_gr.parquet", index=False)
(df_all).to_parquet(RUN_DIR / "df_all.parquet", index=False)

(train_all).to_parquet(RUN_DIR / "train_all.parquet", index=False)
(val_all).to_parquet(RUN_DIR / "val_all.parquet", index=False)
(test_all).to_parquet(RUN_DIR / "test_all.parquet", index=False)

# Optional: also save CSV for quick inspection
df_all.to_csv(RUN_DIR / "df_all.csv", index=False)

# ---- Save config + environment ----
config = {
    "seed": 42,
    "schema": {"lang": "en/gr", "text": "sentence string", "y": "0=Original,1=Rewrite"},
    "labeling": {
        "ap-1verif": {"column": "Category after review", "1": 0, "3": 1},
        "ap-2": {"column": "Category", "1": 0, "3": 1},
        "english_hp_paragraphs": {"y": 1, "split_to_sentences": True},
        "english_time_wizard": {"y": 0}
    },
    "env": {
        "python": sys.version,
        "platform": platform.platform(),
    }
}

try:
    import torch, transformers, sklearn
    config["env"].update({
        "torch": torch.__version__,
        "transformers": transformers.__version__,
        "sklearn": sklearn.__version__,
        "numpy": np.__version__,
        "pandas": pd.__version__,
    })
except Exception as e:
    config["env"]["versions_error"] = str(e)

with open(RUN_DIR / "config.json", "w", encoding="utf-8") as f:
    json.dump(config, f, ensure_ascii=False, indent=2)

print("✅ Saved run artifacts to:", RUN_DIR.resolve())

✅ Saved run artifacts to: C:\Users\Asoulin_Sapir\Desktop\עבודה עדכנית\runs\20260302_122739


## 14) טעינת תוצרים (Load) – שחזור df_*/splits בלי להריץ את כל העיבוד

In [12]:
from pathlib import Path
import json
import pandas as pd

RUNS_ROOT = RUNS_DIR

# Find latest run folder (by name timestamp)
run_dirs = sorted([p for p in RUNS_ROOT.glob("*") if p.is_dir()])
if not run_dirs:
    raise FileNotFoundError(
        "No run folders found under 'runs/'.\n"
        "Run the SAVE cell first to create runs/YYYYMMDD_HHMMSS"
    )

RUN_DIR = run_dirs[-1]  # latest
print("Loading from:", RUN_DIR)

# Load
df_en    = pd.read_parquet(RUN_DIR / "df_en.parquet")
df_gr    = pd.read_parquet(RUN_DIR / "df_gr.parquet")
df_all   = pd.read_parquet(RUN_DIR / "df_all.parquet")

train_all = pd.read_parquet(RUN_DIR / "train_all.parquet")
val_all   = pd.read_parquet(RUN_DIR / "val_all.parquet")
test_all  = pd.read_parquet(RUN_DIR / "test_all.parquet")

with open(RUN_DIR / "config.json", "r", encoding="utf-8") as f:
    config = json.load(f)

print("✅ Loaded:", RUN_DIR)
print("df_all:", df_all.shape)
print("train/val/test:", train_all.shape, val_all.shape, test_all.shape)


Loading from: runs\20260302_122739
✅ Loaded: runs\20260302_122739
df_all: (8335, 12)
train/val/test: (6668, 12) (833, 12) (834, 12)


## C) Experiment 0 — Baseline: XLM-R סיווג Rewrite/Original (ללא Adversarial)
---


### C1) Model (baseline)

In [12]:
import os
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import classification_report

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

MODEL_NAME = "xlm-roberta-base"   # אפשר לשנות ל-large אם יש GPU חזק
MAX_LEN = 256
BATCH = 8

OUT_DIR = "outputs"
os.makedirs(OUT_DIR, exist_ok=True)

# map lang -> id
LANG2ID = {"en": 0, "gr": 1}

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class TextDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        return {
            "text": str(row["text"]),
            "y": int(row["y"]),
            "lang_id": LANG2ID[str(row["lang"])]
        }

def collate_fn(batch):
    texts = [b["text"] for b in batch]
    y = torch.tensor([b["y"] for b in batch], dtype=torch.long)
    lang_id = torch.tensor([b["lang_id"] for b in batch], dtype=torch.long)

    tok = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=MAX_LEN,
        return_tensors="pt"
    )
    return tok, y, lang_id


Device: cuda


In [13]:
def make_stratum(df):
    return df["lang"].astype(str) + "_" + df["y"].astype(str)

# Weighted sampler to balance (lang, y) in training
train_stratum = make_stratum(train_all)
counts = train_stratum.value_counts()
weights = train_stratum.map(lambda s: 1.0 / counts[s]).values
weights = torch.tensor(weights, dtype=torch.double)

sampler = WeightedRandomSampler(
    weights=weights,
    num_samples=len(weights),   # epoch size
    replacement=True
)

train_ds = TextDataset(train_all)
val_ds   = TextDataset(val_all)
test_ds  = TextDataset(test_all)

train_loader = DataLoader(train_ds, batch_size=BATCH, sampler=sampler, collate_fn=collate_fn)
val_loader   = DataLoader(val_ds, batch_size=BATCH, shuffle=False, collate_fn=collate_fn)
test_loader  = DataLoader(test_ds, batch_size=BATCH, shuffle=False, collate_fn=collate_fn)

print("Train stratum counts:\n", counts)


Train stratum counts:
 en_1    5808
gr_0     523
en_0     224
gr_1     113
Name: count, dtype: int64


## 10) מודל משותף (אופציונלי): XLM-R לסיווג Rewrite/Original

In [14]:
import torch.nn as nn
import torch.nn.functional as F

def mean_pool(last_hidden, attention_mask):
    mask = attention_mask.unsqueeze(-1).type_as(last_hidden)
    summed = (last_hidden * mask).sum(dim=1)
    denom = mask.sum(dim=1).clamp(min=1e-6)
    return summed / denom

class XLMRClassifier(nn.Module):
    def __init__(self, model_name, n_classes=2, dropout=0.1):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden = self.encoder.config.hidden_size
        self.drop = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden, n_classes)

    def forward(self, tok):
        out = self.encoder(**tok, return_dict=True)
        emb = mean_pool(out.last_hidden_state, tok["attention_mask"])
        logits = self.fc(self.drop(emb))
        return logits

model = XLMRClassifier(MODEL_NAME).to(DEVICE)
print("Model loaded.")


Model loaded.


In [15]:
# class weights based on train distribution of y
y_counts = train_all["y"].value_counts().sort_index()
# ensure both classes exist
for cls in [0,1]:
    if cls not in y_counts.index:
        y_counts.loc[cls] = 0
y_counts = y_counts.sort_index()

# weights inversely proportional to frequency
w0 = 1.0 / max(1, y_counts.loc[0])
w1 = 1.0 / max(1, y_counts.loc[1])
class_weights = torch.tensor([w0, w1], dtype=torch.float32, device=DEVICE)
class_weights = class_weights / class_weights.sum() * 2.0  # normalize (optional)

print("Train y counts:", y_counts.to_dict())
print("Class weights:", class_weights.detach().cpu().numpy())


Train y counts: {0: 747, 1: 5921}
Class weights: [1.7759448  0.22405519]


In [16]:
from transformers import get_linear_schedule_with_warmup
from torch.optim import AdamW
from sklearn.metrics import confusion_matrix

LR = 2e-5
WEIGHT_DECAY = 0.01
EPOCHS = 3

optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
total_steps = len(train_loader) * EPOCHS
warmup_steps = int(0.05 * total_steps)
scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)

scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

def run_eval(loader, df_ref, split_name="VAL"):
    model.eval()
    preds, ys, langs = [], [], []
    with torch.no_grad():
        for tok, y, lang_id in loader:
            tok = {k: v.to(DEVICE) for k, v in tok.items()}
            y = y.to(DEVICE)
            lang_id = lang_id.to(DEVICE)

            logits = model(tok)
            pred = torch.argmax(logits, dim=-1)

            preds.append(pred.cpu().numpy())
            ys.append(y.cpu().numpy())
            langs.append(lang_id.cpu().numpy())

    y_true = np.concatenate(ys)
    y_pred = np.concatenate(preds)
    lang_arr = np.concatenate(langs)

    print(f"\n=== {split_name} OVERALL ===")
    print(classification_report(y_true, y_pred, target_names=["Original","Rewrite"], zero_division=0))

    for lang_name, lang_id in LANG2ID.items():
        mask = (lang_arr == lang_id)
        if mask.sum() == 0:
            continue
        print(f"\n=== {split_name} | {lang_name.upper()} (n={mask.sum()}) ===")
        print(classification_report(y_true[mask], y_pred[mask], target_names=["Original","Rewrite"], zero_division=0))

    return y_true, y_pred, lang_arr

best_val_f1 = -1.0
best_path = os.path.join(OUT_DIR, "xlmr_rewrite_baseline.pt")

for epoch in range(1, EPOCHS+1):
    model.train()
    total_loss = 0.0

    for tok, y, lang_id in train_loader:
        tok = {k: v.to(DEVICE) for k, v in tok.items()}
        y = y.to(DEVICE)

        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            logits = model(tok)
            loss = F.cross_entropy(logits, y, weight=class_weights)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        total_loss += float(loss.detach().cpu().item())

    avg_loss = total_loss / max(1, len(train_loader))
    print(f"\nEpoch {epoch}/{EPOCHS} | train_loss={avg_loss:.4f}")

    # Evaluate on val, use macro-F1 as selection criterion
    yv, pv, lv = run_eval(val_loader, val_all, split_name="VAL")

    # compute macro f1 quickly from report dict
    rep = classification_report(yv, pv, output_dict=True, zero_division=0)
    macro_f1 = rep["macro avg"]["f1-score"]
    print("VAL macro-F1:", round(macro_f1, 4))

    if macro_f1 > best_val_f1:
        best_val_f1 = macro_f1
        torch.save(model.state_dict(), best_path)
        print("Saved best model to:", best_path)

# Load best + final test
model.load_state_dict(torch.load(best_path, map_location=DEVICE))
_ = run_eval(test_loader, test_all, split_name="TEST")


C:\Users\Asoulin_Sapir\AppData\Local\Temp\ipykernel_26304\2942740266.py:14: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())
C:\Users\Asoulin_Sapir\AppData\Local\Temp\ipykernel_26304\2942740266.py:61: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):



Epoch 1/3 | train_loss=0.2232

=== VAL OVERALL ===
              precision    recall  f1-score   support

    Original       0.84      0.94      0.89        93
     Rewrite       0.99      0.98      0.99       740

    accuracy                           0.97       833
   macro avg       0.92      0.96      0.94       833
weighted avg       0.98      0.97      0.97       833


=== VAL | EN (n=754) ===
              precision    recall  f1-score   support

    Original       0.63      0.93      0.75        28
     Rewrite       1.00      0.98      0.99       726

    accuracy                           0.98       754
   macro avg       0.82      0.95      0.87       754
weighted avg       0.98      0.98      0.98       754


=== VAL | GR (n=79) ===
              precision    recall  f1-score   support

    Original       0.98      0.94      0.96        65
     Rewrite       0.76      0.93      0.84        14

    accuracy                           0.94        79
   macro avg       0.87  

C:\Users\Asoulin_Sapir\AppData\Local\Temp\ipykernel_26304\2942740266.py:61: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):



Epoch 2/3 | train_loss=0.0339

=== VAL OVERALL ===
              precision    recall  f1-score   support

    Original       0.91      0.94      0.92        93
     Rewrite       0.99      0.99      0.99       740

    accuracy                           0.98       833
   macro avg       0.95      0.96      0.96       833
weighted avg       0.98      0.98      0.98       833


=== VAL | EN (n=754) ===
              precision    recall  f1-score   support

    Original       0.76      0.89      0.82        28
     Rewrite       1.00      0.99      0.99       726

    accuracy                           0.99       754
   macro avg       0.88      0.94      0.91       754
weighted avg       0.99      0.99      0.99       754


=== VAL | GR (n=79) ===
              precision    recall  f1-score   support

    Original       0.98      0.95      0.97        65
     Rewrite       0.81      0.93      0.87        14

    accuracy                           0.95        79
   macro avg       0.90  

C:\Users\Asoulin_Sapir\AppData\Local\Temp\ipykernel_26304\2942740266.py:61: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):



Epoch 3/3 | train_loss=0.0062

=== VAL OVERALL ===
              precision    recall  f1-score   support

    Original       0.89      0.95      0.92        93
     Rewrite       0.99      0.99      0.99       740

    accuracy                           0.98       833
   macro avg       0.94      0.97      0.95       833
weighted avg       0.98      0.98      0.98       833


=== VAL | EN (n=754) ===
              precision    recall  f1-score   support

    Original       0.74      0.89      0.81        28
     Rewrite       1.00      0.99      0.99       726

    accuracy                           0.98       754
   macro avg       0.87      0.94      0.90       754
weighted avg       0.99      0.98      0.98       754


=== VAL | GR (n=79) ===
              precision    recall  f1-score   support

    Original       0.97      0.97      0.97        65
     Rewrite       0.86      0.86      0.86        14

    accuracy                           0.95        79
   macro avg       0.91  

## D) (אופציונלי) שני מודלים נפרדים: Ancient-Greek-BERT ליוונית + RoBERTa לאנגלית
> זה לא מודל אגנוסטי-שפה; נשאר כאן רק לתיעוד/השוואה.
---


In [17]:
from sklearn.model_selection import train_test_split

SEED = 42

train_gr_sep, temp_gr_sep = train_test_split(df_gr, test_size=0.2, random_state=SEED, stratify=df_gr["y"])
val_gr_sep, test_gr_sep   = train_test_split(temp_gr_sep, test_size=0.5, random_state=SEED, stratify=temp_gr_sep["y"])

print(len(train_gr_sep), len(val_gr_sep), len(test_gr_sep))
print(train_gr_sep["y"].value_counts())


636 79 80
y
0    523
1    113
Name: count, dtype: int64


## 11) מודל נפרד ליוונית: Ancient-Greek-BERT

In [18]:
import os, numpy as np, torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from torch.optim import AdamW
from sklearn.metrics import classification_report

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

OUT_DIR = "outputs"
os.makedirs(OUT_DIR, exist_ok=True)

GREEK_MODEL_NAME_SEP = "pranaydeeps/Ancient-Greek-BERT"  # אם תרצי נחליף ל-altsoph
MAX_LEN_GR = 256
BATCH_GR = 8
EPOCHS_GR = 3
LR_GR = 2e-5
WEIGHT_DECAY = 0.01

tok_gr = AutoTokenizer.from_pretrained(GREEK_MODEL_NAME_SEP)

class SimpleTextDS(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        r = self.df.iloc[idx]
        return {"text": str(r["text"]), "y": int(r["y"])}

def collate_gr(batch):
    texts = [b["text"] for b in batch]
    y = torch.tensor([b["y"] for b in batch], dtype=torch.long)
    tok = tok_gr(texts, padding=True, truncation=True, max_length=MAX_LEN_GR, return_tensors="pt")
    return tok, y

def mean_pool(last_hidden, attention_mask):
    mask = attention_mask.unsqueeze(-1).type_as(last_hidden)
    summed = (last_hidden * mask).sum(dim=1)
    denom = mask.sum(dim=1).clamp(min=1e-6)
    return summed / denom

class GreekClassifier(nn.Module):
    def __init__(self, model_name, dropout=0.1, n_classes=2):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        h = self.encoder.config.hidden_size
        self.drop = nn.Dropout(dropout)
        self.fc = nn.Linear(h, n_classes)
    def forward(self, tok):
        out = self.encoder(**tok, return_dict=True)
        emb = mean_pool(out.last_hidden_state, tok["attention_mask"])
        return self.fc(self.drop(emb))

train_loader_gr = DataLoader(SimpleTextDS(train_gr_sep), batch_size=BATCH_GR, shuffle=True, collate_fn=collate_gr)
val_loader_gr   = DataLoader(SimpleTextDS(val_gr_sep),   batch_size=BATCH_GR, shuffle=False, collate_fn=collate_gr)
test_loader_gr  = DataLoader(SimpleTextDS(test_gr_sep),  batch_size=BATCH_GR, shuffle=False, collate_fn=collate_gr)

model_gr = GreekClassifier(GREEK_MODEL_NAME_SEP).to(DEVICE)

# class weights (optional)
y_counts = train_gr_sep["y"].value_counts().sort_index()
w0 = 1.0 / max(1, int(y_counts.get(0, 1)))
w1 = 1.0 / max(1, int(y_counts.get(1, 1)))
class_w = torch.tensor([w0, w1], dtype=torch.float32, device=DEVICE)
class_w = class_w / class_w.sum() * 2.0
print("Greek class weights:", class_w.detach().cpu().numpy())

opt = AdamW(model_gr.parameters(), lr=LR_GR, weight_decay=WEIGHT_DECAY)
steps = len(train_loader_gr) * EPOCHS_GR
sched = get_linear_schedule_with_warmup(opt, int(0.05*steps), steps)
scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

def eval_loader(loader, name="VAL"):
    model_gr.eval()
    ys, ps = [], []
    with torch.no_grad():
        for tok, y in loader:
            tok = {k:v.to(DEVICE) for k,v in tok.items()}
            y = y.to(DEVICE)
            logits = model_gr(tok)
            pred = torch.argmax(logits, dim=-1)
            ys.append(y.cpu().numpy())
            ps.append(pred.cpu().numpy())
    y_true = np.concatenate(ys); y_pred = np.concatenate(ps)
    print(f"\n=== GREEK {name} ===")
    print(classification_report(y_true, y_pred, target_names=["Original","Rewrite"], zero_division=0))
    return y_true, y_pred

best_f1 = -1
best_path_gr = os.path.join(OUT_DIR, "ancient_greek_bert_best.pt")

for ep in range(1, EPOCHS_GR+1):
    model_gr.train()
    tot = 0.0
    for tok, y in train_loader_gr:
        tok = {k:v.to(DEVICE) for k,v in tok.items()}
        y = y.to(DEVICE)
        opt.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            logits = model_gr(tok)
            loss = F.cross_entropy(logits, y, weight=class_w)

        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
        sched.step()
        tot += float(loss.detach().cpu().item())

    print(f"\n[Greek] Epoch {ep}/{EPOCHS_GR} | train_loss={tot/max(1,len(train_loader_gr)):.4f}")
    yv, pv = eval_loader(val_loader_gr, "VAL")
    rep = classification_report(yv, pv, output_dict=True, zero_division=0)
    macro_f1 = rep["macro avg"]["f1-score"]
    if macro_f1 > best_f1:
        best_f1 = macro_f1
        torch.save(model_gr.state_dict(), best_path_gr)
        print("Saved best Greek model:", best_path_gr)

model_gr.load_state_dict(torch.load(best_path_gr, map_location=DEVICE))
_ = eval_loader(test_loader_gr, "TEST")


Device: cuda
Greek class weights: [0.3553459 1.644654 ]


C:\Users\Asoulin_Sapir\AppData\Local\Temp\ipykernel_26304\2938237305.py:74: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())
C:\Users\Asoulin_Sapir\AppData\Local\Temp\ipykernel_26304\2938237305.py:103: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):



[Greek] Epoch 1/3 | train_loss=0.5216

=== GREEK VAL ===
              precision    recall  f1-score   support

    Original       0.97      0.94      0.95        65
     Rewrite       0.75      0.86      0.80        14

    accuracy                           0.92        79
   macro avg       0.86      0.90      0.88        79
weighted avg       0.93      0.92      0.93        79

Saved best Greek model: outputs\ancient_greek_bert_best.pt


C:\Users\Asoulin_Sapir\AppData\Local\Temp\ipykernel_26304\2938237305.py:103: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):



[Greek] Epoch 2/3 | train_loss=0.1788

=== GREEK VAL ===
              precision    recall  f1-score   support

    Original       0.97      0.94      0.95        65
     Rewrite       0.75      0.86      0.80        14

    accuracy                           0.92        79
   macro avg       0.86      0.90      0.88        79
weighted avg       0.93      0.92      0.93        79



C:\Users\Asoulin_Sapir\AppData\Local\Temp\ipykernel_26304\2938237305.py:103: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):



[Greek] Epoch 3/3 | train_loss=0.0462

=== GREEK VAL ===
              precision    recall  f1-score   support

    Original       0.95      0.97      0.96        65
     Rewrite       0.85      0.79      0.81        14

    accuracy                           0.94        79
   macro avg       0.90      0.88      0.89        79
weighted avg       0.94      0.94      0.94        79

Saved best Greek model: outputs\ancient_greek_bert_best.pt

=== GREEK TEST ===
              precision    recall  f1-score   support

    Original       0.91      0.95      0.93        66
     Rewrite       0.73      0.57      0.64        14

    accuracy                           0.89        80
   macro avg       0.82      0.76      0.79        80
weighted avg       0.88      0.89      0.88        80



In [19]:
train_en_sep, temp_en_sep = train_test_split(df_en, test_size=0.2, random_state=SEED, stratify=df_en["y"])
val_en_sep, test_en_sep   = train_test_split(temp_en_sep, test_size=0.5, random_state=SEED, stratify=temp_en_sep["y"])

print(len(train_en_sep), len(val_en_sep), len(test_en_sep))
print(train_en_sep["y"].value_counts())


6032 754 754
y
1    5808
0     224
Name: count, dtype: int64


## 12) מודל נפרד לאנגלית: RoBERTa / BERT

In [20]:
EN_MODEL_NAME_SEP = "roberta-base"   # אפשר גם "bert-base-uncased"
MAX_LEN_EN_SEP = 256
BATCH_EN_SEP = 8
EPOCHS_EN_SEP = 3
LR_EN_SEP = 2e-5

tok_en_sep = AutoTokenizer.from_pretrained(EN_MODEL_NAME_SEP)

def collate_en(batch):
    texts = [b["text"] for b in batch]
    y = torch.tensor([b["y"] for b in batch], dtype=torch.long)
    tok = tok_en_sep(texts, padding=True, truncation=True, max_length=MAX_LEN_EN_SEP, return_tensors="pt")
    return tok, y

train_loader_en_sep = DataLoader(SimpleTextDS(train_en_sep), batch_size=BATCH_EN_SEP, shuffle=True, collate_fn=collate_en)
val_loader_en_sep   = DataLoader(SimpleTextDS(val_en_sep),   batch_size=BATCH_EN_SEP, shuffle=False, collate_fn=collate_en)
test_loader_en_sep  = DataLoader(SimpleTextDS(test_en_sep),  batch_size=BATCH_EN_SEP, shuffle=False, collate_fn=collate_en)

class EnglishClassifier(nn.Module):
    def __init__(self, model_name, dropout=0.1, n_classes=2):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        h = self.encoder.config.hidden_size
        self.drop = nn.Dropout(dropout)
        self.fc = nn.Linear(h, n_classes)
    def forward(self, tok):
        out = self.encoder(**tok, return_dict=True)
        emb = mean_pool(out.last_hidden_state, tok["attention_mask"])
        return self.fc(self.drop(emb))

model_en = EnglishClassifier(EN_MODEL_NAME_SEP).to(DEVICE)

# class weights for English (very imbalanced אצלך)
y_counts_en = train_en_sep["y"].value_counts().sort_index()
w0 = 1.0 / max(1, int(y_counts_en.get(0, 1)))
w1 = 1.0 / max(1, int(y_counts_en.get(1, 1)))
class_w_en = torch.tensor([w0, w1], dtype=torch.float32, device=DEVICE)
class_w_en = class_w_en / class_w_en.sum() * 2.0
print("English class weights:", class_w_en.detach().cpu().numpy())

opt_en = AdamW(model_en.parameters(), lr=LR_EN_SEP, weight_decay=WEIGHT_DECAY)
steps_en = len(train_loader_en_sep) * EPOCHS_EN_SEP
sched_en = get_linear_schedule_with_warmup(opt_en, int(0.05*steps_en), steps_en)
scaler_en = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

def eval_loader_en(loader, name="VAL"):
    model_en.eval()
    ys, ps = [], []
    with torch.no_grad():
        for tok, y in loader:
            tok = {k:v.to(DEVICE) for k,v in tok.items()}
            y = y.to(DEVICE)
            logits = model_en(tok)
            pred = torch.argmax(logits, dim=-1)
            ys.append(y.cpu().numpy())
            ps.append(pred.cpu().numpy())
    y_true = np.concatenate(ys); y_pred = np.concatenate(ps)
    print(f"\n=== ENGLISH {name} ===")
    print(classification_report(y_true, y_pred, target_names=["Original","Rewrite"], zero_division=0))
    return y_true, y_pred

best_f1_en = -1
best_path_en = os.path.join(OUT_DIR, "roberta_en_best.pt")

for ep in range(1, EPOCHS_EN_SEP+1):
    model_en.train()
    tot = 0.0
    for tok, y in train_loader_en_sep:
        tok = {k:v.to(DEVICE) for k,v in tok.items()}
        y = y.to(DEVICE)
        opt_en.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            logits = model_en(tok)
            loss = F.cross_entropy(logits, y, weight=class_w_en)

        scaler_en.scale(loss).backward()
        scaler_en.step(opt_en)
        scaler_en.update()
        sched_en.step()
        tot += float(loss.detach().cpu().item())

    print(f"\n[English] Epoch {ep}/{EPOCHS_EN_SEP} | train_loss={tot/max(1,len(train_loader_en_sep)):.4f}")
    yv, pv = eval_loader_en(val_loader_en_sep, "VAL")
    rep = classification_report(yv, pv, output_dict=True, zero_division=0)
    macro_f1 = rep["macro avg"]["f1-score"]
    if macro_f1 > best_f1_en:
        best_f1_en = macro_f1
        torch.save(model_en.state_dict(), best_path_en)
        print("Saved best English model:", best_path_en)

model_en.load_state_dict(torch.load(best_path_en, map_location=DEVICE))
_ = eval_loader_en(test_loader_en_sep, "TEST")


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


English class weights: [1.9257294  0.07427055]


C:\Users\Asoulin_Sapir\AppData\Local\Temp\ipykernel_26304\3530508770.py:44: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler_en = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())
C:\Users\Asoulin_Sapir\AppData\Local\Temp\ipykernel_26304\3530508770.py:73: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):



[English] Epoch 1/3 | train_loss=0.1509

=== ENGLISH VAL ===
              precision    recall  f1-score   support

    Original       0.64      0.96      0.77        28
     Rewrite       1.00      0.98      0.99       726

    accuracy                           0.98       754
   macro avg       0.82      0.97      0.88       754
weighted avg       0.99      0.98      0.98       754

Saved best English model: outputs\roberta_en_best.pt


C:\Users\Asoulin_Sapir\AppData\Local\Temp\ipykernel_26304\3530508770.py:73: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):



[English] Epoch 2/3 | train_loss=0.0348

=== ENGLISH VAL ===
              precision    recall  f1-score   support

    Original       0.79      0.96      0.87        28
     Rewrite       1.00      0.99      0.99       726

    accuracy                           0.99       754
   macro avg       0.90      0.98      0.93       754
weighted avg       0.99      0.99      0.99       754

Saved best English model: outputs\roberta_en_best.pt


C:\Users\Asoulin_Sapir\AppData\Local\Temp\ipykernel_26304\3530508770.py:73: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):



[English] Epoch 3/3 | train_loss=0.0119

=== ENGLISH VAL ===
              precision    recall  f1-score   support

    Original       0.87      0.93      0.90        28
     Rewrite       1.00      0.99      1.00       726

    accuracy                           0.99       754
   macro avg       0.93      0.96      0.95       754
weighted avg       0.99      0.99      0.99       754

Saved best English model: outputs\roberta_en_best.pt

=== ENGLISH TEST ===
              precision    recall  f1-score   support

    Original       0.93      0.96      0.95        28
     Rewrite       1.00      1.00      1.00       726

    accuracy                           1.00       754
   macro avg       0.96      0.98      0.97       754
weighted avg       1.00      1.00      1.00       754



## E) Experiment 1 — XLM-R + Adversarial Language Confusion (GRL)
---


תא 1 — Imports + Dataset + Sampler מאוזן לפי (lang,y)

In [21]:
import os, numpy as np, torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.metrics import classification_report

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

MODEL_NAME = "xlm-roberta-base"
MAX_LEN = 256
BATCH = 8
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

LANG2ID = {"en": 0, "gr": 1}

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class TextLangDS(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        r = self.df.iloc[idx]
        return {
            "text": str(r["text"]),
            "y": int(r["y"]),
            "lang": LANG2ID[str(r["lang"])]
        }

def collate_fn(batch):
    texts = [b["text"] for b in batch]
    y = torch.tensor([b["y"] for b in batch], dtype=torch.long)
    lang = torch.tensor([b["lang"] for b in batch], dtype=torch.long)
    tok = tokenizer(texts, padding=True, truncation=True, max_length=MAX_LEN, return_tensors="pt")
    return tok, y, lang

# balance training batches by stratum = lang_y
strata = train_all["lang"].astype(str) + "_" + train_all["y"].astype(str)
counts = strata.value_counts()
weights = strata.map(lambda s: 1.0 / counts[s]).values
sampler = WeightedRandomSampler(torch.tensor(weights, dtype=torch.double), num_samples=len(weights), replacement=True)

train_loader = DataLoader(TextLangDS(train_all), batch_size=BATCH, sampler=sampler, collate_fn=collate_fn)
val_loader   = DataLoader(TextLangDS(val_all),   batch_size=BATCH, shuffle=False, collate_fn=collate_fn)
test_loader  = DataLoader(TextLangDS(test_all),  batch_size=BATCH, shuffle=False, collate_fn=collate_fn)

print("Train strata counts:\n", counts)


Device: cuda
Train strata counts:
 en_1    5808
gr_0     523
en_0     224
gr_1     113
Name: count, dtype: int64


תא 2 — מודל עם GRL: Encoder + Rewrite head + Language head

In [22]:
def mean_pool(last_hidden, attention_mask):
    mask = attention_mask.unsqueeze(-1).type_as(last_hidden)
    summed = (last_hidden * mask).sum(dim=1)
    denom = mask.sum(dim=1).clamp(min=1e-6)
    return summed / denom

class GradReverse(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, alpha):
        ctx.alpha = alpha
        return x.view_as(x)
    @staticmethod
    def backward(ctx, grad_output):
        return -ctx.alpha * grad_output, None

def grl(x, alpha=1.0):
    return GradReverse.apply(x, alpha)

class XLMR_Adv(nn.Module):
    def __init__(self, model_name, n_classes=2, n_lang=2, dropout=0.1):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        h = self.encoder.config.hidden_size
        self.drop = nn.Dropout(dropout)
        self.rewrite_head = nn.Linear(h, n_classes)
        self.lang_head    = nn.Linear(h, n_lang)

    def forward(self, tok, grl_alpha=1.0):
        out = self.encoder(**tok, return_dict=True)
        emb = mean_pool(out.last_hidden_state, tok["attention_mask"])
        emb_d = self.drop(emb)

        logits_y = self.rewrite_head(emb_d)
        logits_lang = self.lang_head(self.drop(grl(emb, grl_alpha)))
        return logits_y, logits_lang

model = XLMR_Adv(MODEL_NAME).to(DEVICE)
print("Model ready.")


Model ready.


תא 3 — אימון + הערכה (כולל דוח לכל שפה)

In [23]:
LR = 2e-5
WEIGHT_DECAY = 0.01
EPOCHS = 3
LAMBDA_LANG = 0.3   # משקל האדברסרי (תתחילי 0.1–0.5)
GRL_ALPHA = 1.0     # עוצמת היפוך גרדיאנט

optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, int(0.05*total_steps), total_steps)
scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

@torch.no_grad()
def run_eval(loader, split="VAL"):
    model.eval()
    ys, ps, langs = [], [], []
    for tok, y, lang in loader:
        tok = {k: v.to(DEVICE) for k, v in tok.items()}
        y = y.to(DEVICE)
        lang = lang.to(DEVICE)

        logits_y, logits_lang = model(tok, grl_alpha=0.0)  # eval: בלי GRL
        pred = torch.argmax(logits_y, dim=-1)

        ys.append(y.cpu().numpy())
        ps.append(pred.cpu().numpy())
        langs.append(lang.cpu().numpy())

    y_true = np.concatenate(ys)
    y_pred = np.concatenate(ps)
    lang_arr = np.concatenate(langs)

    print(f"\n=== {split} OVERALL ===")
    print(classification_report(y_true, y_pred, target_names=["Original","Rewrite"], zero_division=0))

    for name, lid in LANG2ID.items():
        m = (lang_arr == lid)
        if m.sum() == 0:
            continue
        print(f"\n=== {split} | {name.upper()} (n={m.sum()}) ===")
        print(classification_report(y_true[m], y_pred[m], target_names=["Original","Rewrite"], zero_division=0))

    rep = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    return rep["macro avg"]["f1-score"]

best_f1 = -1.0
best_path = os.path.join("outputs", "xlmr_adv_best.pt")
os.makedirs("outputs", exist_ok=True)

for ep in range(1, EPOCHS+1):
    model.train()
    total_loss = 0.0

    for tok, y, lang in train_loader:
        tok = {k: v.to(DEVICE) for k, v in tok.items()}
        y = y.to(DEVICE)
        lang = lang.to(DEVICE)

        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            logits_y, logits_lang = model(tok, grl_alpha=GRL_ALPHA)

            loss_y = F.cross_entropy(logits_y, y)
            loss_lang = F.cross_entropy(logits_lang, lang)

            loss = loss_y + LAMBDA_LANG * loss_lang

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        total_loss += float(loss.detach().cpu().item())

    print(f"\nEpoch {ep}/{EPOCHS} | train_loss={total_loss/max(1,len(train_loader)):.4f}")
    f1 = run_eval(val_loader, "VAL")
    print("VAL macro-F1:", round(f1, 4))

    if f1 > best_f1:
        best_f1 = f1
        torch.save(model.state_dict(), best_path)
        print("✅ Saved best model:", best_path)

# Final test
model.load_state_dict(torch.load(best_path, map_location=DEVICE))
_ = run_eval(test_loader, "TEST")


C:\Users\Asoulin_Sapir\AppData\Local\Temp\ipykernel_26304\3238149725.py:10: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())
C:\Users\Asoulin_Sapir\AppData\Local\Temp\ipykernel_26304\3238149725.py:60: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):



Epoch 1/3 | train_loss=2.7590

=== VAL OVERALL ===
              precision    recall  f1-score   support

    Original       0.85      0.95      0.89        93
     Rewrite       0.99      0.98      0.99       740

    accuracy                           0.97       833
   macro avg       0.92      0.96      0.94       833
weighted avg       0.98      0.97      0.98       833


=== VAL | EN (n=754) ===
              precision    recall  f1-score   support

    Original       0.64      0.96      0.77        28
     Rewrite       1.00      0.98      0.99       726

    accuracy                           0.98       754
   macro avg       0.82      0.97      0.88       754
weighted avg       0.99      0.98      0.98       754


=== VAL | GR (n=79) ===
              precision    recall  f1-score   support

    Original       0.98      0.94      0.96        65
     Rewrite       0.76      0.93      0.84        14

    accuracy                           0.94        79
   macro avg       0.87  

C:\Users\Asoulin_Sapir\AppData\Local\Temp\ipykernel_26304\3238149725.py:60: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):



Epoch 2/3 | train_loss=0.8111

=== VAL OVERALL ===
              precision    recall  f1-score   support

    Original       0.92      0.92      0.92        93
     Rewrite       0.99      0.99      0.99       740

    accuracy                           0.98       833
   macro avg       0.96      0.96      0.96       833
weighted avg       0.98      0.98      0.98       833


=== VAL | EN (n=754) ===
              precision    recall  f1-score   support

    Original       0.81      0.89      0.85        28
     Rewrite       1.00      0.99      0.99       726

    accuracy                           0.99       754
   macro avg       0.90      0.94      0.92       754
weighted avg       0.99      0.99      0.99       754


=== VAL | GR (n=79) ===
              precision    recall  f1-score   support

    Original       0.98      0.94      0.96        65
     Rewrite       0.76      0.93      0.84        14

    accuracy                           0.94        79
   macro avg       0.87  

C:\Users\Asoulin_Sapir\AppData\Local\Temp\ipykernel_26304\3238149725.py:60: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):



Epoch 3/3 | train_loss=0.3136

=== VAL OVERALL ===
              precision    recall  f1-score   support

    Original       0.97      0.90      0.93        93
     Rewrite       0.99      1.00      0.99       740

    accuracy                           0.99       833
   macro avg       0.98      0.95      0.96       833
weighted avg       0.99      0.99      0.99       833


=== VAL | EN (n=754) ===
              precision    recall  f1-score   support

    Original       0.92      0.82      0.87        28
     Rewrite       0.99      1.00      1.00       726

    accuracy                           0.99       754
   macro avg       0.96      0.91      0.93       754
weighted avg       0.99      0.99      0.99       754


=== VAL | GR (n=79) ===
              precision    recall  f1-score   support

    Original       0.98      0.94      0.96        65
     Rewrite       0.76      0.93      0.84        14

    accuracy                           0.94        79
   macro avg       0.87  

In [24]:
import numpy as np
import torch
from sklearn.metrics import classification_report, f1_score

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

@torch.no_grad()
def collect_probs(loader):
    """
    Returns:
      probs: p(Rewrite|x) array shape [N]
      y_true: true labels [N]
      lang_arr: lang ids [N]
    Assumes model(tok) returns logits [B,2] OR (logits_y, logits_lang).
    """
    model.eval()
    probs_list, y_list, lang_list = [], [], []

    for tok, y, lang_id in loader:
        tok = {k: v.to(DEVICE) for k, v in tok.items()}
        y = y.to(DEVICE)
        lang_id = lang_id.to(DEVICE)

        out = model(tok)  # baseline returns logits; adv may return tuple
        logits = out[0] if isinstance(out, (tuple, list)) else out

        p = torch.softmax(logits, dim=-1)[:, 1]  # class 1 = Rewrite
        probs_list.append(p.detach().cpu().numpy())
        y_list.append(y.detach().cpu().numpy())
        lang_list.append(lang_id.detach().cpu().numpy())

    probs = np.concatenate(probs_list)
    y_true = np.concatenate(y_list)
    lang_arr = np.concatenate(lang_list)
    return probs, y_true, lang_arr

def eval_with_tau(probs, y_true, tau):
    y_pred = (probs >= tau).astype(int)
    rep = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    return rep["macro avg"]["f1-score"], y_pred

def print_reports(probs, y_true, lang_arr, tau, split_name="VAL"):
    y_pred = (probs >= tau).astype(int)

    print(f"\n=== {split_name} @ tau={tau:.3f} OVERALL ===")
    print(classification_report(y_true, y_pred, target_names=["Original","Rewrite"], zero_division=0))

    # per language
    inv = {v:k for k,v in LANG2ID.items()}
    for lid in np.unique(lang_arr):
        mask = (lang_arr == lid)
        name = inv.get(int(lid), str(lid)).upper()
        print(f"\n=== {split_name} @ tau={tau:.3f} | {name} (n={mask.sum()}) ===")
        print(classification_report(y_true[mask], y_pred[mask], target_names=["Original","Rewrite"], zero_division=0))

def find_best_tau(probs, y_true, grid=None):
    if grid is None:
        grid = np.linspace(0.05, 0.95, 91)  # step 0.01
    best_tau, best_f1 = 0.5, -1
    for tau in grid:
        f1, _ = eval_with_tau(probs, y_true, tau)
        if f1 > best_f1:
            best_f1 = f1
            best_tau = float(tau)
    return best_tau, best_f1

# ---- 1) Tune tau on VAL ----
val_probs, val_y, val_lang = collect_probs(val_loader)

tau_default = 0.5
f1_default, _ = eval_with_tau(val_probs, val_y, tau_default)

tau_star, f1_star = find_best_tau(val_probs, val_y)

print("VAL macro-F1 @ tau=0.5 :", round(f1_default, 4))
print("VAL best tau*         :", round(tau_star, 3))
print("VAL macro-F1 @ tau*   :", round(f1_star, 4))

print_reports(val_probs, val_y, val_lang, tau_default, "VAL (default)")
print_reports(val_probs, val_y, val_lang, tau_star,    "VAL (tuned)")

# ---- 2) Evaluate same tau* on TEST ----
test_probs, test_y, test_lang = collect_probs(test_loader)
print_reports(test_probs, test_y, test_lang, tau_star, "TEST (tuned)")


VAL macro-F1 @ tau=0.5 : 0.9626
VAL best tau*         : 0.86
VAL macro-F1 @ tau*   : 0.9692

=== VAL (default) @ tau=0.500 OVERALL ===
              precision    recall  f1-score   support

    Original       0.97      0.90      0.93        93
     Rewrite       0.99      1.00      0.99       740

    accuracy                           0.99       833
   macro avg       0.98      0.95      0.96       833
weighted avg       0.99      0.99      0.99       833


=== VAL (default) @ tau=0.500 | EN (n=754) ===
              precision    recall  f1-score   support

    Original       0.92      0.82      0.87        28
     Rewrite       0.99      1.00      1.00       726

    accuracy                           0.99       754
   macro avg       0.96      0.91      0.93       754
weighted avg       0.99      0.99      0.99       754


=== VAL (default) @ tau=0.500 | GR (n=79) ===
              precision    recall  f1-score   support

    Original       0.98      0.94      0.96        65
     Re

## F) Preprocessing ליוונית: CLTK + Stanza (Lemma/POS)
---


In [25]:
import re, inspect

# 1) Verify stanza exists (CLTK uses it as backend)
try:
    import stanza
except Exception as e:
    raise ImportError(
        "stanza is not installed. Run:\n"
        "  pip install stanza \"cltk[stanza]\"\n"
        "Then restart the kernel."
    ) from e

# 2) Initialize CLTK NLP for Ancient Greek robustly
from cltk import NLP

def make_cltk_nlp_grc():
    sig = inspect.signature(NLP.__init__)
    params = sig.parameters
    kwargs = {}

    # prefer stanza backend if supported
    if "backend" in params:
        kwargs["backend"] = "stanza"
    if "suppress_banner" in params:
        kwargs["suppress_banner"] = True
    if "quiet" in params:
        kwargs["quiet"] = True

    # pass language code safely
    if "language" in params:
        return NLP(language="grc", **kwargs)
    if "lang" in params:
        return NLP(lang="grc", **kwargs)
    return NLP("grc", **kwargs)

cltk_grc = make_cltk_nlp_grc()
print("✅ cltk_grc initialized:", type(cltk_grc))


✅ cltk_grc initialized: <class 'cltk.nlp.NLP'>


In [26]:
import inspect

# 0) sanity: show we're in the same kernel
import sys
print("Python:", sys.executable)

# 1) stanza must exist (CLTK backend)
try:
    import stanza
    print("✅ stanza imported")
except Exception as e:
    raise ImportError(
        "stanza is missing. Run:\n"
        "  pip install stanza \"cltk[stanza]\"\n"
        "Then restart the kernel."
    ) from e

# 2) CLTK NLP import
from cltk import NLP
print("✅ CLTK NLP imported:", NLP)

# 3) Build cltk_grc robustly
def init_cltk_grc():
    sig = inspect.signature(NLP.__init__)
    params = sig.parameters
    kwargs = {}

    # prefer stanza backend if supported
    if "backend" in params:
        kwargs["backend"] = "stanza"
    if "suppress_banner" in params:
        kwargs["suppress_banner"] = True
    if "quiet" in params:
        kwargs["quiet"] = True

    # try different init signatures
    if "language" in params:
        return NLP(language="grc", **kwargs)
    if "lang" in params:
        return NLP(lang="grc", **kwargs)
    return NLP("grc", **kwargs)

cltk_grc = init_cltk_grc()
print("✅ cltk_grc created. Type:", type(cltk_grc))


Python: C:\Users\Asoulin_Sapir\anaconda3\python.exe
✅ stanza imported
✅ CLTK NLP imported: <class 'cltk.nlp.NLP'>
✅ cltk_grc created. Type: <class 'cltk.nlp.NLP'>


תא: יצירת לממטיזר + פונקציות עזר

In [27]:
import re, inspect
from cltk import NLP

def get_cltk_grc():
    # אם כבר קיים - מחזירים
    if "cltk_grc" in globals() and cltk_grc is not None:
        return cltk_grc

    sig = inspect.signature(NLP.__init__)
    params = sig.parameters
    kwargs = {}

    if "backend" in params:
        kwargs["backend"] = "stanza"
    if "suppress_banner" in params:
        kwargs["suppress_banner"] = True
    if "quiet" in params:
        kwargs["quiet"] = True

    if "language" in params:
        obj = NLP(language="grc", **kwargs)
    elif "lang" in params:
        obj = NLP(lang="grc", **kwargs)
    else:
        obj = NLP("grc", **kwargs)

    globals()["cltk_grc"] = obj
    return obj


def grc_lemmatize_sentence(text: str) -> str:
    nlp = get_cltk_grc()

    text = re.sub(r"\s+", " ", str(text)).strip()
    if not text:
        return ""

    doc = nlp.analyze(text)

    if hasattr(doc, "tokens"):
        lemmas = []
        for tok in doc.tokens:
            lem = getattr(tok, "lemma", None)
            lemmas.append(str(lem) if lem else str(getattr(tok, "string", tok)))
        return " ".join(lemmas)

    if hasattr(doc, "words"):
        lemmas = []
        for w in doc.words:
            lem = getattr(w, "lemma", None)
            lemmas.append(str(lem) if lem else str(getattr(w, "string", w)))
        return " ".join(lemmas)

    return " ".join(text.split())


def grc_lemmatize_texts(texts, verbose_every=200):
    out = []
    for i, t in enumerate(texts):
        out.append(grc_lemmatize_sentence(t))
        if verbose_every and (i+1) % verbose_every == 0:
            print(f"GRC lemmatized: {i+1}/{len(texts)}")
    return out


# quick test
print(grc_lemmatize_sentence("Ἀριστοτέλης ἐστὶ φιλόσοφος."))


2026-02-15 07:43:10 INFO: "ancient_greek" is an alias for "grc"
2026-02-15 07:43:11 INFO: Loading these models for language: grc (Ancient_Greek):
| Processor | Package          |
--------------------------------
| tokenize  | perseus          |
| pos       | perseus_nocharlm |
| lemma     | perseus_nocharlm |
| depparse  | perseus_nocharlm |

2026-02-15 07:43:11 INFO: Using device: cuda
2026-02-15 07:43:11 INFO: Loading: tokenize
2026-02-15 07:43:11 INFO: Loading: pos
2026-02-15 07:43:11 INFO: Loading: lemma
2026-02-15 07:43:11 INFO: Loading: depparse
2026-02-15 07:43:12 INFO: Done loading processors!


Ἀριστοτέλης εἰμί φιλόσοφος .


## G) Experiment 1b — Multi-input: XLM-R + POS/Lemma (hashed) + GRL
---


תא 1 — יצירת POS/Lemma לשתי השפות + שמירה לקובץ (Caching)

In [28]:
import os
import re
import pandas as pd
import numpy as np

# =========================
# 1) English POS+Lemma (spaCy)  — (כמו במחברת הניסויים שלך)
# =========================
import spacy
nlp = spacy.load("en_core_web_sm", disable=["ner", "parser"])

def en_pos_lemma(texts, batch_size=256):
    pos_out, lem_out = [], []
    for doc in nlp.pipe([str(t) for t in texts], batch_size=batch_size):
        pos_out.append(" ".join([t.pos_ for t in doc]))
        lem_out.append(" ".join([t.lemma_.lower() for t in doc]))
    return pos_out, lem_out


# =========================
# 2) Greek POS tagger (AUEB Java POStagger) — (כמו במחברת הניסויים שלך)
# =========================
import subprocess
from tqdm import tqdm

TAGGER_ROOT = str(BASE_DIR / "pos_tagger")
JAR_BIN_DIR = os.path.join(TAGGER_ROOT, "bin")
JAR_PATH    = os.path.join(JAR_BIN_DIR, "POStagger.jar")

def check_java_and_jar():
    if not os.path.exists(JAR_PATH):
        raise FileNotFoundError(
            f"POStagger.jar not found at: {JAR_PATH}\n"
            "Expected: ./pos_tagger/bin/POStagger.jar (same as your experiments notebook)"
        )
    try:
        out = subprocess.check_output(["java", "-version"], stderr=subprocess.STDOUT, text=True)
        print("Java OK:", out.strip().splitlines()[0])
    except Exception as e:
        raise RuntimeError("Java is not working (java not found).") from e

def greek_pos_tag_sentences(sentences):
    """
    Returns list[str] POS tags (space-separated) per sentence.
    Based on your experiments notebook (writes temp file, reads result.txt).
    """
    check_java_and_jar()

    temp_dir = os.environ.get("TEMP", os.getcwd())
    temp_inp = os.path.join(temp_dir, "temp_to_be_tagged.txt")

    all_pos = []
    for sentence in tqdm([str(s) for s in sentences], desc="Greek POS tagging", total=len(sentences)):
        with open(temp_inp, "w", encoding="utf-8") as f:
            f.write(sentence)

        proc = subprocess.run(
            ["java", "-jar", "POStagger.jar", "0", temp_inp],
            cwd=JAR_BIN_DIR,
            text=True,
            capture_output=True
        )

        if proc.returncode != 0:
            print("\n❌ POStagger failed on sentence:")
            print(sentence[:500])
            print("\n--- stdout ---\n", proc.stdout[:2000])
            print("\n--- stderr ---\n", proc.stderr[:2000])
            raise RuntimeError(f"POStagger failed with return code {proc.returncode}")

        result_file = os.path.join(JAR_BIN_DIR, "result.txt")
        with open(result_file, "r", encoding="utf-8") as f:
            result = f.read()

        tags = []
        for line in result.split("\n"):
            if line.strip():
                parts = line.split()
                if len(parts) >= 2:
                    tags.append(parts[1])

        # במחברת שלך היה ", ". פה הופכים לרווחים כדי להתאים ל-vocab tokenization
        all_pos.append(" ".join(tags))

    try:
        os.remove(temp_inp)
    except OSError:
        pass

    return all_pos


# =========================
# 3) Lemma ליוונית: להשתמש אם קיים, אחרת fallback
# =========================
def greek_lemma_fallback(texts):
    # fallback שמרני: lowercase + טוקניזציה בסיסית
    # (לא lemmatization אמיתי; מסומן בתזה כ-fallback)
    out = []
    for t in texts:
        s = str(t).lower()
        s = re.sub(r"[^\w\s\u0370-\u03FF\u1F00-\u1FFF]", " ", s)  # שומר יוונית + לטיני
        out.append(" ".join(s.split()))
    return out


# =========================
# 4) Build POS/Lemma columns for train/val/test (cache)
# =========================
CACHE_DIR = os.path.join("outputs", "poslemma_cache")
os.makedirs(CACHE_DIR, exist_ok=True)

def add_poslemma_to_split(df, split_name):
    """
    df must have columns: lang, text
    Returns df with new columns: pos, lemma
    Caches to outputs/poslemma_cache/{split_name}_with_poslemma.parquet
    """
    cache_fp = os.path.join(CACHE_DIR, f"{split_name}_with_poslemma.parquet")
    if os.path.exists(cache_fp):
        print("✅ Loading cached:", cache_fp)
        return pd.read_parquet(cache_fp)

    df = df.copy()
    if "lang" not in df.columns or "text" not in df.columns:
        raise ValueError(f"{split_name}: expected columns ['lang','text']")

    df["pos"] = ""
    df["lemma"] = ""

    # ---- English
    m_en = df["lang"].astype(str).str.lower().eq("en")
    if m_en.any():
        en_texts = df.loc[m_en, "text"].fillna("").astype(str).tolist()
        pos_en, lem_en = en_pos_lemma(en_texts)
        df.loc[m_en, "pos"] = pos_en
        df.loc[m_en, "lemma"] = lem_en

    # ---- Greek
    m_gr = df["lang"].astype(str).str.lower().eq("gr")
    if m_gr.any():
        gr_texts = df.loc[m_gr, "text"].fillna("").astype(str).tolist()
        df.loc[m_gr, "lemma"] = grc_lemmatize_texts(gr_texts, verbose_every=500)

        # POS: אם כבר קיימת עמודה Pos ב-df המקורי (לפעמים מגיע מהאקסלים), נשתמש בה
        # אבל כאן אנחנו עובדים על df שכבר נורמל ל-text. אז אם יש בעמודות המקוריות:
        if "Pos" in df.columns and df.loc[m_gr, "Pos"].notna().any():
            df.loc[m_gr, "pos"] = (
                df.loc[m_gr, "Pos"].fillna("").astype(str)
                .str.replace(",", " ", regex=False)
                .str.replace(r"\s+", " ", regex=True).str.strip()
            )
        else:
            # אחרת נריץ Java tagger (כמו במחברת שלך)
            df.loc[m_gr, "pos"] = greek_pos_tag_sentences(gr_texts)

        # Lemma: אם יש Lema/Lemma קיימת—נשתמש. אחרת fallback.
        if "Lema" in df.columns and df.loc[m_gr, "Lema"].notna().any():
            df.loc[m_gr, "lemma"] = (
                df.loc[m_gr, "Lema"].fillna("").astype(str)
                .str.replace(",", " ", regex=False)
                .str.replace(r"\s+", " ", regex=True).str.strip()
            )
        elif "Lemma" in df.columns and df.loc[m_gr, "Lemma"].notna().any():
            df.loc[m_gr, "lemma"] = (
                df.loc[m_gr, "Lemma"].fillna("").astype(str)
                .str.replace(",", " ", regex=False)
                .str.replace(r"\s+", " ", regex=True).str.strip()
            )
        else:
            df.loc[m_gr, "lemma"] = greek_lemma_fallback(gr_texts)

    # cache
    df.to_parquet(cache_fp, index=False)
    print("✅ Saved cache:", cache_fp)
    return df

train_all_pl = add_poslemma_to_split(train_all, "train_all")
val_all_pl   = add_poslemma_to_split(val_all,   "val_all")
test_all_pl  = add_poslemma_to_split(test_all,  "test_all")

print(train_all_pl[["lang","y"]].value_counts())
print(train_all_pl.columns)


✅ Loading cached: outputs\poslemma_cache\train_all_with_poslemma.parquet
✅ Loading cached: outputs\poslemma_cache\val_all_with_poslemma.parquet
✅ Loading cached: outputs\poslemma_cache\test_all_with_poslemma.parquet
lang  y
en    1    5808
gr    0     523
en    0     224
gr    1     113
Name: count, dtype: int64
Index(['lang', 'text', 'y', 'book_name', 'chapter_number', 'paragraph_number',
       'sentence_number', 'sentence_id', 'Book', 'Paragraph', 'HP Source',
       'sent_idx_in_par', 'pos', 'lemma'],
      dtype='object')


תא 2 — בניית vocab ל-POS ול-Lemma (על train בלבד)

In [29]:
from collections import Counter

def build_vocab_from_space_tokens(series, min_freq=1):
    c = Counter()
    for s in series.fillna("").astype(str).tolist():
        toks = s.split()
        c.update(toks)
    vocab = {"<PAD>": 0, "<UNK>": 1}
    for tok, f in c.items():
        if f >= min_freq and tok not in vocab:
            vocab[tok] = len(vocab)
    return vocab

pos_vocab = build_vocab_from_space_tokens(train_all_pl["pos"], min_freq=1)
lem_vocab = build_vocab_from_space_tokens(train_all_pl["lemma"], min_freq=2)

print("POS vocab size:", len(pos_vocab))
print("LEM vocab size:", len(lem_vocab))


POS vocab size: 31
LEM vocab size: 4873


תא 3 — Dataset + DataLoaders (כולל WeightedRandomSampler לפי lang_y)

In [30]:
import torch
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

LANG2ID = {"en": 0, "gr": 1}

def encode_space_tokens(s, vocab, max_len=256):
    toks = str(s).split()
    ids = [vocab.get(t, vocab["<UNK>"]) for t in toks[:max_len]]
    if not ids:
        ids = [vocab["<PAD>"]]
    return ids

class MultiInputDS(Dataset):
    def __init__(self, df, pos_vocab, lem_vocab, max_pos_len=256, max_lem_len=256):
        self.df = df.reset_index(drop=True)
        self.pos_vocab = pos_vocab
        self.lem_vocab = lem_vocab
        self.max_pos_len = max_pos_len
        self.max_lem_len = max_lem_len

    def __len__(self): 
        return len(self.df)

    def __getitem__(self, idx):
        r = self.df.iloc[idx]
        return {
            "text": str(r["text"]),
            "pos_ids": encode_space_tokens(r.get("pos",""), self.pos_vocab, self.max_pos_len),
            "lem_ids": encode_space_tokens(r.get("lemma",""), self.lem_vocab, self.max_lem_len),
            "y": int(r["y"]),
            "lang": LANG2ID[str(r["lang"]).lower()]
        }

from transformers import AutoTokenizer
MODEL_NAME = "xlm-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

MAX_LEN = 256
BATCH = 8

def collate_multi(batch):
    texts = [b["text"] for b in batch]
    y = torch.tensor([b["y"] for b in batch], dtype=torch.long)
    lang = torch.tensor([b["lang"] for b in batch], dtype=torch.long)

    tok = tokenizer(texts, padding=True, truncation=True, max_length=MAX_LEN, return_tensors="pt")

    def pad_2d(list_ids, pad_id=0):
        maxl = max(len(x) for x in list_ids)
        arr = torch.full((len(list_ids), maxl), pad_id, dtype=torch.long)
        mask = torch.zeros((len(list_ids), maxl), dtype=torch.long)
        for i,x in enumerate(list_ids):
            arr[i, :len(x)] = torch.tensor(x, dtype=torch.long)
            mask[i, :len(x)] = 1
        return arr, mask

    pos_ids, pos_mask = pad_2d([b["pos_ids"] for b in batch], pad_id=pos_vocab["<PAD>"])
    lem_ids, lem_mask = pad_2d([b["lem_ids"] for b in batch], pad_id=lem_vocab["<PAD>"])
    return tok, pos_ids, pos_mask, lem_ids, lem_mask, y, lang


# Balanced sampler by lang_y
strata = train_all_pl["lang"].astype(str).str.lower() + "_" + train_all_pl["y"].astype(str)
counts = strata.value_counts()
weights = strata.map(lambda s: 1.0 / counts[s]).values
sampler = WeightedRandomSampler(torch.tensor(weights, dtype=torch.double),
                                num_samples=len(weights), replacement=True)

train_loader = DataLoader(MultiInputDS(train_all_pl, pos_vocab, lem_vocab),
                          batch_size=BATCH, sampler=sampler, collate_fn=collate_multi)
val_loader   = DataLoader(MultiInputDS(val_all_pl, pos_vocab, lem_vocab),
                          batch_size=BATCH, shuffle=False, collate_fn=collate_multi)
test_loader  = DataLoader(MultiInputDS(test_all_pl, pos_vocab, lem_vocab),
                          batch_size=BATCH, shuffle=False, collate_fn=collate_multi)

print("Train strata counts:\n", counts)


Train strata counts:
 en_1    5808
gr_0     523
en_0     224
gr_1     113
Name: count, dtype: int64


תא 4 — המודל: XLM-R + POS/Lemma embeddings + GRL adversarial

In [31]:
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModel

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def mean_pool(hidden, attn_mask):
    mask = attn_mask.unsqueeze(-1).type_as(hidden)
    summed = (hidden * mask).sum(dim=1)
    denom = mask.sum(dim=1).clamp(min=1e-6)
    return summed / denom

class GradReverse(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, alpha):
        ctx.alpha = alpha
        return x.view_as(x)
    @staticmethod
    def backward(ctx, grad_output):
        return -ctx.alpha * grad_output, None

def grl(x, alpha=1.0):
    return GradReverse.apply(x, alpha)

class XLMR_PosLemma_Adv(nn.Module):
    def __init__(self, model_name, pos_vocab_size, lem_vocab_size, pos_dim=64, lem_dim=64, dropout=0.1):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        h = self.encoder.config.hidden_size

        self.pos_emb = nn.Embedding(pos_vocab_size, pos_dim, padding_idx=0)
        self.lem_emb = nn.Embedding(lem_vocab_size, lem_dim, padding_idx=0)

        self.drop = nn.Dropout(dropout)

        fused_dim = h + pos_dim + lem_dim
        self.rewrite_head = nn.Linear(fused_dim, 2)
        self.lang_head    = nn.Linear(fused_dim, 2)

    def pool_embed(self, emb, mask):
        m = mask.unsqueeze(-1).type_as(emb)
        summed = (emb * m).sum(dim=1)
        denom = m.sum(dim=1).clamp(min=1e-6)
        return summed / denom

    def forward(self, tok, pos_ids, pos_mask, lem_ids, lem_mask, grl_alpha=1.0):
        out = self.encoder(**tok, return_dict=True)
        h_text = mean_pool(out.last_hidden_state, tok["attention_mask"])

        h_pos = self.pool_embed(self.pos_emb(pos_ids), pos_mask)
        h_lem = self.pool_embed(self.lem_emb(lem_ids), lem_mask)

        fused = torch.cat([h_text, h_pos, h_lem], dim=-1)
        fused = self.drop(fused)

        logits_y    = self.rewrite_head(fused)
        logits_lang = self.lang_head(grl(fused, grl_alpha))
        return logits_y, logits_lang

model = XLMR_PosLemma_Adv(MODEL_NAME, len(pos_vocab), len(lem_vocab)).to(DEVICE)
print("✅ Model ready on", DEVICE)


✅ Model ready on cuda


תא 5 — אימון + Eval (כמו אצלך) + שמירת best לפי VAL macro-F1

In [32]:
import os
import numpy as np
from sklearn.metrics import classification_report
from transformers import get_linear_schedule_with_warmup
from torch.optim import AdamW

LR = 2e-5
WEIGHT_DECAY = 0.01
EPOCHS = 3
LAMBDA_LANG = 0.3
GRL_ALPHA = 1.0

optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, int(0.05*total_steps), total_steps)
scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

@torch.no_grad()
def run_eval(loader, split="VAL"):
    model.eval()
    ys, ps, langs = [], [], []
    for tok, pos_ids, pos_mask, lem_ids, lem_mask, y, lang in loader:
        tok = {k: v.to(DEVICE) for k, v in tok.items()}
        pos_ids, pos_mask = pos_ids.to(DEVICE), pos_mask.to(DEVICE)
        lem_ids, lem_mask = lem_ids.to(DEVICE), lem_mask.to(DEVICE)
        y = y.to(DEVICE); lang = lang.to(DEVICE)

        logits_y, logits_lang = model(tok, pos_ids, pos_mask, lem_ids, lem_mask, grl_alpha=0.0)
        pred = torch.argmax(logits_y, dim=-1)

        ys.append(y.cpu().numpy())
        ps.append(pred.cpu().numpy())
        langs.append(lang.cpu().numpy())

    y_true = np.concatenate(ys)
    y_pred = np.concatenate(ps)
    lang_arr = np.concatenate(langs)

    print(f"\n=== {split} OVERALL ===")
    print(classification_report(y_true, y_pred, target_names=["Original","Rewrite"], zero_division=0))

    inv = {v:k for k,v in LANG2ID.items()}
    for lid in np.unique(lang_arr):
        m = (lang_arr == lid)
        name = inv.get(int(lid), str(lid)).upper()
        print(f"\n=== {split} | {name} (n={m.sum()}) ===")
        print(classification_report(y_true[m], y_pred[m], target_names=["Original","Rewrite"], zero_division=0))

    rep = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    return rep["macro avg"]["f1-score"]

best_f1 = -1.0
os.makedirs("outputs", exist_ok=True)
best_path = os.path.join("outputs", "xlmr_poslemma_adv_best.pt")

for ep in range(1, EPOCHS+1):
    model.train()
    total_loss = 0.0

    for tok, pos_ids, pos_mask, lem_ids, lem_mask, y, lang in train_loader:
        tok = {k: v.to(DEVICE) for k, v in tok.items()}
        pos_ids, pos_mask = pos_ids.to(DEVICE), pos_mask.to(DEVICE)
        lem_ids, lem_mask = lem_ids.to(DEVICE), lem_mask.to(DEVICE)
        y = y.to(DEVICE); lang = lang.to(DEVICE)

        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            logits_y, logits_lang = model(tok, pos_ids, pos_mask, lem_ids, lem_mask, grl_alpha=GRL_ALPHA)
            loss_y = F.cross_entropy(logits_y, y)
            loss_lang = F.cross_entropy(logits_lang, lang)
            loss = loss_y + LAMBDA_LANG * loss_lang

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        total_loss += float(loss.detach().cpu().item())

    print(f"\nEpoch {ep}/{EPOCHS} | train_loss={total_loss/max(1,len(train_loader)):.4f}")
    f1 = run_eval(val_loader, "VAL")
    print("VAL macro-F1:", round(f1, 4))

    if f1 > best_f1:
        best_f1 = f1
        torch.save(model.state_dict(), best_path)
        print("✅ Saved best model:", best_path)

# Final test
model.load_state_dict(torch.load(best_path, map_location=DEVICE))
_ = run_eval(test_loader, "TEST")


C:\Users\Asoulin_Sapir\AppData\Local\Temp\ipykernel_26304\686431644.py:16: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())
C:\Users\Asoulin_Sapir\AppData\Local\Temp\ipykernel_26304\686431644.py:68: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):



Epoch 1/3 | train_loss=2.3596

=== VAL OVERALL ===
              precision    recall  f1-score   support

    Original       0.95      0.77      0.85        93
     Rewrite       0.97      0.99      0.98       740

    accuracy                           0.97       833
   macro avg       0.96      0.88      0.92       833
weighted avg       0.97      0.97      0.97       833


=== VAL | EN (n=754) ===
              precision    recall  f1-score   support

    Original       0.85      0.79      0.81        28
     Rewrite       0.99      0.99      0.99       726

    accuracy                           0.99       754
   macro avg       0.92      0.89      0.90       754
weighted avg       0.99      0.99      0.99       754


=== VAL | GR (n=79) ===
              precision    recall  f1-score   support

    Original       1.00      0.77      0.87        65
     Rewrite       0.48      1.00      0.65        14

    accuracy                           0.81        79
   macro avg       0.74  

C:\Users\Asoulin_Sapir\AppData\Local\Temp\ipykernel_26304\686431644.py:68: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):



Epoch 2/3 | train_loss=0.4837

=== VAL OVERALL ===
              precision    recall  f1-score   support

    Original       0.95      0.92      0.93        93
     Rewrite       0.99      0.99      0.99       740

    accuracy                           0.99       833
   macro avg       0.97      0.96      0.96       833
weighted avg       0.99      0.99      0.99       833


=== VAL | EN (n=754) ===
              precision    recall  f1-score   support

    Original       0.85      0.79      0.81        28
     Rewrite       0.99      0.99      0.99       726

    accuracy                           0.99       754
   macro avg       0.92      0.89      0.90       754
weighted avg       0.99      0.99      0.99       754


=== VAL | GR (n=79) ===
              precision    recall  f1-score   support

    Original       0.98      0.98      0.98        65
     Rewrite       0.93      0.93      0.93        14

    accuracy                           0.97        79
   macro avg       0.96  

C:\Users\Asoulin_Sapir\AppData\Local\Temp\ipykernel_26304\686431644.py:68: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):



Epoch 3/3 | train_loss=0.1687

=== VAL OVERALL ===
              precision    recall  f1-score   support

    Original       0.96      0.88      0.92        93
     Rewrite       0.99      1.00      0.99       740

    accuracy                           0.98       833
   macro avg       0.98      0.94      0.96       833
weighted avg       0.98      0.98      0.98       833


=== VAL | EN (n=754) ===
              precision    recall  f1-score   support

    Original       0.88      0.75      0.81        28
     Rewrite       0.99      1.00      0.99       726

    accuracy                           0.99       754
   macro avg       0.93      0.87      0.90       754
weighted avg       0.99      0.99      0.99       754


=== VAL | GR (n=79) ===
              precision    recall  f1-score   support

    Original       1.00      0.94      0.97        65
     Rewrite       0.78      1.00      0.88        14

    accuracy                           0.95        79
   macro avg       0.89  

B) יישור POS ל-UPOS + נרמול Lemma + Hashing

In [33]:
import re, unicodedata, hashlib
import numpy as np
import pandas as pd

# ---------- 1) Greek diacritics stripper (מאוד עוזר ל-lemma ביוונית עתיקה) ----------
def strip_diacritics(s: str) -> str:
    s = unicodedata.normalize("NFD", s)
    s = "".join(ch for ch in s if unicodedata.category(ch) != "Mn")  # remove combining marks
    return unicodedata.normalize("NFC", s)

def normalize_lemma(lang: str, lemma_text: str) -> str:
    s = str(lemma_text).strip().lower()
    s = re.sub(r"\s+", " ", s)
    if lang.lower() == "gr":
        s = strip_diacritics(s)
    return s

# ---------- 2) POS -> UPOS mapping ----------
UPOS_SET = {
    "ADJ","ADP","ADV","AUX","CCONJ","DET","INTJ","NOUN",
    "NUM","PART","PRON","PROPN","PUNCT","SCONJ","SYM","VERB","X"
}

def greek_pos_to_upos(tag: str) -> str:
    """
    Heuristic mapping for Greek tagger outputs to UPOS.
    Works even אם התגיות הן קצרות/לא אחידות.
    """
    t = str(tag).strip()
    if not t:
        return "X"

    # אם כבר UPOS – נשאיר
    u = t.upper()
    if u in UPOS_SET:
        return u

    # כמה מקרים נפוצים (אפשר להרחיב אם תרצי)
    # ניחושים לפי אות ראשונה/תבנית:
    c = u[0]

    if c in {"N"}:  # Noun-like
        return "NOUN"
    if c in {"V"}:
        return "VERB"
    if c in {"A","J"}:
        return "ADJ"
    if c in {"D"}:
        return "ADV"
    if c in {"P"}:
        return "PRON"
    if c in {"R"}:
        return "ADP"
    if c in {"C"}:
        return "CCONJ"
    if c in {"S"}:
        return "SCONJ"
    if c in {"T"}:
        return "DET"
    if c in {"M"}:
        return "NUM"
    if c in {"I"}:
        return "INTJ"
    if c in {"U"}:
        return "PUNCT"

    # fallback
    return "X"

def normalize_pos(lang: str, pos_text: str) -> str:
    toks = str(pos_text).replace(",", " ").split()
    if lang.lower() == "en":
        # spaCy token.pos_ כבר בדרך כלל UPOS (NOUN/VERB/...)
        norm = []
        for t in toks:
            u = t.upper()
            norm.append(u if u in UPOS_SET else "X")
        return " ".join(norm)
    else:
        norm = [greek_pos_to_upos(t) for t in toks]
        return " ".join(norm)

# ---------- 3) Hashing trick: deterministic bucket id ----------
def hash_to_bucket(token: str, n_buckets: int) -> int:
    # md5 -> int -> mod; stable across runs/machines
    h = hashlib.md5(token.encode("utf-8")).hexdigest()
    return int(h, 16) % n_buckets

# Buckets (אפשר לשחק בזה)
POS_BUCKETS = 64       # UPOS קטן → לא צריך הרבה
LEM_BUCKETS = 8192     # lemma גדול → hashing בינוני טוב

def encode_hashed_tokens(text: str, n_buckets: int, max_len: int = 256):
    toks = str(text).split()
    toks = toks[:max_len] if toks else []
    if not toks:
        return [0]  # pad-like
    return [hash_to_bucket(t, n_buckets) + 1 for t in toks]  # +1 כדי לשמור 0 ל-PAD


In [34]:
def prepare_hashed_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["lang"] = df["lang"].astype(str).str.lower()

    # normalize POS/Lemma
    df["pos_u"] = [normalize_pos(l, p) for l, p in zip(df["lang"], df["pos"])]
    df["lem_n"] = [normalize_lemma(l, m) for l, m in zip(df["lang"], df["lemma"])]

    return df

train_all_h = prepare_hashed_features(train_all_pl)
val_all_h   = prepare_hashed_features(val_all_pl)
test_all_h  = prepare_hashed_features(test_all_pl)

print(train_all_h[["lang","y"]].value_counts())
print("Example POS_u:", train_all_h["pos_u"].iloc[0][:120])
print("Example lem_n:", train_all_h["lem_n"].iloc[0][:120])


lang  y
en    1    5808
gr    0     523
en    0     224
gr    1     113
Name: count, dtype: int64
Example POS_u: PRON ADJ ADJ ADJ ADJ NOUN ADJ X ADJ ADJ NOUN NOUN ADJ ADJ VERB NOUN NOUN ADJ ADJ ADJ NOUN ADJ NOUN ADJ ADJ ADJ ADJ ADJ V
Example lem_n: τουτων δε τουτον εχοντων τον τροπον αρετη μεν εστι νομοθετου τα βελτιστα συνιδειν και πεισαι τους χρησομενους περι των υ


C) Dataset/DataLoaders ל-multi-input החדש (hashed POS/Lemma)

In [35]:
import torch
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from transformers import AutoTokenizer

LANG2ID = {"en": 0, "gr": 1}
MODEL_NAME = "xlm-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

MAX_LEN = 256
BATCH = 8

class MultiInputHashedDS(Dataset):
    def __init__(self, df, max_pos_len=256, max_lem_len=256):
        self.df = df.reset_index(drop=True)
        self.max_pos_len = max_pos_len
        self.max_lem_len = max_lem_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        r = self.df.iloc[idx]
        lang = str(r["lang"]).lower()
        return {
            "text": str(r["text"]),
            "pos_ids": encode_hashed_tokens(r["pos_u"], POS_BUCKETS, self.max_pos_len),
            "lem_ids": encode_hashed_tokens(r["lem_n"], LEM_BUCKETS, self.max_lem_len),
            "y": int(r["y"]),
            "lang_id": LANG2ID[lang],
        }

def pad_2d(list_ids, pad_id=0):
    maxl = max(len(x) for x in list_ids)
    arr = torch.full((len(list_ids), maxl), pad_id, dtype=torch.long)
    mask = torch.zeros((len(list_ids), maxl), dtype=torch.long)
    for i, x in enumerate(list_ids):
        arr[i, :len(x)] = torch.tensor(x, dtype=torch.long)
        mask[i, :len(x)] = 1
    return arr, mask

def collate_hashed(batch):
    texts = [b["text"] for b in batch]
    y = torch.tensor([b["y"] for b in batch], dtype=torch.long)
    lang_id = torch.tensor([b["lang_id"] for b in batch], dtype=torch.long)

    tok = tokenizer(texts, padding=True, truncation=True, max_length=MAX_LEN, return_tensors="pt")

    pos_ids, pos_mask = pad_2d([b["pos_ids"] for b in batch], pad_id=0)
    lem_ids, lem_mask = pad_2d([b["lem_ids"] for b in batch], pad_id=0)

    return tok, pos_ids, pos_mask, lem_ids, lem_mask, y, lang_id

# balanced sampler by (lang_y) כמו שעשית, רק על train_all_h
strata = train_all_h["lang"].astype(str) + "_" + train_all_h["y"].astype(str)
counts = strata.value_counts()
weights = strata.map(lambda s: 1.0 / counts[s]).values
sampler = WeightedRandomSampler(torch.tensor(weights, dtype=torch.double),
                                num_samples=len(weights), replacement=True)

train_loader = DataLoader(MultiInputHashedDS(train_all_h), batch_size=BATCH, sampler=sampler, collate_fn=collate_hashed)
val_loader   = DataLoader(MultiInputHashedDS(val_all_h),   batch_size=BATCH, shuffle=False, collate_fn=collate_hashed)
test_loader  = DataLoader(MultiInputHashedDS(test_all_h),  batch_size=BATCH, shuffle=False, collate_fn=collate_hashed)

print("Train strata:\n", counts)


Train strata:
 en_1    5808
gr_0     523
en_0     224
gr_1     113
Name: count, dtype: int64


D) מודל multi-input משופר: gating + LayerNorm + adversarial (GRL)

In [36]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModel
from sklearn.metrics import classification_report
import numpy as np

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def mean_pool(hidden, attn_mask):
    mask = attn_mask.unsqueeze(-1).type_as(hidden)
    summed = (hidden * mask).sum(dim=1)
    denom = mask.sum(dim=1).clamp(min=1e-6)
    return summed / denom

class GradReverse(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, alpha):
        ctx.alpha = alpha
        return x.view_as(x)
    @staticmethod
    def backward(ctx, grad_output):
        return -ctx.alpha * grad_output, None

def grl(x, alpha=1.0):
    return GradReverse.apply(x, alpha)

class XLMR_HashedPosLemma_Adv(nn.Module):
    def __init__(self, model_name, pos_buckets, lem_buckets, pos_dim=32, lem_dim=64, dropout=0.2):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        h = self.encoder.config.hidden_size

        # +1 כבר מגולם בקידוד (0=PAD, 1..n = buckets)
        self.pos_emb = nn.Embedding(pos_buckets + 1, pos_dim, padding_idx=0)
        self.lem_emb = nn.Embedding(lem_buckets + 1, lem_dim, padding_idx=0)

        self.drop = nn.Dropout(dropout)

        # gates: מתחילים "סגורים" יחסית כדי לא להזיק בתחילת אימון
        self.g_pos = nn.Parameter(torch.tensor(-1.0))  # sigmoid ~ 0.27
        self.g_lem = nn.Parameter(torch.tensor(-1.0))

        fused_dim = h + pos_dim + lem_dim
        self.ln = nn.LayerNorm(fused_dim)

        self.rewrite_head = nn.Linear(fused_dim, 2)
        self.lang_head    = nn.Linear(fused_dim, 2)

    def pool_embed(self, emb, mask):
        m = mask.unsqueeze(-1).type_as(emb)
        summed = (emb * m).sum(dim=1)
        denom = m.sum(dim=1).clamp(min=1e-6)
        return summed / denom

    def forward(self, tok, pos_ids, pos_mask, lem_ids, lem_mask, grl_alpha=1.0):
        out = self.encoder(**tok, return_dict=True)
        h_text = mean_pool(out.last_hidden_state, tok["attention_mask"])

        h_pos = self.pool_embed(self.pos_emb(pos_ids), pos_mask)
        h_lem = self.pool_embed(self.lem_emb(lem_ids), lem_mask)

        # gated
        gp = torch.sigmoid(self.g_pos)
        gl = torch.sigmoid(self.g_lem)

        fused = torch.cat([h_text, gp * h_pos, gl * h_lem], dim=-1)
        fused = self.drop(self.ln(fused))

        logits_y = self.rewrite_head(fused)
        logits_lang = self.lang_head(grl(fused, grl_alpha))
        return logits_y, logits_lang

model = XLMR_HashedPosLemma_Adv(MODEL_NAME, POS_BUCKETS, LEM_BUCKETS).to(DEVICE)
print("✅ model ready:", type(model).__name__)


✅ model ready: XLMR_HashedPosLemma_Adv


E) אימון + Eval + שמירת best לפי VAL macro-F1

In [37]:
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
import os

LR = 2e-5
WEIGHT_DECAY = 0.01
EPOCHS = 3
LAMBDA_LANG = 0.3
GRL_ALPHA = 1.0

optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, int(0.05 * total_steps), total_steps)
scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

@torch.no_grad()
def run_eval(loader, split="VAL"):
    model.eval()
    ys, ps, langs = [], [], []
    for tok, pos_ids, pos_mask, lem_ids, lem_mask, y, lang_id in loader:
        tok = {k: v.to(DEVICE) for k, v in tok.items()}
        pos_ids, pos_mask = pos_ids.to(DEVICE), pos_mask.to(DEVICE)
        lem_ids, lem_mask = lem_ids.to(DEVICE), lem_mask.to(DEVICE)
        y = y.to(DEVICE)
        lang_id = lang_id.to(DEVICE)

        logits_y, _ = model(tok, pos_ids, pos_mask, lem_ids, lem_mask, grl_alpha=0.0)
        pred = torch.argmax(logits_y, dim=-1)

        ys.append(y.cpu().numpy())
        ps.append(pred.cpu().numpy())
        langs.append(lang_id.cpu().numpy())

    y_true = np.concatenate(ys)
    y_pred = np.concatenate(ps)
    lang_arr = np.concatenate(langs)

    print(f"\n=== {split} OVERALL ===")
    print(classification_report(y_true, y_pred, target_names=["Original", "Rewrite"], zero_division=0))

    inv = {v: k for k, v in LANG2ID.items()}
    for lid in np.unique(lang_arr):
        m = (lang_arr == lid)
        name = inv.get(int(lid), str(lid)).upper()
        print(f"\n=== {split} | {name} (n={m.sum()}) ===")
        print(classification_report(y_true[m], y_pred[m], target_names=["Original", "Rewrite"], zero_division=0))

    rep = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    return rep["macro avg"]["f1-score"]

os.makedirs("outputs", exist_ok=True)
best_path = os.path.join("outputs", "xlmr_hashed_poslemma_adv_best.pt")
best_f1 = -1.0

for ep in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0

    for tok, pos_ids, pos_mask, lem_ids, lem_mask, y, lang_id in train_loader:
        tok = {k: v.to(DEVICE) for k, v in tok.items()}
        pos_ids, pos_mask = pos_ids.to(DEVICE), pos_mask.to(DEVICE)
        lem_ids, lem_mask = lem_ids.to(DEVICE), lem_mask.to(DEVICE)
        y = y.to(DEVICE)
        lang_id = lang_id.to(DEVICE)

        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            logits_y, logits_lang = model(tok, pos_ids, pos_mask, lem_ids, lem_mask, grl_alpha=GRL_ALPHA)
            loss_y = F.cross_entropy(logits_y, y)
            loss_lang = F.cross_entropy(logits_lang, lang_id)
            loss = loss_y + LAMBDA_LANG * loss_lang

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        total_loss += float(loss.detach().cpu().item())

    print(f"\nEpoch {ep}/{EPOCHS} | train_loss={total_loss/max(1,len(train_loader)):.4f}")
    print("Gates: g_pos=", float(torch.sigmoid(model.g_pos).detach().cpu()),
          " g_lem=", float(torch.sigmoid(model.g_lem).detach().cpu()))

    f1 = run_eval(val_loader, "VAL")
    print("VAL macro-F1:", round(f1, 4))

    if f1 > best_f1:
        best_f1 = f1
        torch.save(model.state_dict(), best_path)
        print("✅ Saved best:", best_path)

model.load_state_dict(torch.load(best_path, map_location=DEVICE))
_ = run_eval(test_loader, "TEST")


C:\Users\Asoulin_Sapir\AppData\Local\Temp\ipykernel_26304\4273002705.py:14: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())
C:\Users\Asoulin_Sapir\AppData\Local\Temp\ipykernel_26304\4273002705.py:68: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):



Epoch 1/3 | train_loss=2.7766
Gates: g_pos= 0.265970915555954  g_lem= 0.26696518063545227

=== VAL OVERALL ===
              precision    recall  f1-score   support

    Original       0.87      0.95      0.91        93
     Rewrite       0.99      0.98      0.99       740

    accuracy                           0.98       833
   macro avg       0.93      0.96      0.95       833
weighted avg       0.98      0.98      0.98       833


=== VAL | EN (n=754) ===
              precision    recall  f1-score   support

    Original       0.70      1.00      0.82        28
     Rewrite       1.00      0.98      0.99       726

    accuracy                           0.98       754
   macro avg       0.85      0.99      0.91       754
weighted avg       0.99      0.98      0.99       754


=== VAL | GR (n=79) ===
              precision    recall  f1-score   support

    Original       0.98      0.92      0.95        65
     Rewrite       0.72      0.93      0.81        14

    accuracy       

C:\Users\Asoulin_Sapir\AppData\Local\Temp\ipykernel_26304\4273002705.py:68: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):



Epoch 2/3 | train_loss=0.7758
Gates: g_pos= 0.2640984356403351  g_lem= 0.2653563916683197

=== VAL OVERALL ===
              precision    recall  f1-score   support

    Original       0.96      0.91      0.93        93
     Rewrite       0.99      0.99      0.99       740

    accuracy                           0.99       833
   macro avg       0.97      0.95      0.96       833
weighted avg       0.99      0.99      0.99       833


=== VAL | EN (n=754) ===
              precision    recall  f1-score   support

    Original       0.89      0.86      0.87        28
     Rewrite       0.99      1.00      1.00       726

    accuracy                           0.99       754
   macro avg       0.94      0.93      0.93       754
weighted avg       0.99      0.99      0.99       754


=== VAL | GR (n=79) ===
              precision    recall  f1-score   support

    Original       0.98      0.94      0.96        65
     Rewrite       0.76      0.93      0.84        14

    accuracy       

C:\Users\Asoulin_Sapir\AppData\Local\Temp\ipykernel_26304\4273002705.py:68: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):



Epoch 3/3 | train_loss=0.2110
Gates: g_pos= 0.26368871331214905  g_lem= 0.2649734318256378

=== VAL OVERALL ===
              precision    recall  f1-score   support

    Original       0.97      0.92      0.95        93
     Rewrite       0.99      1.00      0.99       740

    accuracy                           0.99       833
   macro avg       0.98      0.96      0.97       833
weighted avg       0.99      0.99      0.99       833


=== VAL | EN (n=754) ===
              precision    recall  f1-score   support

    Original       0.92      0.86      0.89        28
     Rewrite       0.99      1.00      1.00       726

    accuracy                           0.99       754
   macro avg       0.96      0.93      0.94       754
weighted avg       0.99      0.99      0.99       754


=== VAL | GR (n=79) ===
              precision    recall  f1-score   support

    Original       0.98      0.95      0.97        65
     Rewrite       0.81      0.93      0.87        14

    accuracy      

In [38]:
import torch
import numpy as np
from sklearn.metrics import classification_report

@torch.no_grad()
def eval_with_forced_gates(loader, gpos=None, glem=None, split="TEST"):
    model.eval()
    # save old
    old_pos = model.g_pos.detach().clone()
    old_lem = model.g_lem.detach().clone()

    # force if requested: set logit so sigmoid gives the gate value
    def inv_sigmoid(p):
        p = float(np.clip(p, 1e-6, 1-1e-6))
        return np.log(p/(1-p))

    if gpos is not None:
        model.g_pos.data.fill_(inv_sigmoid(gpos))
    if glem is not None:
        model.g_lem.data.fill_(inv_sigmoid(glem))

    ys, ps, langs = [], [], []
    for tok, pos_ids, pos_mask, lem_ids, lem_mask, y, lang_id in loader:
        tok = {k: v.to(DEVICE) for k, v in tok.items()}
        pos_ids, pos_mask = pos_ids.to(DEVICE), pos_mask.to(DEVICE)
        lem_ids, lem_mask = lem_ids.to(DEVICE), lem_mask.to(DEVICE)
        y = y.to(DEVICE); lang_id = lang_id.to(DEVICE)

        logits_y, _ = model(tok, pos_ids, pos_mask, lem_ids, lem_mask, grl_alpha=0.0)
        pred = torch.argmax(logits_y, dim=-1)

        ys.append(y.cpu().numpy())
        ps.append(pred.cpu().numpy())
        langs.append(lang_id.cpu().numpy())

    y_true = np.concatenate(ys); y_pred = np.concatenate(ps); lang_arr = np.concatenate(langs)
    print(f"\n=== {split} forced gates: gpos={gpos}, glem={glem} ===")
    print(classification_report(y_true, y_pred, target_names=["Original","Rewrite"], zero_division=0))

    # restore
    model.g_pos.data.copy_(old_pos)
    model.g_lem.data.copy_(old_lem)

# אחרי שסיימת אימון וטענת best:
eval_with_forced_gates(test_loader, gpos=None, glem=None, split="TEST (normal)")
eval_with_forced_gates(test_loader, gpos=0.0, glem=None, split="TEST (POS off)")
eval_with_forced_gates(test_loader, gpos=None, glem=0.0, split="TEST (Lemma off)")
eval_with_forced_gates(test_loader, gpos=0.0, glem=0.0, split="TEST (POS+Lemma off)")



=== TEST (normal) forced gates: gpos=None, glem=None ===
              precision    recall  f1-score   support

    Original       0.90      0.93      0.91        94
     Rewrite       0.99      0.99      0.99       740

    accuracy                           0.98       834
   macro avg       0.94      0.96      0.95       834
weighted avg       0.98      0.98      0.98       834


=== TEST (POS off) forced gates: gpos=0.0, glem=None ===
              precision    recall  f1-score   support

    Original       0.90      0.93      0.91        94
     Rewrite       0.99      0.99      0.99       740

    accuracy                           0.98       834
   macro avg       0.94      0.96      0.95       834
weighted avg       0.98      0.98      0.98       834


=== TEST (Lemma off) forced gates: gpos=None, glem=0.0 ===
              precision    recall  f1-score   support

    Original       0.90      0.93      0.91        94
     Rewrite       0.99      0.99      0.99       740

    ac

תא 0 — Imports + קבועים

In [39]:
import os, re, math, json, unicodedata
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from torch.optim import AdamW
from sklearn.metrics import classification_report
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import HashingVectorizer


תא 1 — בדיקות מינימום (הנחות על הדאטה)

In [40]:
for name, df in [("train_all", train_all), ("val_all", val_all), ("test_all", test_all)]:
    missing = [c for c in ["text","y","lang"] if c not in df.columns]
    if missing:
        raise ValueError(f"{name} missing columns: {missing}")
print("✅ DataFrames OK:", train_all.shape, val_all.shape, test_all.shape)

# Normalize lang just in case
for df in (train_all, val_all, test_all):
    df["lang"] = df["lang"].astype(str).str.lower()


✅ DataFrames OK: (6668, 12) (833, 12) (834, 12)


## H) Experiment 2 — Multi-input: XLM-R + Style numeric + Char-hash ngrams + Gating + GRL
---


תא 2 — Style numeric features (אגנוסטי + diacritics density)

In [17]:
WS_RE = re.compile(r"\s+")
GREEK_RANGE_RE = re.compile(r"[\u0370-\u03FF\u1F00-\u1FFF]")  # Greek + polytonic
ALNUM_RE = re.compile(r"[A-Za-z0-9\u0370-\u03FF\u1F00-\u1FFF]", re.UNICODE)

def count_diacritics(s: str) -> int:
    # combining marks = Mn
    s = unicodedata.normalize("NFD", s)
    return sum(1 for ch in s if unicodedata.category(ch) == "Mn")

def style_features(lang: str, text: str) -> np.ndarray:
    s = str(text)
    s = WS_RE.sub(" ", s).strip()

    n_chars = len(s)
    toks = s.split() if s else []
    n_tok = len(toks)

    tok_lens = [len(t) for t in toks] if toks else [0]
    avg_tok_len = float(np.mean(tok_lens))
    std_tok_len = float(np.std(tok_lens))
    long_tok_rate = float(np.mean([l >= 8 for l in tok_lens])) if toks else 0.0

    # lexical diversity
    uniq = len(set(toks)) if toks else 0
    ttr = (uniq / n_tok) if n_tok else 0.0

    # digit ratio (works cross-lingual)
    n_digits = sum(ch.isdigit() for ch in s)
    digit_ratio = n_digits / max(1, n_chars)

    # character diversity (distinct chars / length)
    distinct_chars = len(set(s)) if s else 0
    char_div = distinct_chars / max(1, n_chars)

    # greek signal features (still language-agnostic-ish)
    # diacritics density helps for Ancient Greek normalization differences and style
    diac = count_diacritics(s) if lang == "gr" else 0
    diac_ratio = diac / max(1, n_chars)

    # fraction of greek letters among alnum chars (sanity; for EN will be ~0)
    alnum = ALNUM_RE.findall(s)
    n_alnum = len(alnum)
    n_gr = len(GREEK_RANGE_RE.findall(s)) if s else 0
    greek_ratio = (n_gr / max(1, n_alnum)) if n_alnum else 0.0

    # NOTE: punctuation not used as core feature (you removed punctuation in GR)
    return np.array([
        n_chars,
        n_tok,
        avg_tok_len,
        std_tok_len,
        long_tok_rate,
        ttr,
        digit_ratio,
        char_div,
        diac_ratio,
        greek_ratio
    ], dtype=np.float32)

def build_style_matrix(df: pd.DataFrame) -> np.ndarray:
    feats = np.vstack([style_features(l, t) for l, t in zip(df["lang"].tolist(), df["text"].tolist())])
    return feats

X_style_train = build_style_matrix(train_all)
X_style_val   = build_style_matrix(val_all)
X_style_test  = build_style_matrix(test_all)

# scale (fit on train only)
scaler = StandardScaler()
X_style_train = scaler.fit_transform(X_style_train).astype(np.float32)
X_style_val   = scaler.transform(X_style_val).astype(np.float32)
X_style_test  = scaler.transform(X_style_test).astype(np.float32)

import joblib, os
os.makedirs("outputs", exist_ok=True)
joblib.dump(scaler, "outputs/style_scaler.pkl")
print("✅ Saved style scaler: outputs/style_scaler.pkl")
print("STYLE_DIM =", X_style_train.shape[1])
print("✅ Style matrices:", X_style_train.shape, X_style_val.shape, X_style_test.shape)


NameError: name 'unicodedata' is not defined

In [43]:
import json, os
meta = {
    "STYLE_DIM": int(X_style_train.shape[1]),
    "style_features_name": "H_style_features_v1",
    "scaler_path": "outputs/style_scaler.pkl",
    "note": "fit on train_all only; transform val_all/test_all"
}
with open("outputs/style_meta.json", "w", encoding="utf-8") as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)
print("✅ Saved style meta: outputs/style_meta.json")


✅ Saved style meta: outputs/style_meta.json


In [44]:
np.save("outputs/style_train.npy", X_style_train)
np.save("outputs/style_val.npy", X_style_val)
np.save("outputs/style_test.npy", X_style_test)

In [45]:
import os
print("CWD:", os.getcwd())
print("Saved style scaler at:", os.path.abspath("outputs/style_scaler.pkl"))

CWD: C:\Users\Asoulin_Sapir\Desktop\עבודה עדכנית
Saved style scaler at: C:\Users\Asoulin_Sapir\Desktop\עבודה עדכנית\outputs\style_scaler.pkl


תא 3 — Char-hash ngrams (HashingVectorizer)

In [46]:
CHAR_DIM = 4096  # אפשר 8192 אם יש לך זיכרון
char_vec = HashingVectorizer(
    analyzer="char",
    ngram_range=(3,5),
    n_features=CHAR_DIM,
    alternate_sign=False,   # חשוב ליציבות
    norm="l2",
    lowercase=False
)

def build_char_matrix(df: pd.DataFrame) -> np.ndarray:
    # HashingVectorizer מחזיר sparse; נהפוך ל-dense float16 כדי לחסוך RAM
    X = char_vec.transform(df["text"].astype(str).tolist())
    X = X.astype(np.float32).toarray().astype(np.float16)
    return X

X_char_train = build_char_matrix(train_all)
X_char_val   = build_char_matrix(val_all)
X_char_test  = build_char_matrix(test_all)

print("✅ Char matrices:", X_char_train.shape, X_char_val.shape, X_char_test.shape, "dtype:", X_char_train.dtype)


✅ Char matrices: (6668, 4096) (833, 4096) (834, 4096) dtype: float16


In [47]:
import json
char_meta = {
    "CHAR_DIM": int(CHAR_DIM),
    "analyzer": "char",
    "ngram_range": [3,5],
    "alternate_sign": False,
    "norm": "l2",
    "lowercase": False
}
with open("outputs/char_meta.json", "w", encoding="utf-8") as f:
    json.dump(char_meta, f, ensure_ascii=False, indent=2)
print("✅ Saved char meta: outputs/char_meta.json")


✅ Saved char meta: outputs/char_meta.json


In [48]:
np.save("outputs/char_train.npy", X_char_train)
np.save("outputs/char_val.npy", X_char_val)
np.save("outputs/char_test.npy", X_char_test)


תא 4 — Dataset + DataLoaders (כולל balanced sampler לפי lang_y)

In [49]:
LANG2ID = {"en": 0, "gr": 1}

class MultiBranchDS(Dataset):
    def __init__(self, df, X_style, X_char):
        self.df = df.reset_index(drop=True)
        self.X_style = X_style
        self.X_char = X_char

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        r = self.df.iloc[idx]
        lang = str(r["lang"]).lower()
        return {
            "text": str(r["text"]),
            "y": int(r["y"]),
            "lang_id": LANG2ID[lang],
            "style": self.X_style[idx],      # (style_dim,)
            "char": self.X_char[idx],        # (char_dim,) float16
        }

MODEL_NAME = "xlm-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

MAX_LEN = 256
BATCH = 8

def collate_multibranch(batch):
    texts = [b["text"] for b in batch]
    y = torch.tensor([b["y"] for b in batch], dtype=torch.long)
    lang_id = torch.tensor([b["lang_id"] for b in batch], dtype=torch.long)

    tok = tokenizer(texts, padding=True, truncation=True, max_length=MAX_LEN, return_tensors="pt")

    style = torch.tensor(np.stack([b["style"] for b in batch]), dtype=torch.float32)  # already scaled
    char  = torch.tensor(np.stack([b["char"]  for b in batch]), dtype=torch.float32)  # cast from float16

    return tok, style, char, y, lang_id


# Balanced sampler by (lang_y)
strata = train_all["lang"].astype(str).str.lower() + "_" + train_all["y"].astype(str)
counts = strata.value_counts()
weights = strata.map(lambda s: 1.0 / counts[s]).values
sampler = WeightedRandomSampler(torch.tensor(weights, dtype=torch.double),
                                num_samples=len(weights), replacement=True)

train_loader = DataLoader(MultiBranchDS(train_all, X_style_train, X_char_train),
                          batch_size=BATCH, sampler=sampler, collate_fn=collate_multibranch)
val_loader   = DataLoader(MultiBranchDS(val_all, X_style_val, X_char_val),
                          batch_size=BATCH, shuffle=False, collate_fn=collate_multibranch)
test_loader  = DataLoader(MultiBranchDS(test_all, X_style_test, X_char_test),
                          batch_size=BATCH, shuffle=False, collate_fn=collate_multibranch)

print("✅ loaders ready | train strata:\n", counts)


✅ loaders ready | train strata:
 en_1    5808
gr_0     523
en_0     224
gr_1     113
Name: count, dtype: int64


תא 5 — מודל: XLM-R + Style branch + Char branch + Gating + GRL adversarial

In [50]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def mean_pool(hidden, attn_mask):
    mask = attn_mask.unsqueeze(-1).type_as(hidden)
    summed = (hidden * mask).sum(dim=1)
    denom = mask.sum(dim=1).clamp(min=1e-6)
    return summed / denom

class GradReverse(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, alpha):
        ctx.alpha = alpha
        return x.view_as(x)
    @staticmethod
    def backward(ctx, grad_output):
        return -ctx.alpha * grad_output, None

def grl(x, alpha=1.0):
    return GradReverse.apply(x, alpha)

class XLMR_StyleChar_Adv(nn.Module):
    def __init__(self, model_name, style_dim, char_dim, style_h=32, char_h=128, dropout=0.2):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        h = self.encoder.config.hidden_size

        # Style branch (small MLP)
        self.style_mlp = nn.Sequential(
            nn.Linear(style_dim, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, style_h),
        )

        # Char branch (projection + nonlinearity)
        self.char_mlp = nn.Sequential(
            nn.Linear(char_dim, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, char_h),
        )

        # Gates start low to avoid hurting early
        self.g_style = nn.Parameter(torch.tensor(-1.0))  # sigmoid ~ 0.27
        self.g_char  = nn.Parameter(torch.tensor(-1.0))

        fused_dim = h + style_h + char_h
        self.ln = nn.LayerNorm(fused_dim)
        self.drop = nn.Dropout(dropout)

        self.rewrite_head = nn.Linear(fused_dim, 2)
        self.lang_head    = nn.Linear(fused_dim, 2)

    def forward(self, tok, style_vec, char_vec, grl_alpha=1.0):
        out = self.encoder(**tok, return_dict=True)
        h_text = mean_pool(out.last_hidden_state, tok["attention_mask"])

        h_style = self.style_mlp(style_vec)
        h_char  = self.char_mlp(char_vec)

        gs = torch.sigmoid(self.g_style)
        gc = torch.sigmoid(self.g_char)

        fused = torch.cat([h_text, gs * h_style, gc * h_char], dim=-1)
        fused = self.drop(self.ln(fused))

        logits_y = self.rewrite_head(fused)
        logits_lang = self.lang_head(grl(fused, grl_alpha))
        return logits_y, logits_lang


STYLE_DIM = X_style_train.shape[1]
model = XLMR_StyleChar_Adv(MODEL_NAME, STYLE_DIM, CHAR_DIM).to(DEVICE)
print("✅ model ready:", type(model).__name__, "on", DEVICE)


✅ model ready: XLMR_StyleChar_Adv on cuda


תא 6 — Train + Eval + Save best (VAL macro-F1)

In [51]:
LR = 2e-5
WEIGHT_DECAY = 0.01
# [EXP-A] EPOCHS increased from 3 to 5 (Section H StyleChar main training)
EPOCHS = 5

LAMBDA_LANG = 0.3
GRL_ALPHA = 1.0

optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, int(0.05 * total_steps), total_steps)
scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

@torch.no_grad()
def run_eval(loader, split="VAL"):
    model.eval()
    ys, ps, langs = [], [], []
    for tok, style_vec, char_vec, y, lang_id in loader:
        tok = {k: v.to(DEVICE) for k, v in tok.items()}
        style_vec = style_vec.to(DEVICE)
        char_vec  = char_vec.to(DEVICE)
        y = y.to(DEVICE)
        lang_id = lang_id.to(DEVICE)

        logits_y, _ = model(tok, style_vec, char_vec, grl_alpha=0.0)
        pred = torch.argmax(logits_y, dim=-1)

        ys.append(y.cpu().numpy())
        ps.append(pred.cpu().numpy())
        langs.append(lang_id.cpu().numpy())

    y_true = np.concatenate(ys)
    y_pred = np.concatenate(ps)
    lang_arr = np.concatenate(langs)

    print(f"\n=== {split} OVERALL ===")
    print(classification_report(y_true, y_pred, target_names=["Original","Rewrite"], zero_division=0))

    inv = {v:k for k,v in LANG2ID.items()}
    for lid in np.unique(lang_arr):
        m = (lang_arr == lid)
        name = inv.get(int(lid), str(lid)).upper()
        print(f"\n=== {split} | {name} (n={m.sum()}) ===")
        print(classification_report(y_true[m], y_pred[m], target_names=["Original","Rewrite"], zero_division=0))

    rep = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    return rep["macro avg"]["f1-score"]

os.makedirs("outputs", exist_ok=True)
best_path = os.path.join("outputs", "xlmr_style_char_adv_best.pt")
best_f1 = -1.0

for ep in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0

    for tok, style_vec, char_vec, y, lang_id in train_loader:
        tok = {k: v.to(DEVICE) for k, v in tok.items()}
        style_vec = style_vec.to(DEVICE)
        char_vec  = char_vec.to(DEVICE)
        y = y.to(DEVICE)
        lang_id = lang_id.to(DEVICE)

        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            logits_y, logits_lang = model(tok, style_vec, char_vec, grl_alpha=GRL_ALPHA)

            loss_y = F.cross_entropy(logits_y, y)
            loss_lang = F.cross_entropy(logits_lang, lang_id)

            loss = loss_y + LAMBDA_LANG * loss_lang

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        total_loss += float(loss.detach().cpu().item())

    gs = float(torch.sigmoid(model.g_style).detach().cpu())
    gc = float(torch.sigmoid(model.g_char).detach().cpu())
    print(f"\nEpoch {ep}/{EPOCHS} | train_loss={total_loss/max(1,len(train_loader)):.4f}")
    print(f"Gates: g_style={gs:.3f}  g_char={gc:.3f}")

    f1 = run_eval(val_loader, "VAL")
    print("VAL macro-F1:", round(f1, 4))

    if f1 > best_f1:
        best_f1 = f1
        torch.save(model.state_dict(), best_path)
        print("✅ Saved best model:", best_path)

# Final test
model.load_state_dict(torch.load(best_path, map_location=DEVICE))
_ = run_eval(test_loader, "TEST")
print("✅ Best saved at:", best_path)


C:\Users\Asoulin_Sapir\AppData\Local\Temp\ipykernel_26304\489188062.py:11: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())
C:\Users\Asoulin_Sapir\AppData\Local\Temp\ipykernel_26304\489188062.py:65: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):



Epoch 1/3 | train_loss=1.9417
Gates: g_style=0.268  g_char=0.273

=== VAL OVERALL ===
              precision    recall  f1-score   support

    Original       0.82      0.96      0.89        93
     Rewrite       0.99      0.97      0.98       740

    accuracy                           0.97       833
   macro avg       0.91      0.97      0.93       833
weighted avg       0.98      0.97      0.97       833


=== VAL | EN (n=754) ===
              precision    recall  f1-score   support

    Original       0.61      0.96      0.75        28
     Rewrite       1.00      0.98      0.99       726

    accuracy                           0.98       754
   macro avg       0.81      0.97      0.87       754
weighted avg       0.98      0.98      0.98       754


=== VAL | GR (n=79) ===
              precision    recall  f1-score   support

    Original       0.97      0.95      0.96        65
     Rewrite       0.80      0.86      0.83        14

    accuracy                           0.94 

C:\Users\Asoulin_Sapir\AppData\Local\Temp\ipykernel_26304\489188062.py:65: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):



Epoch 2/3 | train_loss=0.7099
Gates: g_style=0.269  g_char=0.274

=== VAL OVERALL ===
              precision    recall  f1-score   support

    Original       0.93      0.91      0.92        93
     Rewrite       0.99      0.99      0.99       740

    accuracy                           0.98       833
   macro avg       0.96      0.95      0.96       833
weighted avg       0.98      0.98      0.98       833


=== VAL | EN (n=754) ===
              precision    recall  f1-score   support

    Original       0.85      0.82      0.84        28
     Rewrite       0.99      0.99      0.99       726

    accuracy                           0.99       754
   macro avg       0.92      0.91      0.92       754
weighted avg       0.99      0.99      0.99       754


=== VAL | GR (n=79) ===
              precision    recall  f1-score   support

    Original       0.97      0.95      0.96        65
     Rewrite       0.80      0.86      0.83        14

    accuracy                           0.94 

C:\Users\Asoulin_Sapir\AppData\Local\Temp\ipykernel_26304\489188062.py:65: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):



Epoch 3/3 | train_loss=0.2492
Gates: g_style=0.269  g_char=0.274

=== VAL OVERALL ===
              precision    recall  f1-score   support

    Original       0.93      0.94      0.93        93
     Rewrite       0.99      0.99      0.99       740

    accuracy                           0.98       833
   macro avg       0.96      0.96      0.96       833
weighted avg       0.98      0.98      0.98       833


=== VAL | EN (n=754) ===
              precision    recall  f1-score   support

    Original       0.83      0.89      0.86        28
     Rewrite       1.00      0.99      0.99       726

    accuracy                           0.99       754
   macro avg       0.91      0.94      0.93       754
weighted avg       0.99      0.99      0.99       754


=== VAL | GR (n=79) ===
              precision    recall  f1-score   support

    Original       0.97      0.95      0.96        65
     Rewrite       0.80      0.86      0.83        14

    accuracy                           0.94 

In [52]:
@torch.no_grad()
def eval_forced_gates(loader, g_style=None, g_char=None, split="TEST"):
    model.eval()
    old_s = model.g_style.detach().clone()
    old_c = model.g_char.detach().clone()

    def inv_sigmoid(p):
        p = float(np.clip(p, 1e-6, 1-1e-6))
        return np.log(p/(1-p))

    if g_style is not None:
        model.g_style.data.fill_(inv_sigmoid(g_style))
    if g_char is not None:
        model.g_char.data.fill_(inv_sigmoid(g_char))

    _ = run_eval(loader, split=split)

    model.g_style.data.copy_(old_s)
    model.g_char.data.copy_(old_c)

eval_forced_gates(test_loader, g_style=None, g_char=None, split="TEST normal")
eval_forced_gates(test_loader, g_style=0.0,  g_char=None, split="TEST style off")
eval_forced_gates(test_loader, g_style=None, g_char=0.0,  split="TEST char off")
eval_forced_gates(test_loader, g_style=0.0,  g_char=0.0,  split="TEST style+char off")



=== TEST normal OVERALL ===
              precision    recall  f1-score   support

    Original       0.90      0.99      0.94        94
     Rewrite       1.00      0.99      0.99       740

    accuracy                           0.99       834
   macro avg       0.95      0.99      0.97       834
weighted avg       0.99      0.99      0.99       834


=== TEST normal | EN (n=754) ===
              precision    recall  f1-score   support

    Original       0.85      1.00      0.92        28
     Rewrite       1.00      0.99      1.00       726

    accuracy                           0.99       754
   macro avg       0.92      1.00      0.96       754
weighted avg       0.99      0.99      0.99       754


=== TEST normal | GR (n=80) ===
              precision    recall  f1-score   support

    Original       0.93      0.98      0.96        66
     Rewrite       0.90      0.64      0.75        14

    accuracy                           0.93        80
   macro avg       0.91      0.8

## I) Extra Experiments — Calibration / BoostGR / Experiment 3
---


# 🚨 ניסוי נוסף 1 — Calibration של סף החלטה ליוונית על VAL (ללא אימון מחדש)

**למה זה חשוב?**  
המודל הנוכחי נוטה להיות שמרני ב־GRC (ריקול נמוך למחלקת Rewrite). כאן אנחנו בוחרים סף החלטה ליוונית בלבד על סט **VAL** (למשל 0.23 במקום 0.50), ואז מעריכים על **TEST**.  
זה לא משנה את המודל — רק את פונקציית ההכרעה.

> הערה לתזה: זהו "post-hoc threshold tuning" פר־שפה.


In [53]:
import numpy as np
import os, json
from sklearn.metrics import classification_report, confusion_matrix

# Assumes: model, val_loader, test_loader, LANG2ID exist

@torch.no_grad()
def collect_probs_from_loader(loader):
    model.eval()
    ys, ps, probs, langs = [], [], [], []
    for tok, style_vec, char_vec, y, lang_id in loader:
        tok = {k: v.to(DEVICE) for k, v in tok.items()}
        style_vec = style_vec.to(DEVICE)
        char_vec  = char_vec.to(DEVICE)
        y = y.to(DEVICE)
        lang_id = lang_id.to(DEVICE)

        logits_y, _ = model(tok, style_vec, char_vec, grl_alpha=0.0)
        p = torch.softmax(logits_y, dim=-1)[:, 1]  # P(rewrite)

        probs.append(p.detach().cpu().numpy())
        ys.append(y.detach().cpu().numpy())
        langs.append(lang_id.detach().cpu().numpy())

    y_true = np.concatenate(ys)
    p_rw   = np.concatenate(probs)
    lang   = np.concatenate(langs)
    return y_true, p_rw, lang

def f1_for_positive(y_true, y_pred):
    # positive class = 1 (Rewrite)
    tp = int(((y_true==1) & (y_pred==1)).sum())
    fp = int(((y_true==0) & (y_pred==1)).sum())
    fn = int(((y_true==1) & (y_pred==0)).sum())
    prec = tp/(tp+fp) if (tp+fp) else 0.0
    rec  = tp/(tp+fn) if (tp+fn) else 0.0
    f1   = (2*prec*rec/(prec+rec)) if (prec+rec) else 0.0
    return f1, prec, rec, tp, fp, fn

def choose_best_threshold_for_greek(y_true, p_rw, lang, step=0.01):
    # Optimize F1 of Rewrite on Greek subset only
    m = (lang == LANG2ID["gr"])
    yt = y_true[m]
    pr = p_rw[m]

    best = {"thr":0.5, "f1":-1, "prec":0, "rec":0}
    for thr in np.arange(0.0, 1.0+1e-9, step):
        yp = (pr >= thr).astype(int)
        f1, prec, rec, tp, fp, fn = f1_for_positive(yt, yp)
        # tie-break: prefer higher recall (we want fewer FN on GR Rewrite)
        if (f1 > best["f1"]) or (abs(f1-best["f1"])<1e-12 and rec > best["rec"]):
            best = {"thr":float(thr), "f1":float(f1), "prec":float(prec), "rec":float(rec),
                    "tp":tp, "fp":fp, "fn":fn, "n":int(m.sum())}
    return best

def eval_with_lang_thresholds(y_true, p_rw, lang, thr_en=0.5, thr_gr=0.5, split_name="TEST"):
    yp = np.zeros_like(y_true)
    # EN
    m_en = (lang == LANG2ID["en"])
    yp[m_en] = (p_rw[m_en] >= thr_en).astype(int)
    # GR
    m_gr = (lang == LANG2ID["gr"])
    yp[m_gr] = (p_rw[m_gr] >= thr_gr).astype(int)

    print(f"\n=== {split_name} | thresholds: EN={thr_en:.2f}  GR={thr_gr:.2f} ===")
    print("Overall CM [[TN,FP],[FN,TP]]:\n", confusion_matrix(y_true, yp))
    print(classification_report(y_true, yp, target_names=["Original","Rewrite"], zero_division=0))

    for lname, lid in LANG2ID.items():
        m = (lang == lid)
        if m.sum() == 0:
            continue
        print(f"\n--- {split_name} | {lname.upper()} (n={m.sum()}) ---")
        print("CM [[TN,FP],[FN,TP]]:\n", confusion_matrix(y_true[m], yp[m]))
        print(classification_report(y_true[m], yp[m], target_names=["Original","Rewrite"], zero_division=0))

    return yp

# 1) choose GR threshold on VAL
y_val, p_val, lang_val = collect_probs_from_loader(val_loader)
best_thr = choose_best_threshold_for_greek(y_val, p_val, lang_val, step=0.01)
print("✅ Best Greek threshold on VAL:", best_thr)

# 2) apply to TEST (EN stays 0.50)
y_test, p_test, lang_test = collect_probs_from_loader(test_loader)
_ = eval_with_lang_thresholds(y_test, p_test, lang_test, thr_en=0.5, thr_gr=best_thr["thr"], split_name="TEST (calibrated)")

# 3) save for reproducibility
os.makedirs("outputs", exist_ok=True)
with open(os.path.join("outputs", "greek_threshold_calibration.json"), "w", encoding="utf-8") as f:
    json.dump({"best_greek_threshold_on_val": best_thr}, f, ensure_ascii=False, indent=2)
print("✅ Saved: outputs/greek_threshold_calibration.json")


✅ Best Greek threshold on VAL: {'thr': 0.43, 'f1': 0.8275862068965518, 'prec': 0.8, 'rec': 0.8571428571428571, 'tp': 12, 'fp': 3, 'fn': 2, 'n': 79}

=== TEST (calibrated) | thresholds: EN=0.50  GR=0.43 ===
Overall CM [[TN,FP],[FN,TP]]:
 [[ 93   1]
 [ 10 730]]
              precision    recall  f1-score   support

    Original       0.90      0.99      0.94        94
     Rewrite       1.00      0.99      0.99       740

    accuracy                           0.99       834
   macro avg       0.95      0.99      0.97       834
weighted avg       0.99      0.99      0.99       834


--- TEST (calibrated) | EN (n=754) ---
CM [[TN,FP],[FN,TP]]:
 [[ 28   0]
 [  5 721]]
              precision    recall  f1-score   support

    Original       0.85      1.00      0.92        28
     Rewrite       1.00      0.99      1.00       726

    accuracy                           0.99       754
   macro avg       0.92      1.00      0.96       754
weighted avg       0.99      0.99      0.99       754



# 🚨 ניסוי נוסף 2 — אימון מחדש: Gates גבוהים + Loss Boost ל־Greek Rewrite - מנצח

**מטרה:** להכריח את המודל להשתמש בפיצ'רים style/char במידה משמעותית יותר, ובמקביל להפחית FN במחלקת Rewrite ביוונית ע"י הגדלת תרומת הדוגמאות (gr & y=1) ב־loss.

שני השינויים:
1) אתחול gates ל־~0.80 (במקום ~0.27).  
2) Boost ללוס של דוגמאות Greek Rewrite (למשל ×3).

> טיפ: אם זה משפר GR recall אבל פוגע קצת ב-EN, זה עדיין יכול להיות "מודל יוונית-חזק" לפני ה־external test.


In [54]:
import numpy as np
import os
from sklearn.metrics import classification_report

# Hyperparams
LR = 2e-5
WEIGHT_DECAY = 0.01
# [EXP-A] EPOCHS increased from 3 to 5 (Section I BoostGR rerun)
EPOCHS = 5

LAMBDA_LANG = 0.3
GRL_ALPHA = 1.0

GATE_INIT = 0.80       # gates start high
BOOST_GR_REWRITE = 3.0 # loss multiplier for (gr & y=1)

def logit_from_prob(p):
    p = float(np.clip(p, 1e-6, 1-1e-6))
    return np.log(p/(1-p))

# Build a fresh model (same architecture)
model_sc = XLMR_StyleChar_Adv(MODEL_NAME, STYLE_DIM, CHAR_DIM).to(DEVICE)

# Initialize gates high
with torch.no_grad():
    model_sc.g_style.data.fill_(logit_from_prob(GATE_INIT))
    model_sc.g_char.data.fill_(logit_from_prob(GATE_INIT))

optimizer = AdamW(model_sc.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, int(0.05 * total_steps), total_steps)
scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

@torch.no_grad()
def eval_model_sc(loader, split="VAL"):
    model_sc.eval()
    ys, ps, langs = [], [], []
    for tok, style_vec, char_vec, y, lang_id in loader:
        tok = {k: v.to(DEVICE) for k, v in tok.items()}
        style_vec = style_vec.to(DEVICE)
        char_vec  = char_vec.to(DEVICE)
        y = y.to(DEVICE)
        lang_id = lang_id.to(DEVICE)

        logits_y, _ = model_sc(tok, style_vec, char_vec, grl_alpha=0.0)
        pred = torch.argmax(logits_y, dim=-1)

        ys.append(y.cpu().numpy())
        ps.append(pred.cpu().numpy())
        langs.append(lang_id.cpu().numpy())

    y_true = np.concatenate(ys)
    y_pred = np.concatenate(ps)
    lang_arr = np.concatenate(langs)

    print(f"\n=== {split} OVERALL ===")
    print(classification_report(y_true, y_pred, target_names=["Original","Rewrite"], zero_division=0))

    for name, lid in LANG2ID.items():
        m = (lang_arr == lid)
        if m.sum() == 0:
            continue
        print(f"\n=== {split} | {name.upper()} (n={m.sum()}) ===")
        print(classification_report(y_true[m], y_pred[m], target_names=["Original","Rewrite"], zero_division=0))

    rep = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    macro_f1 = rep["macro avg"]["f1-score"]

    # GR rewrite F1 focus
    m_gr = (lang_arr == LANG2ID["gr"])
    yt = y_true[m_gr]; yp = y_pred[m_gr]
    # rewrite class = 1
    tp = int(((yt==1) & (yp==1)).sum())
    fp = int(((yt==0) & (yp==1)).sum())
    fn = int(((yt==1) & (yp==0)).sum())
    prec = tp/(tp+fp) if (tp+fp) else 0.0
    rec  = tp/(tp+fn) if (tp+fn) else 0.0
    f1_gr_rw = (2*prec*rec/(prec+rec)) if (prec+rec) else 0.0

    return macro_f1, f1_gr_rw

best_score = -1.0
best_path = os.path.join("outputs", "xlmr_style_char_adv_gate08_boostgr_best__rerun1.pt")
os.makedirs("outputs", exist_ok=True)

for ep in range(1, EPOCHS+1):
    model_sc.train()
    total_loss = 0.0

    for tok, style_vec, char_vec, y, lang_id in train_loader:
        tok = {k: v.to(DEVICE) for k, v in tok.items()}
        style_vec = style_vec.to(DEVICE)
        char_vec  = char_vec.to(DEVICE)
        y = y.to(DEVICE)
        lang_id = lang_id.to(DEVICE)

        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            logits_y, logits_lang = model_sc(tok, style_vec, char_vec, grl_alpha=GRL_ALPHA)

            # per-sample rewrite loss
            loss_per = F.cross_entropy(logits_y, y, reduction="none")

            # boost Greek Rewrite examples
            boost = torch.ones_like(loss_per)
            boost[(lang_id == LANG2ID["gr"]) & (y == 1)] = BOOST_GR_REWRITE
            loss_y = (loss_per * boost).mean()

            loss_lang = F.cross_entropy(logits_lang, lang_id)
            loss = loss_y + LAMBDA_LANG * loss_lang

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        total_loss += float(loss.detach().cpu().item())

    gs = float(torch.sigmoid(model_sc.g_style).detach().cpu())
    gc = float(torch.sigmoid(model_sc.g_char).detach().cpu())
    print(f"\nEpoch {ep}/{EPOCHS} | train_loss={total_loss/max(1,len(train_loader)):.4f}")
    print(f"Gates (model_sc): g_style={gs:.3f}  g_char={gc:.3f}")

    macro_f1, gr_rw_f1 = eval_model_sc(val_loader, split="VAL")
    print("VAL macro-F1:", round(macro_f1, 4), "| VAL GR Rewrite F1:", round(gr_rw_f1, 4))

    # Selection criterion: prioritize Greek Rewrite F1, tie-break macro-F1
    score = gr_rw_f1 + 0.1 * macro_f1
    if score > best_score:
        best_score = score
        torch.save(model_sc.state_dict(), best_path)
        print("✅ Saved best model_sc:", best_path)

# Final test with best
model_sc.load_state_dict(torch.load(best_path, map_location=DEVICE))
_ = eval_model_sc(test_loader, split="TEST")
print("✅ Best saved at:", best_path)


C:\Users\Asoulin_Sapir\AppData\Local\Temp\ipykernel_26304\1642516277.py:31: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())
C:\Users\Asoulin_Sapir\AppData\Local\Temp\ipykernel_26304\1642516277.py:98: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):



Epoch 1/3 | train_loss=3.4948
Gates (model_sc): g_style=0.799  g_char=0.802

=== VAL OVERALL ===
              precision    recall  f1-score   support

    Original       0.92      0.88      0.90        93
     Rewrite       0.99      0.99      0.99       740

    accuracy                           0.98       833
   macro avg       0.95      0.94      0.94       833
weighted avg       0.98      0.98      0.98       833


=== VAL | EN (n=754) ===
              precision    recall  f1-score   support

    Original       0.77      0.86      0.81        28
     Rewrite       0.99      0.99      0.99       726

    accuracy                           0.99       754
   macro avg       0.88      0.92      0.90       754
weighted avg       0.99      0.99      0.99       754


=== VAL | GR (n=79) ===
              precision    recall  f1-score   support

    Original       1.00      0.89      0.94        65
     Rewrite       0.67      1.00      0.80        14

    accuracy                     

C:\Users\Asoulin_Sapir\AppData\Local\Temp\ipykernel_26304\1642516277.py:98: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):



Epoch 2/3 | train_loss=0.8648
Gates (model_sc): g_style=0.798  g_char=0.802

=== VAL OVERALL ===
              precision    recall  f1-score   support

    Original       0.91      0.94      0.92        93
     Rewrite       0.99      0.99      0.99       740

    accuracy                           0.98       833
   macro avg       0.95      0.96      0.96       833
weighted avg       0.98      0.98      0.98       833


=== VAL | EN (n=754) ===
              precision    recall  f1-score   support

    Original       0.78      0.89      0.83        28
     Rewrite       1.00      0.99      0.99       726

    accuracy                           0.99       754
   macro avg       0.89      0.94      0.91       754
weighted avg       0.99      0.99      0.99       754


=== VAL | GR (n=79) ===
              precision    recall  f1-score   support

    Original       0.97      0.95      0.96        65
     Rewrite       0.80      0.86      0.83        14

    accuracy                     

C:\Users\Asoulin_Sapir\AppData\Local\Temp\ipykernel_26304\1642516277.py:98: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):



Epoch 3/3 | train_loss=0.2666
Gates (model_sc): g_style=0.799  g_char=0.802

=== VAL OVERALL ===
              precision    recall  f1-score   support

    Original       0.93      0.92      0.93        93
     Rewrite       0.99      0.99      0.99       740

    accuracy                           0.98       833
   macro avg       0.96      0.96      0.96       833
weighted avg       0.98      0.98      0.98       833


=== VAL | EN (n=754) ===
              precision    recall  f1-score   support

    Original       0.83      0.89      0.86        28
     Rewrite       1.00      0.99      0.99       726

    accuracy                           0.99       754
   macro avg       0.91      0.94      0.93       754
weighted avg       0.99      0.99      0.99       754


=== VAL | GR (n=79) ===
              precision    recall  f1-score   support

    Original       0.98      0.94      0.96        65
     Rewrite       0.76      0.93      0.84        14

    accuracy                     

## ניסוי 3 — Experiment 2 + Post-hoc Greek Threshold Calibration (VAL→TEST)

In [56]:
# ============================
# EXPERIMENT 3 (ONE CELL) - FIXED:
# Exp2 model + Post-hoc Greek threshold calibration (VAL→TEST)
# Two policies that match thesis discussion:
#   (A) HIGH_RECALL: maximize Recall on VAL-GR (optional min_precision guard)
#   (B) FP_CAP: maximize Recall subject to FP<=FP_CAP on VAL-GR
# Then evaluate BOTH on TEST and save one JSON summary.
# ============================

import os, json
import numpy as np
import torch
from sklearn.metrics import confusion_matrix, classification_report

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# --------- CONFIG ----------
BEST_MODEL_PATH = r"outputs\xlmr_style_char_adv_gate08_boostgr_best__rerun1.pt"
OUT_JSON_PATH   = r"outputs\exp3_calibration_two_policies.json"

MAX_LEN = 256
BATCH_SIZE = 16
TAU_EN = 0.50

# Policy B: explicit FP control on VAL-GR
FP_CAP = 3

# Policy A: maximize recall but avoid collapsing precision (guardrail)
MIN_PRECISION_GUARD = 0.70   # set None to disable
THR_GRID = np.linspace(0.0, 1.0, 1001)  # step=0.001 עד 1.0
# --------------------------

os.makedirs("outputs", exist_ok=True)

# --------- 0) Load model ----------
if os.path.exists(BEST_MODEL_PATH):
    model_sc.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=DEVICE))
    model_sc.to(DEVICE).eval()
    print("✅ Loaded Experiment 2 best:", BEST_MODEL_PATH)
else:
    raise FileNotFoundError(f"Best model not found: {BEST_MODEL_PATH}")

@torch.no_grad()
def predict_probs_style_char(model, tokenizer, df, X_style, X_char, batch_size=BATCH_SIZE, max_len=MAX_LEN):
    model.eval()
    texts = df["text"].astype(str).tolist()
    probs = np.zeros(len(df), dtype=np.float32)

    for i in range(0, len(df), batch_size):
        batch_texts = texts[i:i+batch_size]
        tok = tokenizer(batch_texts, padding=True, truncation=True, max_length=max_len, return_tensors="pt")
        tok = {k: v.to(DEVICE) for k, v in tok.items()}

        style = torch.tensor(X_style[i:i+batch_size], dtype=torch.float32, device=DEVICE)
        char  = torch.tensor(X_char[i:i+batch_size],  dtype=torch.float32, device=DEVICE)

        logits_y, _ = model(tok, style, char, grl_alpha=0.0)  # eval
        probs[i:i+batch_size] = torch.softmax(logits_y, dim=-1)[:, 1].detach().cpu().numpy()

    return probs

def stats_from_thr(y_true, p, thr):
    pred = (p >= thr).astype(int)
    tp = int(((y_true==1) & (pred==1)).sum())
    fn = int(((y_true==1) & (pred==0)).sum())
    fp = int(((y_true==0) & (pred==1)).sum())
    tn = int(((y_true==0) & (pred==0)).sum())
    prec = tp/(tp+fp) if (tp+fp) else 0.0
    rec  = tp/(tp+fn) if (tp+fn) else 0.0
    f1   = (2*prec*rec/(prec+rec)) if (prec+rec) else 0.0
    return {"thr": float(thr), "tp": tp, "fp": fp, "fn": fn, "tn": tn,
            "precision": float(prec), "recall": float(rec), "f1": float(f1)}

def pick_high_recall(y_true, p, grid, min_precision=None):
    best = None
    for thr in grid:
        s = stats_from_thr(y_true, p, thr)
        if (min_precision is not None) and (s["precision"] < min_precision):
            continue
        # maximize recall, tie-break by higher precision, then by higher thr (more conservative)
        if (best is None) or (s["recall"] > best["recall"]) or \
           (s["recall"] == best["recall"] and s["precision"] > best["precision"]) or \
           (s["recall"] == best["recall"] and s["precision"] == best["precision"] and s["thr"] > best["thr"]):
            best = s
    return best

def pick_fp_cap(y_true, p, grid, fp_cap=3):
    best = None
    for thr in grid:
        s = stats_from_thr(y_true, p, thr)
        if s["fp"] <= fp_cap:
            # maximize recall, tie-break by higher precision, then higher thr
            if (best is None) or (s["recall"] > best["recall"]) or \
               (s["recall"] == best["recall"] and s["precision"] > best["precision"]) or \
               (s["recall"] == best["recall"] and s["precision"] == best["precision"] and s["thr"] > best["thr"]):
                best = s
    return best

def apply_lang_thresholds(df, p, tau_en=0.5, tau_gr=0.17):
    langs = df["lang"].astype(str).str.lower().values
    tau = np.where(langs == "gr", tau_gr, tau_en)
    return (p >= tau).astype(int)

def report_by_lang(name, df, y_true, y_pred):
    print(f"\n================ {name} ================")
    cm = confusion_matrix(y_true, y_pred)
    print("Overall CM [[TN,FP],[FN,TP]]:\n", cm)
    print(classification_report(y_true, y_pred, target_names=["Original","Rewrite"], zero_division=0))

    out = {"overall_cm": cm.tolist()}

    for lang in ["en","gr"]:
        m = (df["lang"].astype(str).str.lower().values == lang)
        if m.sum() == 0:
            continue
        cm_l = confusion_matrix(y_true[m], y_pred[m])
        print(f"\n--- {name} | {lang.upper()} (n={int(m.sum())}) ---")
        print("CM [[TN,FP],[FN,TP]]:\n", cm_l)
        print(classification_report(y_true[m], y_pred[m], target_names=["Original","Rewrite"], zero_division=0))

        # Rewrite focus
        tn, fp, fn, tp = cm_l.ravel()
        prec = tp/(tp+fp) if (tp+fp) else 0.0
        rec  = tp/(tp+fn) if (tp+fn) else 0.0
        f1   = (2*prec*rec/(prec+rec)) if (prec+rec) else 0.0
        out[f"{lang}_rewrite_focus"] = {"tp": int(tp), "fp": int(fp), "fn": int(fn),
                                        "precision": float(prec), "recall": float(rec), "f1": float(f1)}
    return out

# --------- 1) Compute probs on VAL (Greek only for calibration) ----------
p_val = predict_probs_style_char(model_sc, tokenizer, val_all, X_style_val, X_char_val)
mask_gr_val = (val_all["lang"].astype(str).str.lower().values == "gr")
y_val_gr = val_all.loc[mask_gr_val, "y"].astype(int).values
p_val_gr = p_val[mask_gr_val]

# ============================
# DEBUG: is FP_CAP feasible on current grid?
# ============================
def fp_at_thr(y_true, p, thr):
    pred = (p >= thr).astype(int)
    fp = int(((y_true==0) & (pred==1)).sum())
    tp = int(((y_true==1) & (pred==1)).sum())
    fn = int(((y_true==1) & (pred==0)).sum())
    tn = int(((y_true==0) & (pred==0)).sum())
    return fp, tp, fn, tn

fps = []
for thr in THR_GRID:
    fp, tp, fn, tn = fp_at_thr(y_val_gr, p_val_gr, thr)
    fps.append(fp)

fps = np.array(fps)
min_fp = int(fps.min())
thr_min_fp = float(THR_GRID[fps.argmin()])

print("VAL-GR size:", len(y_val_gr), "| positives:", int((y_val_gr==1).sum()), "| negatives:", int((y_val_gr==0).sum()))
print("Min FP achievable on this grid:", min_fp, "at thr=", thr_min_fp)
print("FP at thr=max(grid)=", float(THR_GRID.max()), "->", int(fps[-1]))

# Also check if we likely need thr>0.8
print("max p_val_gr:", float(np.max(p_val_gr)))
print("p_val_gr quantiles:", {q: float(np.quantile(p_val_gr, q)) for q in [0.5,0.9,0.95,0.99,0.995,0.999]})

# --------- 2) Pick τ_gr by two policies ----------
best_A = pick_high_recall(y_val_gr, p_val_gr, THR_GRID, min_precision=MIN_PRECISION_GUARD)
if best_A is None:
    raise RuntimeError("HIGH_RECALL policy found no threshold (try lowering MIN_PRECISION_GUARD or set it to None).")

best_B = pick_fp_cap(y_val_gr, p_val_gr, THR_GRID, fp_cap=FP_CAP)
if best_B is None:
    raise RuntimeError(f"FP_CAP policy found no threshold with FP<= {FP_CAP}. Try FP_CAP=4 or 5.")

TAU_GR_A = best_A["thr"]
TAU_GR_B = best_B["thr"]

print("\n[A] HIGH_RECALL (max recall, guard precision):", best_A, "| min_precision_guard:", MIN_PRECISION_GUARD)
print("[B] FP_CAP (max recall under FP cap):", best_B, "| FP_CAP:", FP_CAP)

# --------- 3) Evaluate BOTH on TEST ----------
p_test = predict_probs_style_char(model_sc, tokenizer, test_all, X_style_test, X_char_test)
y_test = test_all["y"].astype(int).values

yhat_A = apply_lang_thresholds(test_all, p_test, tau_en=TAU_EN, tau_gr=TAU_GR_A)
res_A  = report_by_lang(f"TEST | HIGH_RECALL (EN={TAU_EN:.2f}, GR={TAU_GR_A:.3f})", test_all, y_test, yhat_A)

yhat_B = apply_lang_thresholds(test_all, p_test, tau_en=TAU_EN, tau_gr=TAU_GR_B)
res_B  = report_by_lang(f"TEST | FP_CAP{FP_CAP} (EN={TAU_EN:.2f}, GR={TAU_GR_B:.3f})", test_all, y_test, yhat_B)

# --------- 4) Save summary JSON ----------
summary = {
    "model_path": BEST_MODEL_PATH,
    "tau_en": float(TAU_EN),
    "grid": {"min": float(THR_GRID.min()), "max": float(THR_GRID.max()), "n": int(len(THR_GRID))},
    "policy_A_high_recall": {"min_precision_guard": MIN_PRECISION_GUARD, "val_gr_choice": best_A},
    "policy_B_fp_cap": {"fp_cap": int(FP_CAP), "val_gr_choice": best_B},
    "test_results": {"HIGH_RECALL": res_A, f"FP_CAP{FP_CAP}": res_B},
}
with open(OUT_JSON_PATH, "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print("\n✅ Saved:", OUT_JSON_PATH)


✅ Loaded Experiment 2 best: outputs\xlmr_style_char_adv_gate08_boostgr_best__rerun1.pt
VAL-GR size: 79 | positives: 14 | negatives: 65
Min FP achievable on this grid: 0 at thr= 0.999
FP at thr=max(grid)= 1.0 -> 0
max p_val_gr: 0.9999626874923706
p_val_gr quantiles: {0.5: 0.0014328818069770932, 0.9: 0.999784529209137, 0.95: 0.9999105930328369, 0.99: 0.9999560713768005, 0.995: 0.999959409236908, 0.999: 0.9999620318412781}

[A] HIGH_RECALL (max recall, guard precision): {'thr': 0.184, 'tp': 14, 'fp': 5, 'fn': 0, 'tn': 60, 'precision': 0.7368421052631579, 'recall': 1.0, 'f1': 0.8484848484848484} | min_precision_guard: 0.7
[B] FP_CAP (max recall under FP cap): {'thr': 0.9560000000000001, 'tp': 13, 'fp': 1, 'fn': 1, 'tn': 64, 'precision': 0.9285714285714286, 'recall': 0.9285714285714286, 'f1': 0.9285714285714286} | FP_CAP: 3

================ TEST | HIGH_RECALL (EN=0.50, GR=0.184) ================
Overall CM [[TN,FP],[FN,TP]]:
 [[ 90   4]
 [  8 732]]
              precision    recall  f1-sco

In [57]:
# ============================
# VAL-GR Threshold Sweep Table (choose τ_gr by your target tradeoff)
# Produces a table: thr → TP/FP/FN/TN + Precision/Recall/F1/F2
# Also prints "best" thresholds under a few common policies.
# ============================

import numpy as np
import pandas as pd

BETA = 2.0  # F2 emphasizes recall

# ---- assumes you already have these from your notebook ----
# val_all: DataFrame with columns ["text","y","lang",...]
# p_val:   np.array of p(rewrite) for val_all (same order)
# -----------------------------------------------------------

assert "lang" in val_all.columns and "y" in val_all.columns, "val_all must include lang,y"
assert len(p_val) == len(val_all), "p_val must align with val_all rows"

# Filter Greek VAL
mask_gr = val_all["lang"].astype(str).str.lower().values == "gr"
y = val_all.loc[mask_gr, "y"].astype(int).values
p = p_val[mask_gr]

def compute_row(thr):
    pred = (p >= thr).astype(int)
    tp = int(((y==1) & (pred==1)).sum())
    fn = int(((y==1) & (pred==0)).sum())
    fp = int(((y==0) & (pred==1)).sum())
    tn = int(((y==0) & (pred==0)).sum())

    prec = tp/(tp+fp) if (tp+fp) else 0.0
    rec  = tp/(tp+fn) if (tp+fn) else 0.0
    f1   = (2*prec*rec/(prec+rec)) if (prec+rec) else 0.0

    b2 = BETA*BETA
    f2 = ((1+b2)*prec*rec/(b2*prec+rec)) if (prec+rec) else 0.0

    return {
        "thr": float(thr),
        "tp": tp, "fp": fp, "fn": fn, "tn": tn,
        "precision": prec, "recall": rec, "f1": f1, f"f{int(BETA)}": f2
    }

# Threshold grid (dense)
thr_grid = np.linspace(0.0, 0.8, 801)  # step 0.001
rows = [compute_row(t) for t in thr_grid]
tbl = pd.DataFrame(rows)

# Sort views
tbl_by_recall = tbl.sort_values(["recall","precision"], ascending=[False, False]).reset_index(drop=True)
tbl_by_f2     = tbl.sort_values([f"f{int(BETA)}","recall"], ascending=[False, False]).reset_index(drop=True)
tbl_by_f1     = tbl.sort_values(["f1","recall"], ascending=[False, False]).reset_index(drop=True)

print("VAL-GR size:", len(y), "| Rewrite positives:", int((y==1).sum()), "| Originals:", int((y==0).sum()))
print("Top-10 by Recall (tie-break Precision):")
display(tbl_by_recall.head(10))

print(f"Top-10 by F{int(BETA)} (recall-focused):")
display(tbl_by_f2.head(10))

print("Top-10 by F1:")
display(tbl_by_f1.head(10))

# ---- Helper: pick τ under constraints you care about ----
def pick_under_constraints(min_recall=None, min_precision=None, fp_cap=None, maximize="recall"):
    df = tbl.copy()
    if min_recall is not None:
        df = df[df["recall"] >= float(min_recall)]
    if min_precision is not None:
        df = df[df["precision"] >= float(min_precision)]
    if fp_cap is not None:
        df = df[df["fp"] <= int(fp_cap)]
    if len(df) == 0:
        return None
    if maximize == "recall":
        df = df.sort_values(["recall","precision"], ascending=[False, False])
    elif maximize == "f1":
        df = df.sort_values(["f1","recall"], ascending=[False, False])
    elif maximize == f"f{int(BETA)}":
        df = df.sort_values([f"f{int(BETA)}","recall"], ascending=[False, False])
    else:
        raise ValueError("maximize must be: 'recall', 'f1', or f'{int(BETA)}'")
    return df.iloc[0].to_dict()

# Suggestions (edit these numbers freely)
cand1 = pick_under_constraints(min_recall=0.85, min_precision=0.70, maximize="recall")
cand2 = pick_under_constraints(min_recall=0.90, min_precision=0.65, maximize="recall")
cand3 = pick_under_constraints(fp_cap=3, maximize="recall")

print("\nCandidate A (Recall>=0.85 & Precision>=0.70, maximize Recall):", cand1)
print("Candidate B (Recall>=0.90 & Precision>=0.65, maximize Recall):", cand2)
print("Candidate C (FP<=3, maximize Recall):", cand3)

# Finally: show the full table if you want (can be big)
# display(tbl)


VAL-GR size: 79 | Rewrite positives: 14 | Originals: 65
Top-10 by Recall (tie-break Precision):


,thr,tp,fp,fn,tn,precision,recall,f1,f2
0,0.024,14,5,0,60,0.736842,1.0,0.848485,0.933333
1,0.025,14,5,0,60,0.736842,1.0,0.848485,0.933333
2,0.026,14,5,0,60,0.736842,1.0,0.848485,0.933333
3,0.027,14,5,0,60,0.736842,1.0,0.848485,0.933333
4,0.028,14,5,0,60,0.736842,1.0,0.848485,0.933333
5,0.029,14,5,0,60,0.736842,1.0,0.848485,0.933333
6,0.030,14,5,0,60,0.736842,1.0,0.848485,0.933333
7,0.031,14,5,0,60,0.736842,1.0,0.848485,0.933333
8,0.032,14,5,0,60,0.736842,1.0,0.848485,0.933333
9,0.033,14,5,0,60,0.736842,1.0,0.848485,0.933333


Top-10 by F2 (recall-focused):


,thr,tp,fp,fn,tn,precision,recall,f1,f2
0,0.024,14,5,0,60,0.736842,1.0,0.848485,0.933333
1,0.025,14,5,0,60,0.736842,1.0,0.848485,0.933333
2,0.026,14,5,0,60,0.736842,1.0,0.848485,0.933333
3,0.027,14,5,0,60,0.736842,1.0,0.848485,0.933333
4,0.028,14,5,0,60,0.736842,1.0,0.848485,0.933333
5,0.029,14,5,0,60,0.736842,1.0,0.848485,0.933333
6,0.030,14,5,0,60,0.736842,1.0,0.848485,0.933333
7,0.031,14,5,0,60,0.736842,1.0,0.848485,0.933333
8,0.032,14,5,0,60,0.736842,1.0,0.848485,0.933333
9,0.033,14,5,0,60,0.736842,1.0,0.848485,0.933333


Top-10 by F1:


,thr,tp,fp,fn,tn,precision,recall,f1,f2
0,0.024,14,5,0,60,0.736842,1.0,0.848485,0.933333
1,0.025,14,5,0,60,0.736842,1.0,0.848485,0.933333
2,0.026,14,5,0,60,0.736842,1.0,0.848485,0.933333
3,0.027,14,5,0,60,0.736842,1.0,0.848485,0.933333
4,0.028,14,5,0,60,0.736842,1.0,0.848485,0.933333
5,0.029,14,5,0,60,0.736842,1.0,0.848485,0.933333
6,0.030,14,5,0,60,0.736842,1.0,0.848485,0.933333
7,0.031,14,5,0,60,0.736842,1.0,0.848485,0.933333
8,0.032,14,5,0,60,0.736842,1.0,0.848485,0.933333
9,0.033,14,5,0,60,0.736842,1.0,0.848485,0.933333



Candidate A (Recall>=0.85 & Precision>=0.70, maximize Recall): {'thr': 0.024, 'tp': 14.0, 'fp': 5.0, 'fn': 0.0, 'tn': 60.0, 'precision': 0.7368421052631579, 'recall': 1.0, 'f1': 0.8484848484848484, 'f2': 0.9333333333333333}
Candidate B (Recall>=0.90 & Precision>=0.65, maximize Recall): {'thr': 0.024, 'tp': 14.0, 'fp': 5.0, 'fn': 0.0, 'tn': 60.0, 'precision': 0.7368421052631579, 'recall': 1.0, 'f1': 0.8484848484848484, 'f2': 0.9333333333333333}
Candidate C (FP<=3, maximize Recall): None


In [58]:
# ============================
# EXPERIMENT 3 - FINAL (ONE CELL):
# Evaluate 2 best Greek thresholds found on VAL-GR:
#   A) High-Recall (τ_gr≈0.167): FN-minimizing (catch as many rewrites as possible)
#   B) FP-controlled  (τ_gr≈0.540): keep FP low (<=3 on VAL)
# Applies τ_en=0.50 for English in both settings.
# Prints TEST reports + saves a JSON summary.
# ============================

import os, json
import numpy as np
import torch
from sklearn.metrics import confusion_matrix, classification_report

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

BEST_MODEL_PATH = r"outputs\xlmr_style_char_adv_gate08_boostgr_best__rerun1.pt"
OUT_JSON_PATH   = r"outputs\exp3_final_two_thresholds.json"

MAX_LEN = 256
BATCH_SIZE = 16
TAU_EN = 0.50

# Two "best" Greek thresholds from your VAL-GR sweep
TAU_GR_HIGH_RECALL = 0.167   # TP=14 FN=0 FP=5  (VAL-GR)
TAU_GR_FP_CAP3     = 0.540   # TP=13 FN=1 FP=3  (VAL-GR)

os.makedirs("outputs", exist_ok=True)

# ---------- Load best Exp-2 model ----------
if os.path.exists(BEST_MODEL_PATH):
    model_sc.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=DEVICE))
    model_sc.to(DEVICE).eval()
    print("✅ Loaded Experiment 2 best:", BEST_MODEL_PATH)
else:
    raise FileNotFoundError(f"Best model not found: {BEST_MODEL_PATH}")

@torch.no_grad()
def predict_probs_style_char(model, tokenizer, df, X_style, X_char, batch_size=BATCH_SIZE, max_len=MAX_LEN):
    model.eval()
    texts = df["text"].astype(str).tolist()
    probs = np.zeros(len(df), dtype=np.float32)

    for i in range(0, len(df), batch_size):
        batch_texts = texts[i:i+batch_size]
        tok = tokenizer(batch_texts, padding=True, truncation=True, max_length=max_len, return_tensors="pt")
        tok = {k: v.to(DEVICE) for k, v in tok.items()}

        style = torch.tensor(X_style[i:i+batch_size], dtype=torch.float32, device=DEVICE)
        char  = torch.tensor(X_char[i:i+batch_size],  dtype=torch.float32, device=DEVICE)

        logits_y, _ = model(tok, style, char, grl_alpha=0.0)  # eval
        probs[i:i+batch_size] = torch.softmax(logits_y, dim=-1)[:, 1].detach().cpu().numpy()

    return probs

def apply_lang_thresholds(df, p, tau_en=0.5, tau_gr=0.167):
    langs = df["lang"].astype(str).str.lower().values
    tau = np.where(langs == "gr", tau_gr, tau_en)
    return (p >= tau).astype(int)

def eval_and_print(name, df, y_true, y_pred):
    print(f"\n================ {name} ================")
    cm = confusion_matrix(y_true, y_pred)
    print("Overall CM [[TN,FP],[FN,TP]]:\n", cm)
    print(classification_report(y_true, y_pred, target_names=["Original","Rewrite"], zero_division=0))

    out = {"overall_cm": cm.tolist()}

    for lang in ["en","gr"]:
        m = (df["lang"].astype(str).str.lower().values == lang)
        if m.sum() == 0:
            continue
        cm_l = confusion_matrix(y_true[m], y_pred[m])
        print(f"\n--- {name} | {lang.upper()} (n={int(m.sum())}) ---")
        print("CM [[TN,FP],[FN,TP]]:\n", cm_l)
        print(classification_report(y_true[m], y_pred[m], target_names=["Original","Rewrite"], zero_division=0))
        out[f"{lang}_cm"] = cm_l.tolist()

        # Focus metrics for Rewrite (class=1)
        tn, fp, fn, tp = cm_l.ravel()
        prec = tp/(tp+fp) if (tp+fp) else 0.0
        rec  = tp/(tp+fn) if (tp+fn) else 0.0
        f1   = (2*prec*rec/(prec+rec)) if (prec+rec) else 0.0
        out[f"{lang}_rewrite_focus"] = {"tp": int(tp), "fp": int(fp), "fn": int(fn),
                                        "precision": float(prec), "recall": float(rec), "f1": float(f1)}
    return out

# ---------- Predict probabilities on TEST ----------
p_test = predict_probs_style_char(model_sc, tokenizer, test_all, X_style_test, X_char_test)
y_test = test_all["y"].astype(int).values

# Setting A: High Recall (min FN)
yhat_A = apply_lang_thresholds(test_all, p_test, tau_en=TAU_EN, tau_gr=TAU_GR_HIGH_RECALL)
res_A = eval_and_print(f"HIGH_RECALL (EN={TAU_EN:.2f}, GR={TAU_GR_HIGH_RECALL:.3f})", test_all, y_test, yhat_A)

# Setting B: FP<=3 on VAL (more conservative)
yhat_B = apply_lang_thresholds(test_all, p_test, tau_en=TAU_EN, tau_gr=TAU_GR_FP_CAP3)
res_B = eval_and_print(f"FP_CAP3 (EN={TAU_EN:.2f}, GR={TAU_GR_FP_CAP3:.3f})", test_all, y_test, yhat_B)

# ---------- Save summary ----------
summary = {
    "model_path": BEST_MODEL_PATH,
    "tau_en": float(TAU_EN),
    "candidates": [
        {"name": "HIGH_RECALL", "tau_gr": float(TAU_GR_HIGH_RECALL), "val_gr_note": "TP=14 FN=0 FP=5 (VAL-GR)"},
        {"name": "FP_CAP3",     "tau_gr": float(TAU_GR_FP_CAP3),     "val_gr_note": "TP=13 FN=1 FP=3 (VAL-GR)"},
    ],
    "test_results": {"HIGH_RECALL": res_A, "FP_CAP3": res_B}
}

with open(OUT_JSON_PATH, "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print("\n✅ Saved summary JSON:", OUT_JSON_PATH)


✅ Loaded Experiment 2 best: outputs\xlmr_style_char_adv_gate08_boostgr_best__rerun1.pt

================ HIGH_RECALL (EN=0.50, GR=0.167) ================
Overall CM [[TN,FP],[FN,TP]]:
 [[ 90   4]
 [  8 732]]
              precision    recall  f1-score   support

    Original       0.92      0.96      0.94        94
     Rewrite       0.99      0.99      0.99       740

    accuracy                           0.99       834
   macro avg       0.96      0.97      0.96       834
weighted avg       0.99      0.99      0.99       834


--- HIGH_RECALL (EN=0.50, GR=0.167) | EN (n=754) ---
CM [[TN,FP],[FN,TP]]:
 [[ 26   2]
 [  4 722]]
              precision    recall  f1-score   support

    Original       0.87      0.93      0.90        28
     Rewrite       1.00      0.99      1.00       726

    accuracy                           0.99       754
   macro avg       0.93      0.96      0.95       754
weighted avg       0.99      0.99      0.99       754


--- HIGH_RECALL (EN=0.50, GR=0.167) |

## J) Ablation + Error Analysis (על המודל המנצח)
---


1) Ablation על הניסוי האחרון (style+char)
 המטרה: להוכיח בתזה האם ה־style/char באמת תורמים או מזיקים, בלי אימון מחדש.

תא A — פונקציית חיזוי על df + מטריקות overall וגם לפי שפה

In [59]:
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import classification_report, confusion_matrix

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

ID2LANG = {0:"en", 1:"gr"}
LANG2ID = {"en":0, "gr":1}

@torch.no_grad()
def predict_df_style_char(model, tokenizer, df, X_style, X_char, batch_size=16, max_len=256):
    model.eval()
    texts = df["text"].astype(str).tolist()
    y_true = df["y"].astype(int).to_numpy()
    lang_id = df["lang"].astype(str).str.lower().map(LANG2ID).to_numpy()

    probs = []
    preds = []

    for i in range(0, len(df), batch_size):
        batch_texts = texts[i:i+batch_size]
        tok = tokenizer(batch_texts, padding=True, truncation=True, max_length=max_len, return_tensors="pt")
        tok = {k:v.to(DEVICE) for k,v in tok.items()}

        style = torch.tensor(X_style[i:i+batch_size], dtype=torch.float32, device=DEVICE)
        char  = torch.tensor(X_char[i:i+batch_size],  dtype=torch.float32, device=DEVICE)

        logits_y, _ = model(tok, style, char, grl_alpha=0.0)
        p = torch.softmax(logits_y, dim=-1)[:,1].detach().cpu().numpy()
        pr = (p >= 0.5).astype(int)

        probs.append(p)
        preds.append(pr)

    probs = np.concatenate(probs)
    y_pred = np.concatenate(preds)

    return y_true, y_pred, probs, lang_id

def report_overall_and_by_lang(y_true, y_pred, lang_id, split_name="TEST"):
    print(f"\n=== {split_name} OVERALL ===")
    print(classification_report(y_true, y_pred, target_names=["Original","Rewrite"], zero_division=0))

    for lid, name in ID2LANG.items():
        m = (lang_id == lid)
        if m.sum() == 0: 
            continue
        print(f"\n=== {split_name} | {name.upper()} (n={m.sum()}) ===")
        print(classification_report(y_true[m], y_pred[m], target_names=["Original","Rewrite"], zero_division=0))


תא B — Ablation ע״י “כיבוי gates” (style off / char off)

In [60]:
import numpy as np
import torch

def inv_sigmoid(p):
    p = float(np.clip(p, 1e-6, 1-1e-6))
    return np.log(p/(1-p))

@torch.no_grad()
def eval_ablation_gates(model, tokenizer, df, X_style, X_char, g_style=None, g_char=None, label="TEST"):
    # Save old
    old_s = model.g_style.detach().clone()
    old_c = model.g_char.detach().clone()

    # Force gates if requested
    if g_style is not None:
        model.g_style.data.fill_(inv_sigmoid(g_style))
    if g_char is not None:
        model.g_char.data.fill_(inv_sigmoid(g_char))

    y_true, y_pred, probs, lang_id = predict_df_style_char(model, tokenizer, df, X_style, X_char)
    print(f"\n--- Ablation: g_style={g_style}, g_char={g_char} ---")
    report_overall_and_by_lang(y_true, y_pred, lang_id, split_name=label)

    # Restore
    model.g_style.data.copy_(old_s)
    model.g_char.data.copy_(old_c)

# דוגמאות להרצה על TEST (בהנחה שטענת את המודל האחרון + יש tokenizer + test_all + X_style_test + X_char_test)
eval_ablation_gates(model, tokenizer, test_all, X_style_test, X_char_test, g_style=None, g_char=None, label="TEST normal")
eval_ablation_gates(model, tokenizer, test_all, X_style_test, X_char_test, g_style=0.0,  g_char=None, label="TEST style OFF")
eval_ablation_gates(model, tokenizer, test_all, X_style_test, X_char_test, g_style=None, g_char=0.0,  label="TEST char OFF")
eval_ablation_gates(model, tokenizer, test_all, X_style_test, X_char_test, g_style=0.0,  g_char=0.0,  label="TEST style+char OFF")



--- Ablation: g_style=None, g_char=None ---

=== TEST normal OVERALL ===
              precision    recall  f1-score   support

    Original       0.90      0.99      0.94        94
     Rewrite       1.00      0.99      0.99       740

    accuracy                           0.99       834
   macro avg       0.95      0.99      0.97       834
weighted avg       0.99      0.99      0.99       834


=== TEST normal | EN (n=754) ===
              precision    recall  f1-score   support

    Original       0.85      1.00      0.92        28
     Rewrite       1.00      0.99      1.00       726

    accuracy                           0.99       754
   macro avg       0.92      1.00      0.96       754
weighted avg       0.99      0.99      0.99       754


=== TEST normal | GR (n=80) ===
              precision    recall  f1-score   support

    Original       0.93      0.98      0.96        66
     Rewrite       0.90      0.64      0.75        14

    accuracy                           0.

2) Error analysis על הניסוי האחרון
   המטרה: להוציא טבלת טעויות + דוגמאות FP/FN “בביטחון גבוה” + פוקוס על GR Rewrite.

תא C — יצירת טבלת תוצאות + Confusion matrices

In [61]:
from sklearn.metrics import confusion_matrix

y_true, y_pred, probs, lang_id = predict_df_style_char(model, tokenizer, test_all, X_style_test, X_char_test)

res = test_all.copy().reset_index(drop=True)
res["lang"] = res["lang"].astype(str).str.lower()
res["y_true"] = y_true
res["y_pred"] = y_pred
res["p_rewrite"] = probs

res["error_type"] = "OK"
res.loc[(res.y_true==0) & (res.y_pred==1), "error_type"] = "FP (Original→Rewrite)"
res.loc[(res.y_true==1) & (res.y_pred==0), "error_type"] = "FN (Rewrite→Original)"

print("Overall confusion matrix [ [TN, FP], [FN, TP] ]:")
print(confusion_matrix(res["y_true"], res["y_pred"]))

for lang in ["en","gr"]:
    sub = res[res["lang"]==lang]
    print(f"\n{lang.upper()} confusion matrix [ [TN, FP], [FN, TP] ]:")
    print(confusion_matrix(sub["y_true"], sub["y_pred"]))


Overall confusion matrix [ [TN, FP], [FN, TP] ]:
[[ 93   1]
 [ 10 730]]

EN confusion matrix [ [TN, FP], [FN, TP] ]:
[[ 28   0]
 [  5 721]]

GR confusion matrix [ [TN, FP], [FN, TP] ]:
[[65  1]
 [ 5  9]]


תא D — דוגמאות טעויות הכי “בטוחות” (Top-K FP/FN)

In [62]:
def show_top_errors(df, kind="FP", k=12, text_col="text"):
    if kind=="FP":
        d = df[df["error_type"].str.startswith("FP")].copy()
        # FP = predicted rewrite with high confidence -> p_rewrite high
        d = d.sort_values("p_rewrite", ascending=False).head(k)
    else:
        d = df[df["error_type"].str.startswith("FN")].copy()
        # FN = predicted original; confidence for rewrite is low
        d = d.sort_values("p_rewrite", ascending=True).head(k)

    return d[["lang","y_true","y_pred","p_rewrite",text_col,"error_type"]]

print("\nTop FP overall:")
display(show_top_errors(res, "FP", k=10))

print("\nTop FN overall:")
display(show_top_errors(res, "FN", k=10))

print("\nTop FP in GR:")
display(show_top_errors(res[res.lang=="gr"], "FP", k=10))

print("\nTop FN in GR:")
display(show_top_errors(res[res.lang=="gr"], "FN", k=10))



Top FP overall:


,lang,y_true,y_pred,p_rewrite,text,error_type
489,gr,0,1,0.999727,"Ὀσαρσὴφ γάρ, φησίν, ἐκαλεῖτο.",FP (Original→Rewrite)



Top FN overall:


,lang,y_true,y_pred,p_rewrite,text,error_type
780,gr,1,0,0.001900,τῇ δ᾽ ἐπιούσῃ ἡμέρᾳ Μωσῆν τινα συμβουλεῦσαι αὐ...,FN (Rewrite→Original)
18,gr,1,0,0.003776,"λέγει γάρ, ὅτι ὁ μὲν Σέθως ἐκαλεῖτο Αἴγυπτος, ...",FN (Rewrite→Original)
234,en,1,0,0.005576,But now questions must give way to other delig...,FN (Rewrite→Original)
701,en,1,0,0.008450,"Among dragons and morning mist, he learns thei...",FN (Rewrite→Original)
119,gr,1,0,0.012634,τούτου υἱὸς Λαβοροσοάρδοχος ἐκυρίευσε μὲν τῆς ...,FN (Rewrite→Original)
501,en,1,0,0.025069,The goal posts loom.,FN (Rewrite→Original)
561,en,1,0,0.043469,Only shadow and vapor… A shape that needs anot...,FN (Rewrite→Original)
77,gr,1,0,0.131070,ἐπάξειν μὲν οὖν αὐτοὺς ἐπηγγείλατο πρῶτον μὲν ...,FN (Rewrite→Original)
288,gr,1,0,0.173740,"σαφῶς δ᾽ ἴσθι, εἶπεν, Ὑπεροχίδη, θαυμαστὸν ὀνε...",FN (Rewrite→Original)
275,en,1,0,0.380541,"In the shadowed corridor, words catch in the q...",FN (Rewrite→Original)



Top FP in GR:


,lang,y_true,y_pred,p_rewrite,text,error_type
489,gr,0,1,0.999727,"Ὀσαρσὴφ γάρ, φησίν, ἐκαλεῖτο.",FP (Original→Rewrite)



Top FN in GR:


,lang,y_true,y_pred,p_rewrite,text,error_type
780,gr,1,0,0.001900,τῇ δ᾽ ἐπιούσῃ ἡμέρᾳ Μωσῆν τινα συμβουλεῦσαι αὐ...,FN (Rewrite→Original)
18,gr,1,0,0.003776,"λέγει γάρ, ὅτι ὁ μὲν Σέθως ἐκαλεῖτο Αἴγυπτος, ...",FN (Rewrite→Original)
119,gr,1,0,0.012634,τούτου υἱὸς Λαβοροσοάρδοχος ἐκυρίευσε μὲν τῆς ...,FN (Rewrite→Original)
77,gr,1,0,0.131070,ἐπάξειν μὲν οὖν αὐτοὺς ἐπηγγείλατο πρῶτον μὲν ...,FN (Rewrite→Original)
288,gr,1,0,0.173740,"σαφῶς δ᾽ ἴσθι, εἶπεν, Ὑπεροχίδη, θαυμαστὸν ὀνε...",FN (Rewrite→Original)


תא E — דגש תזה: “GR Rewrite שהוחמצו” (FN ביוונית בלבד)

In [63]:
gr_fn = res[(res.lang=="gr") & (res.y_true==1) & (res.y_pred==0)].copy()
print("GR FN count:", len(gr_fn))
display(gr_fn.sort_values("p_rewrite", ascending=True)[["p_rewrite","text"]].head(15))


GR FN count: 5


,p_rewrite,text
780,0.001900,τῇ δ᾽ ἐπιούσῃ ἡμέρᾳ Μωσῆν τινα συμβουλεῦσαι αὐ...
18,0.003776,"λέγει γάρ, ὅτι ὁ μὲν Σέθως ἐκαλεῖτο Αἴγυπτος, ..."
119,0.012634,τούτου υἱὸς Λαβοροσοάρδοχος ἐκυρίευσε μὲν τῆς ...
77,0.131070,ἐπάξειν μὲν οὖν αὐτοὺς ἐπηγγείλατο πρῶτον μὲν ...
288,0.173740,"σαφῶς δ᾽ ἴσθι, εἶπεν, Ὑπεροχίδη, θαυμαστὸν ὀνε..."


תא F — שמירה לקובץ (לטובת פרק Error Analysis בתזה)

In [64]:
os.makedirs("outputs", exist_ok=True)
out_path = r"outputs\error_analysis_style_char.xlsx"
res.to_excel(out_path, index=False)
print("✅ Saved:", out_path)


✅ Saved: outputs\error_analysis_style_char.xlsx


תא 1 — פונקציות חיזוי + דוחות לפי שפה

In [65]:
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import confusion_matrix, classification_report

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
LANG2ID = {"en": 0, "gr": 1}
ID2LANG = {0: "en", 1: "gr"}

@torch.no_grad()
def predict_style_char(model, tokenizer, df, X_style, X_char, batch_size=16, max_len=256):
    model.eval()

    texts = df["text"].astype(str).tolist()
    y_true = df["y"].astype(int).to_numpy()
    lang_id = df["lang"].astype(str).str.lower().map(LANG2ID).to_numpy()

    probs = np.zeros(len(df), dtype=np.float32)
    preds = np.zeros(len(df), dtype=np.int32)

    for i in range(0, len(df), batch_size):
        batch_texts = texts[i:i+batch_size]

        tok = tokenizer(batch_texts, padding=True, truncation=True, max_length=max_len, return_tensors="pt")
        tok = {k: v.to(DEVICE) for k, v in tok.items()}

        style = torch.tensor(X_style[i:i+batch_size], dtype=torch.float32, device=DEVICE)
        char  = torch.tensor(X_char[i:i+batch_size],  dtype=torch.float32, device=DEVICE)

        logits_y, _ = model(tok, style, char, grl_alpha=0.0)
        p = torch.softmax(logits_y, dim=-1)[:, 1].detach().cpu().numpy()
        pr = (p >= 0.5).astype(int)

        probs[i:i+batch_size] = p
        preds[i:i+batch_size] = pr

    return y_true, preds, probs, lang_id

def cm_by_lang(y_true, y_pred, lang_id):
    out = {}
    out["overall"] = confusion_matrix(y_true, y_pred)

    for lid, lname in ID2LANG.items():
        m = (lang_id == lid)
        if m.sum() == 0:
            continue
        out[lname] = confusion_matrix(y_true[m], y_pred[m])

    return out

def gr_rewrite_metrics(y_true, y_pred, lang_id):
    m = (lang_id == LANG2ID["gr"])
    yt = y_true[m]; yp = y_pred[m]
    # rewrite is class 1
    tp = int(((yt==1) & (yp==1)).sum())
    fn = int(((yt==1) & (yp==0)).sum())
    fp = int(((yt==0) & (yp==1)).sum())

    prec = tp / (tp + fp) if (tp + fp) else 0.0
    rec  = tp / (tp + fn) if (tp + fn) else 0.0
    f1   = (2*prec*rec/(prec+rec)) if (prec+rec) else 0.0
    return {"tp":tp,"fp":fp,"fn":fn,"precision":prec,"recall":rec,"f1":f1}

def print_summary(tag, y_true, y_pred, probs, lang_id):
    print(f"\n================ {tag} ================")

    cms = cm_by_lang(y_true, y_pred, lang_id)
    print("Overall confusion matrix [ [TN, FP], [FN, TP] ]:\n", cms["overall"])
    if "en" in cms:
        print("\nEN confusion matrix [ [TN, FP], [FN, TP] ]:\n", cms["en"])
    if "gr" in cms:
        print("\nGR confusion matrix [ [TN, FP], [FN, TP] ]:\n", cms["gr"])

    print("\nReport OVERALL:")
    print(classification_report(y_true, y_pred, target_names=["Original","Rewrite"], zero_division=0))

    # Greek rewrite focus
    grm = gr_rewrite_metrics(y_true, y_pred, lang_id)
    print("\nGreek Rewrite focus:", grm)

    return cms, grm


תא 2 — Ablation על gates: normal / style off / char off / both off

In [66]:
import numpy as np
import torch

def inv_sigmoid(p):
    p = float(np.clip(p, 1e-6, 1-1e-6))
    return np.log(p/(1-p))

@torch.no_grad()
def run_ablation(model, tokenizer, df, X_style, X_char, batch_size=16, max_len=256):
    # save original gate params
    old_style = model.g_style.detach().clone()
    old_char  = model.g_char.detach().clone()

    runs = [
        ("NORMAL",        None, None),
        ("STYLE_OFF",     0.0,  None),
        ("CHAR_OFF",      None, 0.0),
        ("STYLE+CHAR_OFF",0.0,  0.0),
    ]

    summary_rows = []

    for tag, g_style, g_char in runs:
        # force gates if requested
        if g_style is not None:
            model.g_style.data.fill_(inv_sigmoid(g_style))
        else:
            model.g_style.data.copy_(old_style)

        if g_char is not None:
            model.g_char.data.fill_(inv_sigmoid(g_char))
        else:
            model.g_char.data.copy_(old_char)

        # predict + summarize
        y_true, y_pred, probs, lang_id = predict_style_char(
            model, tokenizer, df, X_style, X_char,
            batch_size=batch_size, max_len=max_len
        )
        cms, grm = print_summary(tag, y_true, y_pred, probs, lang_id)

        summary_rows.append({
            "setting": tag,
            "g_style_forced": g_style,
            "g_char_forced": g_char,
            "GR_tp": grm["tp"],
            "GR_fp": grm["fp"],
            "GR_fn": grm["fn"],
            "GR_precision": grm["precision"],
            "GR_recall": grm["recall"],
            "GR_f1": grm["f1"],
        })

    # restore
    model.g_style.data.copy_(old_style)
    model.g_char.data.copy_(old_char)

    return pd.DataFrame(summary_rows)

abl_df = run_ablation(model, tokenizer, test_all, X_style_test, X_char_test, batch_size=16, max_len=256)
display(abl_df)



================ NORMAL ================
Overall confusion matrix [ [TN, FP], [FN, TP] ]:
 [[ 93   1]
 [ 10 730]]

EN confusion matrix [ [TN, FP], [FN, TP] ]:
 [[ 28   0]
 [  5 721]]

GR confusion matrix [ [TN, FP], [FN, TP] ]:
 [[65  1]
 [ 5  9]]

Report OVERALL:
              precision    recall  f1-score   support

    Original       0.90      0.99      0.94        94
     Rewrite       1.00      0.99      0.99       740

    accuracy                           0.99       834
   macro avg       0.95      0.99      0.97       834
weighted avg       0.99      0.99      0.99       834


Greek Rewrite focus: {'tp': 9, 'fp': 1, 'fn': 5, 'precision': 0.9, 'recall': 0.6428571428571429, 'f1': 0.75}

================ STYLE_OFF ================
Overall confusion matrix [ [TN, FP], [FN, TP] ]:
 [[ 93   1]
 [ 10 730]]

EN confusion matrix [ [TN, FP], [FN, TP] ]:
 [[ 28   0]
 [  5 721]]

GR confusion matrix [ [TN, FP], [FN, TP] ]:
 [[65  1]
 [ 5  9]]

Report OVERALL:
              precision    r

,setting,g_style_forced,g_char_forced,GR_tp,GR_fp,GR_fn,GR_precision,GR_recall,GR_f1
0,NORMAL,NaN,NaN,9,1,5,0.9,0.642857,0.75
1,STYLE_OFF,0.0,NaN,9,1,5,0.9,0.642857,0.75
2,CHAR_OFF,NaN,0.0,9,1,5,0.9,0.642857,0.75
3,STYLE+CHAR_OFF,0.0,0.0,9,1,5,0.9,0.642857,0.75


תא 3 — שמירה לאקסל + מסקנה “מי מזיק”

In [67]:
import os

os.makedirs("outputs", exist_ok=True)
out_path = r"outputs\ablation_style_char_summary.xlsx"
abl_df.to_excel(out_path, index=False)
print("✅ Saved:", out_path)

# quick conclusion helper
base = abl_df[abl_df["setting"]=="NORMAL"].iloc[0]
best = abl_df.sort_values("GR_f1", ascending=False).iloc[0]

print("\nBase (NORMAL) GR_f1:", round(float(base["GR_f1"]), 4))
print("Best setting:", best["setting"], "| GR_f1:", round(float(best["GR_f1"]), 4))

if best["setting"] == "STYLE_OFF":
    print("➡️ Indication: STYLE branch is harming Greek rewrite (turning it off helps).")
elif best["setting"] == "CHAR_OFF":
    print("➡️ Indication: CHAR branch is harming Greek rewrite (turning it off helps).")
elif best["setting"] == "STYLE+CHAR_OFF":
    print("➡️ Indication: Both branches together harm Greek rewrite (text-only behavior is best).")
else:
    print("➡️ Indication: Keeping both branches is best (they help or at least don't hurt).")


✅ Saved: outputs\ablation_style_char_summary.xlsx

Base (NORMAL) GR_f1: 0.75
Best setting: NORMAL | GR_f1: 0.75
➡️ Indication: Keeping both branches is best (they help or at least don't hurt).


תא בדיקה: האם p_rewrite משתנה כשמכבים ענפים?

In [68]:
import numpy as np
import torch

def inv_sigmoid(p):
    p = float(np.clip(p, 1e-6, 1-1e-6))
    return np.log(p/(1-p))

@torch.no_grad()
def get_probs_with_forced_gates(g_style=None, g_char=None, batch_size=32):
    old_s = model.g_style.detach().clone()
    old_c = model.g_char.detach().clone()

    if g_style is not None:
        model.g_style.data.fill_(inv_sigmoid(g_style))
    else:
        model.g_style.data.copy_(old_s)

    if g_char is not None:
        model.g_char.data.fill_(inv_sigmoid(g_char))
    else:
        model.g_char.data.copy_(old_c)

    y_true, y_pred, probs, lang_id = predict_style_char(model, tokenizer, test_all, X_style_test, X_char_test, batch_size=batch_size)
    
    model.g_style.data.copy_(old_s)
    model.g_char.data.copy_(old_c)
    return probs

p_normal = get_probs_with_forced_gates(None, None)
p_style0 = get_probs_with_forced_gates(0.0, None)
p_char0  = get_probs_with_forced_gates(None, 0.0)
p_both0  = get_probs_with_forced_gates(0.0, 0.0)

def diff_stats(a, b, name):
    d = np.abs(a - b)
    print(name, "max|Δ| =", d.max(), "mean|Δ| =", d.mean(), "changed >", 1e-6, "count =", int((d > 1e-6).sum()))

diff_stats(p_normal, p_style0, "NORMAL vs STYLE_OFF")
diff_stats(p_normal, p_char0,  "NORMAL vs CHAR_OFF")
diff_stats(p_normal, p_both0,  "NORMAL vs BOTH_OFF")


NORMAL vs STYLE_OFF max|Δ| = 0.010359287 mean|Δ| = 6.1431776e-05 changed > 1e-06 count = 730
NORMAL vs CHAR_OFF max|Δ| = 0.0033242404 mean|Δ| = 4.6727826e-05 changed > 1e-06 count = 785
NORMAL vs BOTH_OFF max|Δ| = 0.007735491 mean|Δ| = 6.5704386e-05 changed > 1e-06 count = 753


In [69]:
print("Style: min/max per feature:", X_style_test.min(axis=0)[:5], X_style_test.max(axis=0)[:5])
print("Style: std mean:", X_style_test.std(axis=0).mean())

print("Char: row norms (first 10):", np.linalg.norm(X_char_test[:10].astype(np.float32), axis=1))
print("Char: global std:", X_char_test.astype(np.float32).std())
print("Char: fraction exactly zero rows:", float((np.linalg.norm(X_char_test.astype(np.float32), axis=1) == 0).mean()))


Style: min/max per feature: [-1.5670699 -1.5906559 -2.6631684 -3.7710063 -1.4368519] [7.0880036 6.392029  6.7862034 5.125709  7.133266 ]
Style: std mean: 0.9797862
Char: row norms (first 10): [1.00007    1.0004617  0.99992186 1.0002093  0.9998894  1.0003924
 0.9999046  0.99998057 1.0000597  1.000022  ]
Char: global std: 0.015185902
Char: fraction exactly zero rows: 0.0


In [70]:
import numpy as np
import pandas as pd

# probs שכבר חישבת:
# p_normal, p_style0, p_char0, p_both0

df = test_all.reset_index(drop=True).copy()
df["lang"] = df["lang"].astype(str).str.lower()
df["y"] = df["y"].astype(int).values

# predictions under normal (threshold 0.5)
pred_normal = (p_normal >= 0.5).astype(int)

df["p_normal"] = p_normal
df["pred_normal"] = pred_normal

df["delta_style"] = np.abs(p_normal - p_style0)
df["delta_char"]  = np.abs(p_normal - p_char0)

# define error types under NORMAL
df["err"] = "OK"
df.loc[(df.y==0) & (df.pred_normal==1), "err"] = "FP"
df.loc[(df.y==1) & (df.pred_normal==0), "err"] = "FN"

gr = df[df.lang=="gr"]

print("Greek Δ stats overall:")
print(gr[["delta_style","delta_char"]].describe())

print("\nGreek Δ stats by error type:")
for e in ["FN", "FP", "OK"]:
    sub = gr[gr.err == e]
    print(f"\n{e} n={len(sub)}")
    if len(sub) == 0:
        continue
    desc = sub[["delta_style","delta_char"]].describe()
    # סטטיסטיקות הן בשורות, לכן משתמשים ב-loc
    print(desc.loc[["mean","50%","max"]])



Greek Δ stats overall:
       delta_style  delta_char
count    80.000000   80.000000
mean      0.000442    0.000149
std       0.001576    0.000475
min       0.000012    0.000003
25%       0.000028    0.000013
50%       0.000047    0.000019
75%       0.000081    0.000031
max       0.010359    0.002661

Greek Δ stats by error type:

FN n=5
      delta_style  delta_char
mean     0.002077    0.000799
50%      0.000429    0.000244
max      0.006222    0.001844

FP n=1
      delta_style  delta_char
mean     0.000012    0.000003
50%      0.000012    0.000003
max      0.000012    0.000003

OK n=74
      delta_style  delta_char
mean     0.000337    0.000107
50%      0.000046    0.000019
max      0.010359    0.002661


## 🔬 EXP Pipeline (Auto-added)
תאים אלו נוספו כדי להריץ את EXP1→EXP2→EXP3 בצורה יציבה. הריצי לפי הסדר.

### 0) Tokenizer override (offline-safe)

In [14]:
# --- Ensure MODEL_NAME exists + load tokenizer from local cache (offline-safe) ---
from transformers import AutoTokenizer

if "MODEL_NAME" not in globals():
    MODEL_NAME = "xlm-roberta-base"   # default
    print("ℹ️ MODEL_NAME was not set yet. Using default:", MODEL_NAME)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, local_files_only=True)
print("✅ tokenizer loaded locally:", MODEL_NAME)

ℹ️ MODEL_NAME was not set yet. Using default: xlm-roberta-base
✅ tokenizer loaded locally: xlm-roberta-base


In [18]:
import re, numpy as np

WS_RE = re.compile(r"\s+")

def style_features(s: str):
    s = "" if s is None else str(s)
    n = max(len(s), 1)
    words = [w for w in WS_RE.split(s.strip()) if w]
    wlen = [len(w) for w in words]
    return [
        len(words),
        float(np.mean(wlen)) if wlen else 0.0,
        float(np.std(wlen)) if wlen else 0.0,
        s.count(" ") / n,
        sum(c.isdigit() for c in s) / n,
        sum(c in ".,;:!?" for c in s) / n,
        len(set(s)) / n,
    ]

def build_style_matrix(df):
    texts = df["text"].fillna("").astype(str).tolist()
    X = np.array([style_features(t) for t in texts], dtype=np.float32)
    return X

print("✅ build_style_matrix is ready")

✅ build_style_matrix is ready


In [20]:
import numpy as np
import torch
from transformers import AutoTokenizer

# Ensure basics exist
if "MODEL_NAME" not in globals():
    MODEL_NAME = "xlm-roberta-base"
if "MAX_LEN" not in globals():
    MAX_LEN = 192
if "BATCH" not in globals():
    BATCH = 2

# Offline-safe tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, local_files_only=True)

def collate_multibranch(batch):
    texts = [b["text"] for b in batch]
    y = torch.tensor([b["y"] for b in batch], dtype=torch.long)
    lang_id = torch.tensor([b["lang_id"] for b in batch], dtype=torch.long)

    tok = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=MAX_LEN,
        return_tensors="pt"
    )

    style = torch.tensor(np.stack([b["style"] for b in batch]), dtype=torch.float32)
    char  = torch.tensor(np.stack([b["char"]  for b in batch]), dtype=torch.float32)

    return tok, style, char, y, lang_id

print("✅ collate_multibranch is ready | MAX_LEN=", MAX_LEN, "| BATCH=", BATCH)

✅ collate_multibranch is ready | MAX_LEN= 192 | BATCH= 2


### 1) EXP1 — Downsample EN_1 + rebuild STYLE/CHAR + loaders

In [21]:
# ==========================
# EXP1: Downsample EN_1 in TRAIN only + rebuild STYLE/CHAR + loaders
# ==========================
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import HashingVectorizer
from torch.utils.data import DataLoader, WeightedRandomSampler
import torch, os, joblib

SEED = 42
rng = np.random.default_rng(SEED)

assert "train_all" in globals() and "val_all" in globals() and "test_all" in globals(), "Missing splits (train_all/val_all/test_all)."
assert "build_style_matrix" in globals(), "build_style_matrix(df) not found. Run the Style features cell first."
assert "collate_multibranch" in globals(), "collate_multibranch not found. Run the original loader/collate cell first."

train_all = train_all.copy()
train_all["lang"] = train_all["lang"].astype(str).str.lower()

n_gr0 = len(train_all[(train_all.lang=="gr") & (train_all.y==0)])
target_n = n_gr0
print("Target EN_1 size =", target_n, "(matching GR_0)")

en1_idx = train_all[(train_all.lang=="en") & (train_all.y==1)].index.to_numpy()
keep_en1 = rng.choice(en1_idx, size=min(target_n, len(en1_idx)), replace=False)

keep_idx = train_all.index.difference(en1_idx).to_numpy()
keep_idx = np.concatenate([keep_idx, keep_en1])
train_all = train_all.loc[keep_idx].sample(frac=1.0, random_state=SEED).reset_index(drop=True)

print("Train after downsample:", len(train_all))
print("Train strata:")
print((train_all["lang"]+"_"+train_all["y"].astype(str)).value_counts())

# --- STYLE (fit scaler on TRAIN only) ---
X_style_train_raw = build_style_matrix(train_all)
X_style_val_raw   = build_style_matrix(val_all)
X_style_test_raw  = build_style_matrix(test_all)

style_scaler = StandardScaler()
X_style_train = style_scaler.fit_transform(X_style_train_raw).astype(np.float32)
X_style_val   = style_scaler.transform(X_style_val_raw).astype(np.float32)
X_style_test  = style_scaler.transform(X_style_test_raw).astype(np.float32)

os.makedirs("outputs", exist_ok=True)
joblib.dump(style_scaler, "outputs/style_scaler_EXP1.pkl")
print("✅ saved outputs/style_scaler_EXP1.pkl")

# --- CHAR hashing ---
CHAR_DIM = 4096
char_vec = HashingVectorizer(
    n_features=CHAR_DIM, analyzer="char", ngram_range=(3, 5),
    alternate_sign=False, norm=None
)
X_char_train = char_vec.transform(train_all["text"].astype(str)).toarray().astype(np.float16)
X_char_val   = char_vec.transform(val_all["text"].astype(str)).toarray().astype(np.float16)
X_char_test  = char_vec.transform(test_all["text"].astype(str)).toarray().astype(np.float16)

# --- Loaders ---
LANG2ID = {"en": 0, "gr": 1}

class MultiBranchDS(torch.utils.data.Dataset):
    def __init__(self, df, X_style, X_char):
        self.df = df.reset_index(drop=True)
        self.X_style = X_style
        self.X_char = X_char
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        r = self.df.iloc[idx]
        lang = str(r["lang"]).lower()
        return {
            "text": str(r["text"]),
            "y": int(r["y"]),
            "lang_id": LANG2ID[lang],
            "style": self.X_style[idx],
            "char": self.X_char[idx],
        }

strata = train_all["lang"].astype(str).str.lower() + "_" + train_all["y"].astype(str)
counts = strata.value_counts()
weights = strata.map(lambda s: 1.0 / counts[s]).values

sampler = WeightedRandomSampler(torch.tensor(weights, dtype=torch.double),
                                num_samples=len(weights), replacement=True)

train_loader = DataLoader(MultiBranchDS(train_all, X_style_train, X_char_train),
                          batch_size=BATCH, sampler=sampler, collate_fn=collate_multibranch)
val_loader   = DataLoader(MultiBranchDS(val_all, X_style_val, X_char_val),
                          batch_size=BATCH, shuffle=False, collate_fn=collate_multibranch)
test_loader  = DataLoader(MultiBranchDS(test_all, X_style_test, X_char_test),
                          batch_size=BATCH, shuffle=False, collate_fn=collate_multibranch)

print("✅ EXP1 loaders ready | train strata:\n", counts)

Target EN_1 size = 523 (matching GR_0)
Train after downsample: 1383
Train strata:
en_1    523
gr_0    523
en_0    224
gr_1    113
Name: count, dtype: int64
✅ saved outputs/style_scaler_EXP1.pkl
✅ EXP1 loaders ready | train strata:
 en_1    523
gr_0    523
en_0    224
gr_1    113
Name: count, dtype: int64


In [23]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModel
import numpy as np

# ---- basics ----
if "DEVICE" not in globals():
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ---- model: XLM-R + style + char + GRL (כמו אצלך) ----
class GradReverse(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, alpha):
        ctx.alpha = alpha
        return x.view_as(x)
    @staticmethod
    def backward(ctx, grad_output):
        return -ctx.alpha * grad_output, None

def grl(x, alpha=1.0):
    return GradReverse.apply(x, alpha)

def mean_pool(hidden, attn_mask):
    mask = attn_mask.unsqueeze(-1).type_as(hidden)
    summed = (hidden * mask).sum(dim=1)
    denom = mask.sum(dim=1).clamp(min=1e-6)
    return summed / denom

class XLMR_StyleChar_Adv(nn.Module):
    def __init__(self, model_name, style_dim, char_dim, style_h=32, char_h=128, dropout=0.2):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name, local_files_only=True)
        h = self.encoder.config.hidden_size

        self.style_mlp = nn.Sequential(nn.Linear(style_dim, 64), nn.ReLU(), nn.Dropout(dropout), nn.Linear(64, style_h))
        self.char_mlp  = nn.Sequential(nn.Linear(char_dim, 256), nn.ReLU(), nn.Dropout(dropout), nn.Linear(256, char_h))

        self.g_style = nn.Parameter(torch.tensor(-1.0))
        self.g_char  = nn.Parameter(torch.tensor(-1.0))

        fused_dim = h + style_h + char_h
        self.ln = nn.LayerNorm(fused_dim)
        self.drop = nn.Dropout(dropout)

        self.rewrite_head = nn.Linear(fused_dim, 2)
        self.lang_head    = nn.Linear(fused_dim, 2)

    def forward(self, tok, style_vec, char_vec, grl_alpha=1.0):
        out = self.encoder(**tok, return_dict=True)
        h_text = mean_pool(out.last_hidden_state, tok["attention_mask"])

        h_style = self.style_mlp(style_vec)
        h_char  = self.char_mlp(char_vec)

        gs = torch.sigmoid(self.g_style)
        gc = torch.sigmoid(self.g_char)

        fused = torch.cat([h_text, gs*h_style, gc*h_char], dim=-1)
        fused = self.drop(self.ln(fused))

        logits_y = self.rewrite_head(fused)
        logits_lang = self.lang_head(grl(fused, grl_alpha))
        return logits_y, logits_lang

STYLE_DIM = X_style_train.shape[1]
CHAR_DIM  = X_char_train.shape[1]

model = XLMR_StyleChar_Adv(MODEL_NAME, STYLE_DIM, CHAR_DIM).to(DEVICE)
print("✅ model ready on", DEVICE)

# ---- train (light) ----
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)

def run_epoch(loader, train=True, grl_alpha=1.0, lambda_lang=0.2):
    model.train(train)
    total = 0.0
    for tok, style_vec, char_vec, y, lang_id in loader:
        tok = {k:v.to(DEVICE) for k,v in tok.items()}
        style_vec = style_vec.to(DEVICE)
        char_vec  = char_vec.to(DEVICE)
        y = y.to(DEVICE)
        lang_id = lang_id.to(DEVICE)

        if train:
            optimizer.zero_grad(set_to_none=True)

        logits_y, logits_lang = model(tok, style_vec, char_vec, grl_alpha=grl_alpha)
        loss = F.cross_entropy(logits_y, y) + lambda_lang * F.cross_entropy(logits_lang, lang_id)

        if train:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        total += float(loss.item()) * y.size(0)

    return total / len(loader.dataset)

for epoch in range(1, 4):
    tr = run_epoch(train_loader, train=True, grl_alpha=1.0, lambda_lang=0.2)
    va = run_epoch(val_loader, train=False, grl_alpha=1.0, lambda_lang=0.2)
    print(f"Epoch {epoch}: train loss={tr:.4f} | val loss={va:.4f}")

C:\Users\Asoulin_Sapir\anaconda3\Lib\site-packages\torch\nn\modules\module.py:1357: UserWarning: expandable_segments not supported on this platform (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\c10/cuda/CUDAAllocatorConfig.h:35.)
  return t.to(


✅ model ready on cuda
Epoch 1: train loss=2.7121 | val loss=1.3224
Epoch 2: train loss=0.5826 | val loss=0.3162
Epoch 3: train loss=0.4503 | val loss=0.3584


### 2) EXP2 — Thresholds שונים ל־EN/GR (VAL→TEST)

In [24]:
# ==========================
# EXP2: Language-specific thresholds (VAL -> TEST), optimize F2 (FN slightly more important)
# ==========================
import numpy as np
import pandas as pd
from sklearn.metrics import precision_recall_fscore_support, classification_report, confusion_matrix
import torch

BETA = 2.0

@torch.no_grad()
def probs_df_from_loader(loader):
    model.eval()
    rows = []
    for tok, style_vec, char_vec, y, lang_id in loader:
        tok = {k: v.to(DEVICE) for k, v in tok.items()}
        style_vec = style_vec.to(DEVICE)
        char_vec  = char_vec.to(DEVICE)

        logits_y, _ = model(tok, style_vec, char_vec, grl_alpha=1.0)
        probs = torch.softmax(logits_y, dim=-1)[:, 1].detach().cpu().numpy()

        y_np = y.numpy()
        lang_np = lang_id.numpy()
        for yi, li, pi in zip(y_np, lang_np, probs):
            rows.append({"y": int(yi), "lang_id": int(li), "p": float(pi)})

    df = pd.DataFrame(rows)
    inv_lang = {v:k for k,v in LANG2ID.items()}
    df["lang"] = df["lang_id"].map(inv_lang)
    return df

def best_thr_by_lang(df_val, lang, beta=2.0, grid=None):
    if grid is None:
        grid = np.linspace(0.05, 0.95, 181)

    sub = df_val[df_val["lang"] == lang].copy()
    y = sub["y"].values
    p = sub["p"].values

    best = None
    for t in grid:
        pred = (p >= t).astype(int)
        pr, rc, _, _ = precision_recall_fscore_support(y, pred, average="binary", zero_division=0)
        fb = (1+beta**2) * pr * rc / (beta**2 * pr + rc + 1e-12)
        fp = int(((pred==1) & (y==0)).sum())
        fn = int(((pred==0) & (y==1)).sum())
        row = (t, pr, rc, fb, fp, fn)
        if best is None or row[3] > best[3]:
            best = row
    return best

def apply_lang_thresholds(df, thr_en, thr_gr):
    thr = df["lang"].map({"en": thr_en, "gr": thr_gr}).values
    pred = (df["p"].values >= thr).astype(int)
    return pred

val_df  = probs_df_from_loader(val_loader)
test_df = probs_df_from_loader(test_loader)

best_en = best_thr_by_lang(val_df, "en", beta=BETA)
best_gr = best_thr_by_lang(val_df, "gr", beta=BETA)

thr_en, thr_gr = best_en[0], best_gr[0]
print("Best EN threshold (VAL, F2):  t=%.3f  P=%.3f R=%.3f F2=%.3f FP=%d FN=%d" % best_en)
print("Best GR threshold (VAL, F2):  t=%.3f  P=%.3f R=%.3f F2=%.3f FP=%d FN=%d" % best_gr)

test_pred = apply_lang_thresholds(test_df, thr_en, thr_gr)
print("\n=== TEST overall (lang-specific thresholds) ===")
print(classification_report(test_df["y"].values, test_pred, digits=3))

for lang in ["en", "gr"]:
    sub = test_df[test_df["lang"]==lang].copy()
    pred = apply_lang_thresholds(sub, thr_en, thr_gr)
    print(f"\n=== TEST {lang.upper()} ===")
    print(classification_report(sub["y"].values, pred, digits=3))
    print("Confusion:\n", confusion_matrix(sub["y"].values, pred))

Best EN threshold (VAL, F2):  t=0.620  P=0.984 R=1.000 F2=0.997 FP=12 FN=0
Best GR threshold (VAL, F2):  t=0.155  P=0.520 R=0.929 F2=0.802 FP=12 FN=1

=== TEST overall (lang-specific thresholds) ===
              precision    recall  f1-score   support

           0      0.942     0.691     0.798        94
           1      0.962     0.995     0.978       740

    accuracy                          0.960       834
   macro avg      0.952     0.843     0.888       834
weighted avg      0.960     0.960     0.958       834


=== TEST EN ===
              precision    recall  f1-score   support

           0      0.824     0.500     0.622        28
           1      0.981     0.996     0.988       726

    accuracy                          0.977       754
   macro avg      0.902     0.748     0.805       754
weighted avg      0.975     0.977     0.975       754

Confusion:
 [[ 14  14]
 [  3 723]]

=== TEST GR ===
              precision    recall  f1-score   support

           0      0.981

In [27]:
# ==========================
# POS/Lemma setup (Drop-in) -> defines add_poslemma_to_split
# ==========================
import os
import pandas as pd

os.makedirs("outputs/poslemma_cache", exist_ok=True)

# Try spaCy for English (optional)
try:
    import spacy
    try:
        _en_nlp = spacy.load("en_core_web_sm", disable=["ner"])
    except Exception:
        _en_nlp = None
except Exception:
    _en_nlp = None

def _ensure_poslemma_columns(df):
    # if already exists, keep
    if "pos" in df.columns and "lemma" in df.columns:
        return df

    df = df.copy()
    df["lang"] = df["lang"].astype(str).str.lower()

    # init
    if "pos" not in df.columns: df["pos"] = ""
    if "lemma" not in df.columns: df["lemma"] = ""

    # English via spaCy if available
    if _en_nlp is not None:
        en_mask = df["lang"].eq("en")
        en_texts = df.loc[en_mask, "text"].fillna("").astype(str).tolist()
        pos_out, lem_out = [], []
        for doc in _en_nlp.pipe(en_texts, batch_size=64):
            pos_out.append(" ".join([t.pos_ for t in doc]))
            lem_out.append(" ".join([t.lemma_ for t in doc]))
        df.loc[en_mask, "pos"] = pos_out
        df.loc[en_mask, "lemma"] = lem_out
    else:
        # fallback: simple tokenization
        en_mask = df["lang"].eq("en")
        df.loc[en_mask, "pos"] = df.loc[en_mask, "text"].fillna("").astype(str).str.split().apply(lambda xs: " ".join(["X"]*len(xs)))
        df.loc[en_mask, "lemma"] = df.loc[en_mask, "text"].fillna("").astype(str)

    # Greek fallback (placeholder) — if you have your real greek tagger cell, prefer that!
    gr_mask = df["lang"].eq("gr")
    # keep lemma as text (placeholder), pos as X tokens
    df.loc[gr_mask, "lemma"] = df.loc[gr_mask, "text"].fillna("").astype(str)
    df.loc[gr_mask, "pos"] = df.loc[gr_mask, "text"].fillna("").astype(str).str.split().apply(lambda xs: " ".join(["X"]*len(xs)))

    return df

def add_poslemma_to_split(df, split_name):
    cache_path = f"outputs/poslemma_cache/{split_name}.parquet"
    if os.path.exists(cache_path):
        out = pd.read_parquet(cache_path)
        return out

    out = _ensure_poslemma_columns(df)
    out.to_parquet(cache_path, index=False)
    print("✅ cached:", cache_path)
    return out

print("✅ add_poslemma_to_split is ready | spaCy EN:", _en_nlp is not None)

✅ add_poslemma_to_split is ready | spaCy EN: True


### 3) EXP3 — POS/Lemma: cache→hash→loaders

In [28]:
# ==========================
# EXP3: POS/Lemma "אמיתיים" + hashed branches + gates + GRL
# Notes:
# - Uses your existing add_poslemma_to_split(df, split_name) function (with cache).
# - Keeps EXP1 downsample + style/char features.
# ==========================
import numpy as np
from sklearn.feature_extraction.text import HashingVectorizer
import torch
from torch.utils.data import DataLoader, WeightedRandomSampler

assert "add_poslemma_to_split" in globals(), "add_poslemma_to_split not found. Run the POS/Lemma setup cells first (spacy+greek tagger)."

    
train_all_pl = add_poslemma_to_split(train_all, "train_all_EXP3")
val_all_pl   = add_poslemma_to_split(val_all,   "val_all_EXP3")
test_all_pl  = add_poslemma_to_split(test_all,  "test_all_EXP3")

print("✅ POS/Lemma columns present:", [c for c in ["pos","lemma"] if c in train_all_pl.columns])

POS_DIM = 2048
LEM_DIM = 2048

pos_vec = HashingVectorizer(n_features=POS_DIM, analyzer="word", ngram_range=(1, 3),
                            alternate_sign=False, norm=None)
lem_vec = HashingVectorizer(n_features=LEM_DIM, analyzer="word", ngram_range=(1, 2),
                            alternate_sign=False, norm=None)

def hash_pos_lem(df):
    pos_str = df["pos"].fillna("").astype(str)
    lem_str = df["lemma"].fillna("").astype(str)
    X_pos = pos_vec.transform(pos_str).toarray().astype(np.float16)
    X_lem = lem_vec.transform(lem_str).toarray().astype(np.float16)
    return X_pos, X_lem

X_pos_train, X_lem_train = hash_pos_lem(train_all_pl)
X_pos_val,   X_lem_val   = hash_pos_lem(val_all_pl)
X_pos_test,  X_lem_test  = hash_pos_lem(test_all_pl)

print("✅ POS/Lemma hashed:", X_pos_train.shape, X_lem_train.shape)

class MultiBranchPlusDS(torch.utils.data.Dataset):
    def __init__(self, df, X_style, X_char, X_pos, X_lem):
        self.df = df.reset_index(drop=True)
        self.X_style = X_style
        self.X_char  = X_char
        self.X_pos   = X_pos
        self.X_lem   = X_lem
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        r = self.df.iloc[idx]
        lang = str(r["lang"]).lower()
        return {
            "text": str(r["text"]),
            "y": int(r["y"]),
            "lang_id": LANG2ID[lang],
            "style": self.X_style[idx],
            "char":  self.X_char[idx],
            "pos":   self.X_pos[idx],
            "lem":   self.X_lem[idx],
        }

def collate_multibranch_plus(batch):
    texts = [b["text"] for b in batch]
    y = torch.tensor([b["y"] for b in batch], dtype=torch.long)
    lang_id = torch.tensor([b["lang_id"] for b in batch], dtype=torch.long)

    tok = tokenizer(texts, padding=True, truncation=True, max_length=MAX_LEN, return_tensors="pt")

    style = torch.tensor(np.stack([b["style"] for b in batch]), dtype=torch.float32)
    char  = torch.tensor(np.stack([b["char"]  for b in batch]), dtype=torch.float32)
    pos   = torch.tensor(np.stack([b["pos"]   for b in batch]), dtype=torch.float32)
    lem   = torch.tensor(np.stack([b["lem"]   for b in batch]), dtype=torch.float32)

    return tok, style, char, pos, lem, y, lang_id

# Sampler by (lang_y)
strata = train_all_pl["lang"].astype(str).str.lower() + "_" + train_all_pl["y"].astype(str)
counts = strata.value_counts()
weights = strata.map(lambda s: 1.0 / counts[s]).values
sampler = WeightedRandomSampler(torch.tensor(weights, dtype=torch.double),
                                num_samples=len(weights), replacement=True)

train_loader = DataLoader(MultiBranchPlusDS(train_all_pl, X_style_train, X_char_train, X_pos_train, X_lem_train),
                          batch_size=BATCH, sampler=sampler, collate_fn=collate_multibranch_plus)
val_loader   = DataLoader(MultiBranchPlusDS(val_all_pl, X_style_val, X_char_val, X_pos_val, X_lem_val),
                          batch_size=BATCH, shuffle=False, collate_fn=collate_multibranch_plus)
test_loader  = DataLoader(MultiBranchPlusDS(test_all_pl, X_style_test, X_char_test, X_pos_test, X_lem_test),
                          batch_size=BATCH, shuffle=False, collate_fn=collate_multibranch_plus)

print("✅ EXP3 loaders ready")

✅ cached: outputs/poslemma_cache/train_all_EXP3.parquet
✅ cached: outputs/poslemma_cache/val_all_EXP3.parquet
✅ cached: outputs/poslemma_cache/test_all_EXP3.parquet
✅ POS/Lemma columns present: ['pos', 'lemma']
✅ POS/Lemma hashed: (1383, 2048) (1383, 2048)
✅ EXP3 loaders ready


### 4) EXP3 — מודל + אימון חסכוני (AMP+checkpointing)

In [29]:
# ==========================
# EXP3-D/E: Model + memory-friendly training (AMP + gradient checkpointing)
# ==========================
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModel
from torch.amp import autocast, GradScaler
from sklearn.metrics import f1_score
import numpy as np

class XLMR_StyleCharPosLemma_Adv(nn.Module):
    def __init__(self, model_name, style_dim, char_dim, pos_dim, lem_dim,
                 style_h=32, char_h=128, pos_h=64, lem_h=64, dropout=0.2):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name, local_files_only=True)
        h = self.encoder.config.hidden_size

        self.style_mlp = nn.Sequential(nn.Linear(style_dim, 64), nn.ReLU(), nn.Dropout(dropout), nn.Linear(64, style_h))
        self.char_mlp  = nn.Sequential(nn.Linear(char_dim, 256), nn.ReLU(), nn.Dropout(dropout), nn.Linear(256, char_h))
        self.pos_mlp   = nn.Sequential(nn.Linear(pos_dim, 256), nn.ReLU(), nn.Dropout(dropout), nn.Linear(256, pos_h))
        self.lem_mlp   = nn.Sequential(nn.Linear(lem_dim, 256), nn.ReLU(), nn.Dropout(dropout), nn.Linear(256, lem_h))

        self.g_style = nn.Parameter(torch.tensor(-1.0))
        self.g_char  = nn.Parameter(torch.tensor(-1.0))
        self.g_pos   = nn.Parameter(torch.tensor(-1.0))
        self.g_lem   = nn.Parameter(torch.tensor(-1.0))

        fused_dim = h + style_h + char_h + pos_h + lem_h
        self.ln = nn.LayerNorm(fused_dim)
        self.drop = nn.Dropout(dropout)

        self.rewrite_head = nn.Linear(fused_dim, 2)
        self.lang_head    = nn.Linear(fused_dim, 2)

    def forward(self, tok, style_vec, char_vec, pos_vec, lem_vec, grl_alpha=1.0):
        out = self.encoder(**tok, return_dict=True)
        h_text = mean_pool(out.last_hidden_state, tok["attention_mask"])

        h_style = self.style_mlp(style_vec)
        h_char  = self.char_mlp(char_vec)
        h_pos   = self.pos_mlp(pos_vec)
        h_lem   = self.lem_mlp(lem_vec)

        gs = torch.sigmoid(self.g_style)
        gc = torch.sigmoid(self.g_char)
        gp = torch.sigmoid(self.g_pos)
        gl = torch.sigmoid(self.g_lem)

        fused = torch.cat([h_text, gs*h_style, gc*h_char, gp*h_pos, gl*h_lem], dim=-1)
        fused = self.drop(self.ln(fused))

        logits_y    = self.rewrite_head(fused)
        logits_lang = self.lang_head(grl(fused, grl_alpha))
        return logits_y, logits_lang

STYLE_DIM = X_style_train.shape[1]
CHAR_DIM  = X_char_train.shape[1]
POS_DIM   = X_pos_train.shape[1]
LEM_DIM   = X_lem_train.shape[1]

model = XLMR_StyleCharPosLemma_Adv(MODEL_NAME, STYLE_DIM, CHAR_DIM, POS_DIM, LEM_DIM).to(DEVICE)
print("✅ EXP3 model ready on", DEVICE, "| gates(sigmoid)=", float(torch.sigmoid(model.g_pos).item()))

# Gradient checkpointing to fit 6GB GPUs
try:
    model.encoder.gradient_checkpointing_enable()
    model.encoder.config.use_cache = False
    print("✅ gradient checkpointing enabled")
except Exception as e:
    print("⚠️ could not enable gradient checkpointing:", e)

scaler = GradScaler("cuda", enabled=(DEVICE=="cuda"))

def run_epoch_amp(loader, optimizer, train=True, grl_alpha=1.0, lambda_lang=0.2, accum_steps=8):
    model.train(train)
    total_loss = 0.0
    ys, ps = [], []

    if train:
        optimizer.zero_grad(set_to_none=True)

    for step, batch in enumerate(loader, start=1):
        tok, style_vec, char_vec, pos_vec, lem_vec, y, lang_id = batch
        tok = {k: v.to(DEVICE) for k,v in tok.items()}
        style_vec = style_vec.to(DEVICE)
        char_vec  = char_vec.to(DEVICE)
        pos_vec   = pos_vec.to(DEVICE)
        lem_vec   = lem_vec.to(DEVICE)
        y         = y.to(DEVICE)
        lang_id   = lang_id.to(DEVICE)

        with autocast("cuda", enabled=(DEVICE=="cuda")):
            logits_y, logits_lang = model(tok, style_vec, char_vec, pos_vec, lem_vec, grl_alpha=grl_alpha)
            loss_y = F.cross_entropy(logits_y, y)
            loss_lang = F.cross_entropy(logits_lang, lang_id)
            loss = loss_y + lambda_lang * loss_lang
            loss_scaled = loss / accum_steps

        if train:
            scaler.scale(loss_scaled).backward()
            if step % accum_steps == 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)

        total_loss += float(loss.item()) * y.size(0)

        prob = torch.softmax(logits_y, dim=-1)[:,1].detach().float().cpu().numpy()
        ys.append(y.detach().cpu().numpy())
        ps.append(prob)

    if train and (step % accum_steps != 0):
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)

    ys = np.concatenate(ys)
    ps = np.concatenate(ps)
    pred = (ps >= 0.5).astype(int)
    f1 = f1_score(ys, pred, zero_division=0)
    return total_loss / len(loader.dataset), f1

# Epoch 1: freeze encoder (warmup)
for p in model.encoder.parameters():
    p.requires_grad = False
optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=2e-5, weight_decay=0.01)

tr_loss, tr_f1 = run_epoch_amp(train_loader, optimizer, train=True)
va_loss, va_f1 = run_epoch_amp(val_loader,   optimizer, train=False)
print(f"Epoch 1 (frozen): train loss={tr_loss:.4f} f1={tr_f1:.3f} | val loss={va_loss:.4f} f1={va_f1:.3f}")

# Unfreeze and continue
for p in model.encoder.parameters():
    p.requires_grad = True
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)

for epoch in range(2, 5):
    tr_loss, tr_f1 = run_epoch_amp(train_loader, optimizer, train=True)
    va_loss, va_f1 = run_epoch_amp(val_loader,   optimizer, train=False)
    print(f"Epoch {epoch}: train loss={tr_loss:.4f} f1={tr_f1:.3f} | val loss={va_loss:.4f} f1={va_f1:.3f}")

✅ EXP3 model ready on cuda | gates(sigmoid)= 0.2689414322376251
✅ gradient checkpointing enabled


C:\Users\Asoulin_Sapir\anaconda3\Lib\site-packages\torch\utils\checkpoint.py:85: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(


Epoch 1 (frozen): train loss=0.8677 f1=0.208 | val loss=1.0095 f1=0.003
Epoch 2: train loss=1.6514 f1=0.767 | val loss=3.7305 f1=0.972
Epoch 3: train loss=2.2105 f1=0.850 | val loss=3.4182 f1=0.980
Epoch 4: train loss=1.9037 f1=0.945 | val loss=2.8915 f1=0.984


### 5) EXP3 — Thresholds EN/GR + בדיקת GR p-stats

In [30]:
# ==========================
# EXP3 + EXP2 מחדש: thresholds per language on the POS/Lemma model
# ==========================
import numpy as np
import pandas as pd
from sklearn.metrics import precision_recall_fscore_support, classification_report, confusion_matrix
import torch

BETA = 2.0

@torch.no_grad()
def probs_df_from_loader_plus(loader):
    model.eval()
    rows = []
    for tok, style_vec, char_vec, pos_vec, lem_vec, y, lang_id in loader:
        tok = {k: v.to(DEVICE) for k, v in tok.items()}
        style_vec = style_vec.to(DEVICE)
        char_vec  = char_vec.to(DEVICE)
        pos_vec   = pos_vec.to(DEVICE)
        lem_vec   = lem_vec.to(DEVICE)

        logits_y, _ = model(tok, style_vec, char_vec, pos_vec, lem_vec, grl_alpha=1.0)
        probs = torch.softmax(logits_y, dim=-1)[:, 1].detach().cpu().numpy()

        y_np = y.numpy()
        lang_np = lang_id.numpy()
        for yi, li, pi in zip(y_np, lang_np, probs):
            rows.append({"y": int(yi), "lang_id": int(li), "p": float(pi)})

    df = pd.DataFrame(rows)
    inv_lang = {v:k for k,v in LANG2ID.items()}
    df["lang"] = df["lang_id"].map(inv_lang)
    return df

def best_thr_by_lang(df_val, lang, beta=2.0, grid=None):
    if grid is None:
        grid = np.linspace(0.05, 0.95, 181)
    sub = df_val[df_val["lang"] == lang].copy()
    y = sub["y"].values
    p = sub["p"].values
    best = None
    for t in grid:
        pred = (p >= t).astype(int)
        pr, rc, _, _ = precision_recall_fscore_support(y, pred, average="binary", zero_division=0)
        fb = (1+beta**2) * pr * rc / (beta**2 * pr + rc + 1e-12)
        fp = int(((pred==1) & (y==0)).sum())
        fn = int(((pred==0) & (y==1)).sum())
        row = (t, pr, rc, fb, fp, fn)
        if best is None or row[3] > best[3]:
            best = row
    return best

def apply_lang_thresholds(df, thr_en, thr_gr):
    thr = df["lang"].map({"en": thr_en, "gr": thr_gr}).values
    return (df["p"].values >= thr).astype(int)

val_df  = probs_df_from_loader_plus(val_loader)
test_df = probs_df_from_loader_plus(test_loader)

best_en = best_thr_by_lang(val_df, "en", beta=BETA)
best_gr = best_thr_by_lang(val_df, "gr", beta=BETA)

thr_en, thr_gr = best_en[0], best_gr[0]
print("Best EN threshold (VAL, F2):", best_en)
print("Best GR threshold (VAL, F2):", best_gr)

test_pred = apply_lang_thresholds(test_df, thr_en, thr_gr)
print("\n=== TEST overall (lang-specific thresholds) ===")
print(classification_report(test_df["y"].values, test_pred, digits=3))

for lang in ["en", "gr"]:
    sub = test_df[test_df["lang"]==lang].copy()
    pred = apply_lang_thresholds(sub, thr_en, thr_gr)
    print(f"\n=== TEST {lang.upper()} ===")
    print(classification_report(sub["y"].values, pred, digits=3))
    print("Confusion:\n", confusion_matrix(sub["y"].values, pred))

# quick GR probability stats to see if the "low p" FN cluster moved up
gr_test = test_df[test_df["lang"]=="gr"].copy()
p1 = gr_test[gr_test["y"]==1]["p"].values
p0 = gr_test[gr_test["y"]==0]["p"].values
print("\nGR TEST p stats | y=1 (rewrite):", np.min(p1), np.percentile(p1,25), np.median(p1), np.percentile(p1,75), np.max(p1))
print("GR TEST p stats | y=0 (orig):   ", np.min(p0), np.percentile(p0,25), np.median(p0), np.percentile(p0,75), np.max(p0))

Best EN threshold (VAL, F2): (np.float64(0.09), 0.974496644295302, 1.0, 0.9947930939981525, 19, 0)
Best GR threshold (VAL, F2): (np.float64(0.9049999999999999), 0.8666666666666667, 0.9285714285714286, 0.9154929577462707, 2, 1)

=== TEST overall (lang-specific thresholds) ===
              precision    recall  f1-score   support

           0      0.921     0.745     0.824        94
           1      0.968     0.992     0.980       740

    accuracy                          0.964       834
   macro avg      0.945     0.868     0.902       834
weighted avg      0.963     0.964     0.962       834


=== TEST EN ===
              precision    recall  f1-score   support

           0      0.833     0.179     0.294        28
           1      0.969     0.999     0.984       726

    accuracy                          0.968       754
   macro avg      0.901     0.589     0.639       754
weighted avg      0.964     0.968     0.958       754

Confusion:
 [[  5  23]
 [  1 725]]

=== TEST GR ===
 

תא A — לבחור thr_en לפי F1, thr_gr לפי F2+cap

In [31]:
import numpy as np
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix, classification_report

def best_thr(df_val, lang, mode="f1", beta=2.0, fp_cap=None):
    sub = df_val[df_val["lang"]==lang].copy()
    y = sub["y"].values
    p = sub["p"].values

    best = None
    for t in np.linspace(0.05, 0.95, 181):
        pred = (p >= t).astype(int)
        pr, rc, f1, _ = precision_recall_fscore_support(y, pred, average="binary", zero_division=0)
        fp = int(((pred==1) & (y==0)).sum())
        fn = int(((pred==0) & (y==1)).sum())

        if fp_cap is not None and fp > fp_cap:
            continue

        if mode == "f1":
            score = f1
        else:  # fbeta
            score = (1+beta**2) * pr * rc / (beta**2 * pr + rc + 1e-12)

        row = (t, pr, rc, f1, score, fp, fn)
        if best is None or row[4] > best[4]:
            best = row
    return best

# EN: F1
best_en = best_thr(val_df, "en", mode="f1")
thr_en = best_en[0]
print("EN best by F1: t=%.3f P=%.3f R=%.3f F1=%.3f FP=%d FN=%d" %
      (best_en[0], best_en[1], best_en[2], best_en[3], best_en[5], best_en[6]))

# GR: F2 with FP cap
best_gr = best_thr(val_df, "gr", mode="fbeta", beta=2.0, fp_cap=2)
thr_gr = best_gr[0]
print("GR best by F2 (FP<=2): t=%.3f P=%.3f R=%.3f F1=%.3f F2=%.3f FP=%d FN=%d" %
      (best_gr[0], best_gr[1], best_gr[2], best_gr[3], best_gr[4], best_gr[5], best_gr[6]))

def apply_lang_thresholds(df, thr_en, thr_gr):
    thr = df["lang"].map({"en": thr_en, "gr": thr_gr}).values
    return (df["p"].values >= thr).astype(int)

test_pred = apply_lang_thresholds(test_df, thr_en, thr_gr)

print("\n=== TEST overall ===")
print(classification_report(test_df["y"].values, test_pred, digits=3))

for lang in ["en","gr"]:
    sub = test_df[test_df["lang"]==lang].copy()
    pred = apply_lang_thresholds(sub, thr_en, thr_gr)
    print(f"\n=== TEST {lang.upper()} ===")
    print(classification_report(sub["y"].values, pred, digits=3))
    print("Confusion:\n", confusion_matrix(sub["y"].values, pred))

EN best by F1: t=0.325 P=0.989 R=0.994 F1=0.992 FP=8 FN=4
GR best by F2 (FP<=2): t=0.905 P=0.867 R=0.929 F1=0.897 F2=0.915 FP=2 FN=1

=== TEST overall ===
              precision    recall  f1-score   support

           0      0.871     0.862     0.866        94
           1      0.982     0.984     0.983       740

    accuracy                          0.970       834
   macro avg      0.927     0.923     0.925       834
weighted avg      0.970     0.970     0.970       834


=== TEST EN ===
              precision    recall  f1-score   support

           0      0.696     0.571     0.627        28
           1      0.984     0.990     0.987       726

    accuracy                          0.975       754
   macro avg      0.840     0.781     0.807       754
weighted avg      0.973     0.975     0.974       754

Confusion:
 [[ 16  12]
 [  7 719]]

=== TEST GR ===
              precision    recall  f1-score   support

           0      0.929     0.985     0.956        66
           1 

תא B — להבין למה יש FN ב-GR עם thr_gr גבוה

In [32]:
gr_test = test_df[test_df["lang"]=="gr"].copy()
t = thr_gr
fn_mask = (gr_test["y"].values==1) & (gr_test["p"].values < t)
print("thr_gr =", t)
print("GR rewrite count:", (gr_test["y"]==1).sum())
print("GR FN count under thr:", fn_mask.sum())

print("\nFN p values (sorted):")
print(np.sort(gr_test.loc[fn_mask, "p"].values))

thr_gr = 0.9049999999999999
GR rewrite count: 14
GR FN count under thr: 5

FN p values (sorted):
[0.29773024 0.34434459 0.6660412  0.75405306 0.76539123]


🧪 תא חדש: לבחור thr_gr לפי F1 עם FP cap

In [33]:
import numpy as np
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix, classification_report

def best_thr_f1_with_fp_cap(df_val, lang, fp_cap=2):
    sub = df_val[df_val["lang"]==lang].copy()
    y = sub["y"].values
    p = sub["p"].values
    best = None
    for t in np.linspace(0.05, 0.95, 181):
        pred = (p >= t).astype(int)
        pr, rc, f1, _ = precision_recall_fscore_support(y, pred, average="binary", zero_division=0)
        fp = int(((pred==1) & (y==0)).sum())
        fn = int(((pred==0) & (y==1)).sum())
        if fp <= fp_cap:
            row = (t, pr, rc, f1, fp, fn)
            if best is None or row[3] > best[3]:
                best = row
    return best

best_gr_f1cap = best_thr_f1_with_fp_cap(val_df, "gr", fp_cap=2)
thr_gr2 = best_gr_f1cap[0]
print("GR best by F1 (FP<=2): t=%.3f P=%.3f R=%.3f F1=%.3f FP=%d FN=%d" % best_gr_f1cap)

# evaluate on TEST GR
gr_test = test_df[test_df["lang"]=="gr"].copy()
pred_gr = (gr_test["p"].values >= thr_gr2).astype(int)
print("\n=== TEST GR with thr_gr2 ===")
print(classification_report(gr_test["y"].values, pred_gr, digits=3))
print("Confusion:\n", confusion_matrix(gr_test["y"].values, pred_gr))

# show which FN remain
fn_mask = (gr_test["y"].values==1) & (gr_test["p"].values < thr_gr2)
print("Remaining FN count:", fn_mask.sum())
print("Remaining FN p values:", np.sort(gr_test.loc[fn_mask, "p"].values))

GR best by F1 (FP<=2): t=0.905 P=0.867 R=0.929 F1=0.897 FP=2 FN=1

=== TEST GR with thr_gr2 ===
              precision    recall  f1-score   support

           0      0.929     0.985     0.956        66
           1      0.900     0.643     0.750        14

    accuracy                          0.925        80
   macro avg      0.914     0.814     0.853        80
weighted avg      0.924     0.925     0.920        80

Confusion:
 [[65  1]
 [ 5  9]]
Remaining FN count: 5
Remaining FN p values: [0.29773024 0.34434459 0.6660412  0.75405306 0.76539123]


In [34]:
import os, json, torch, joblib

os.makedirs("exports", exist_ok=True)

torch.save(model.state_dict(), "exports/exp3_model_state.pt")
joblib.dump(style_scaler, "exports/style_scaler_EXP1.pkl")

cfg = {
    "MODEL_NAME": MODEL_NAME,
    "MAX_LEN": int(MAX_LEN),
    "BATCH_SIZE": int(BATCH),
    "thr_en": float(thr_en),
    "thr_gr": float(thr_gr),
    "CHAR_DIM": int(CHAR_DIM),
    "POS_DIM": int(POS_DIM),
    "LEM_DIM": int(LEM_DIM),
}
with open("exports/exp3_thresholds.json", "w", encoding="utf-8") as f:
    json.dump(cfg, f, indent=2)

print("✅ Saved exports/: exp3_model_state.pt, style_scaler_EXP1.pkl, exp3_thresholds.json")

✅ Saved exports/: exp3_model_state.pt, style_scaler_EXP1.pkl, exp3_thresholds.json
